# Turn-Based Lexical Combat Game — Self-Trained Embedding Models

This notebook replaces the game's scorer (`all-MiniLM-L6-v2`, pretrained) with **word embeddings trained from random initialization** on a corpus we collect and prepare ourselves. The engine, calibration, holdout evaluation and play screens from the original notebook (`Lexical_Combat_Game_refactored.ipynb`) are reused as much as possible. The original notebook is not modified.

**Game rules (unchanged):** the player types a 1–3 word English phrase to attack a boss that has a **domain** (Business / Technology / Science / Arts) and a **target word**. Each guess is judged in two steps:
1. **Domain gate** — top-k anchor cosine minus each domain's background → *primary* ×1.00 / *secondary* ×0.75 / *weak secondary* ×0.50 / *blocked*
2. **Target relevance → damage** — `r = (cos − noise_floor) / (ceiling − noise_floor)`, then mapped to damage by a curve re-calibrated for each model

**Models (all trained on the same corpus):**

| Model | Type | Role |
|---|---|---|
| PPMI + Truncated SVD | count-based (classic distributional semantics) | easy-to-explain baseline |
| Word2Vec (skip-gram / CBOW, negative sampling) | prediction-based | main candidate |
| FastText (skip-gram + character n-grams) | prediction-based + subword | handles OOV / misspelled words |
| MiniLM (`all-MiniLM-L6-v2`) | pretrained Transformer | **reference baseline only**, not the game's main model |

**How to run:** set the flags in 0.1, then **Runtime → Run all**. The first run downloads the corpus and trains the models (about 1–1.5 hours on free Colab). Every step saves its results to Google Drive, so later runs (or a run after a disconnect) load them and finish in a few minutes. Set `QUICK_RUN = True` to test the whole pipeline on a small slice of data in a few minutes.

| Flag | Default | Meaning |
|---|---|---|
| `USE_DRIVE` | `True` | store corpus / models / calibration on Drive (if mounting fails → local paths, nothing persists) |
| `QUICK_RUN` | `False` | small data slice + small search to test the pipeline (stored in a separate `_quick` folder) |
| `RUN_DATA_PREP` | `True` | build the corpus if no cached file exists (a cached corpus is always loaded) |
| `RUN_TRAINING` | `True` | train models that have no saved file (saved models are always loaded) |
| `RUN_HP_SEARCH` | `True` | small hyper-parameter search on a corpus sample (selection uses intrinsic + calibration set only) |
| `FORCE_REBUILD` | `False` | ignore every cache and rebuild everything |
| `MODEL_CHOICE` | `"auto"` | `"auto"` = best self-trained model (chosen on the calibration set) / `"ppmi_svd"` / `"word2vec"` / `"fasttext"` / `"minilm_reference"` |
| `RUN_MINILM_REFERENCE` | `True` | load MiniLM for comparison only |
| `LOAD_TUNED_PARAMS` | `True` | reuse a matching `tuned_params_<model>.json` instead of re-running the grids |
| `RUN_HOLDOUT` | `True` | evaluate the holdout **once**, after every choice is fixed |
| `PLAY_INTERACTIVE` | `False` | Section 11: `True` = type guesses with `input()` / `False` = simulated battles |
| `PLAY_MODE` | `"random"` | `"random"` = random topic + target / `"classic"` = the 3 fixed stages |
| `SEED` | `42` | seed for numpy / random / gensim / the random game |

**Sections:** 0 Setup · 1 Data registry · 2 Text utilities · 3 Engine + calibration functions · 4 Corpus · 5 Encoders (adapter) · 6 Training · 7 Model loading, unit tests, intrinsic evaluation · 8 Calibration · 9 Report + model comparison · 10 Holdout + error analysis · 11 Play (simulated battles) · 12 Report summary · Play Game (last cell)

**Data-leakage rules (enforced in code):** game data (anchors, pools, test cases, holdout) is used **only to check coverage** and is never written to the corpus · the holdout is never used to choose hyper-parameters, checkpoints, calibration or supplement articles · WordSim-353 / SimLex-999 are used **for evaluation only**

## 0. Setup
- **0.1** Run flags (the only cell you normally edit).
- **0.2** Installs only the packages Colab lacks (`gensim`; never upgrades numpy / pandas / scipy / torch), all imports, seeds and the per-section timer.
- **0.3** Paths on Google Drive with a local fallback, plus a local work folder for large files (faster I/O than Drive).

In [ ]:
# 0.1 Flags — the only cell you normally edit. Then: Runtime → Run all.
# What this cell does:
#   - Run flags that control every section.
#   - Edit these, then use Runtime -> Run all.
USE_DRIVE = True              # persist corpus, models and calibration on Drive; False = local, session only
QUICK_RUN = False             # small corpus slice + tiny search: tests the whole pipeline in a few minutes
RUN_DATA_PREP = True          # build the corpus if no cached copy exists (a cached copy is always loaded)
RUN_TRAINING = True           # train models that have no saved copy (saved models are always loaded)
RUN_HP_SEARCH = True          # small hyper-parameter search on a corpus sample (calibration set only)
FORCE_REBUILD = False         # ignore every cache (corpus, search, models, calibration) and redo it
MODEL_CHOICE = "auto"         # "auto" | "ppmi_svd" | "word2vec" | "fasttext" | "minilm_reference"
RUN_MINILM_REFERENCE = True   # MiniLM as a reference baseline only (comparison tables)
LOAD_TUNED_PARAMS = True      # reuse a matching tuned_params_<model>.json (skips the grid searches)
RUN_HOLDOUT = True            # Section 10: evaluate the holdout once, after every choice is fixed
PLAY_INTERACTIVE = False      # Section 11: True = type guesses with input(); False = simulated battles
PLAY_MODE = "random"          # "random" (random topic + target) | "classic" (3 fixed stages)
SEED = 42                     # numpy / random / gensim / random-game seed

In [ ]:
# 0.2 Installs (only what Colab lacks), imports, seeds, run tracker
# What this cell does:
#   - Installs only missing packages (gensim, datasets, sentence-transformers).
#   - Pins numpy / pandas / scipy / torch so Colab never needs a restart.
#   - Imports every library, sets random seeds.
#   - Defines section() / log_section() to time each section for the run summary.
import importlib.metadata as _metadata
import subprocess
import sys

GENSIM_REQUIREMENT = "gensim>=4.3,<5"                 # Word2Vec / FastText / Phrases (4.4+ supports NumPy 2)
DATASETS_REQUIREMENT = "datasets>=2.14"               # Hugging Face corpus download (preinstalled on Colab)
ST_REQUIREMENT = "sentence-transformers>=3.0,<6"      # MiniLM reference baseline only
NEVER_UPGRADE = ["numpy", "pandas", "scipy", "torch"]  # upgrading these forces a runtime restart

def _installed_version(package: str):
    try:
        return _metadata.version(package)
    except _metadata.PackageNotFoundError:
        return None

def _requirement_met(requirement: str) -> bool:
    from packaging.requirements import Requirement
    req = Requirement(requirement)
    version = _installed_version(req.name)
    return version is not None and req.specifier.contains(version, prereleases=True)

def pip_install(requirements: list) -> bool:
    """Install packages while pinning numpy/pandas/scipy/torch to the installed versions."""
    pins = [f"{p}=={_installed_version(p)}" for p in NEVER_UPGRADE if _installed_version(p)]
    constraints = "/tmp/lexical_constraints.txt"
    with open(constraints, "w") as handle:
        handle.write("\n".join(pins))
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *requirements, "-c", constraints],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(f"WARNING: pip install {requirements} failed; continuing with installed packages. "
              f"{result.stderr.strip().splitlines()[-1] if result.stderr.strip() else ''}")
        return False
    return True

_needed = [r for r in (GENSIM_REQUIREMENT, DATASETS_REQUIREMENT) if not _requirement_met(r)]
if (RUN_MINILM_REFERENCE or MODEL_CHOICE == "minilm_reference") and not _requirement_met(ST_REQUIREMENT):
    _needed.append(ST_REQUIREMENT)
if _needed:
    print("Installing:", _needed)
    pip_install(_needed)
else:
    print("All required packages are already installed.")

import copy
import gc
import hashlib
import html as _html
import itertools
import json
import os
import random
import re
import shutil
import time
from collections import Counter
from contextlib import contextmanager
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from IPython.display import Markdown, display
from scipy.ndimage import uniform_filter
from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score, roc_auc_score, silhouette_score
from sklearn.utils.extmath import randomized_svd

import gensim
from gensim.models import FastText, KeyedVectors, Word2Vec
from gensim.models.callbacks import CallbackAny2Vec
from gensim.models.phrases import ENGLISH_CONNECTOR_WORDS, FrozenPhrases, Phrases
from gensim.models.word2vec import LineSentence
from gensim.test.utils import datapath

np.random.seed(SEED)
random.seed(SEED)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)
WORKERS = max(1, os.cpu_count() or 1)   # Colab free: 2 cores

# Run tracker: one entry per section, reset on every Run all.
RUN_LOG = []

class _SectionEntry(dict):
    def skip(self, reason: str) -> None:
        """Mark the section as skipped and say why."""
        self["status"], self["reason"] = "skipped", reason
        print(f"Skipped: {reason}")

@contextmanager
def section(name: str):
    """Time a section and record ran / skipped for the final summary."""
    entry = _SectionEntry(section=name, status="ran", reason="", seconds=0.0)
    RUN_LOG[:] = [e for e in RUN_LOG if e["section"] != name]   # idempotent re-runs
    RUN_LOG.append(entry)
    start = time.perf_counter()
    try:
        yield entry
    finally:
        entry["seconds"] = round(time.perf_counter() - start, 2)

def log_section(name: str, start: float) -> None:
    """Record a section that has no skip path."""
    RUN_LOG[:] = [e for e in RUN_LOG if e["section"] != name]
    RUN_LOG.append(_SectionEntry(section=name, status="ran", reason="",
                                 seconds=round(time.perf_counter() - start, 2)))

RUN_START = time.perf_counter()
print(f"gensim {gensim.__version__} | datasets {_installed_version('datasets')} | numpy {np.__version__} | "
      f"pandas {pd.__version__} | workers {WORKERS} | seed {SEED} | QUICK_RUN {QUICK_RUN}")

In [ ]:
# 0.3 Paths with Google Drive fallback (+ a local work folder for large files)
# What this cell does:
#   - Mounts Google Drive (falls back to local folders if it fails).
#   - Creates the corpus / models / artifacts / labels / work folders.
#   - Helpers to copy files between Drive and the fast local work folder.
DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_DIRNAME = "lexical_combat_selftrained" + ("_quick" if QUICK_RUN else "")

@dataclass(frozen=True)
class Paths:
    """Where files live. persistent=False means nothing here survives the session."""
    root: Path
    corpus: Path       # processed corpus, phrase model, counts, corpus stats
    models: Path       # one folder per trained model (+ config.json)
    artifacts: Path    # search results, tuned params, holdout logs, report
    labels: Path       # annotator_*.xlsx / .csv (shared with the original notebook)
    work: Path         # local scratch: fast I/O for large files (never on Drive)
    persistent: bool

def mount_drive(use_drive: bool) -> bool:
    """Try to mount Drive; return True only if MyDrive is reachable."""
    if not use_drive:
        print("WARNING: USE_DRIVE = False — using local paths; nothing will persist.")
        return False
    try:
        from google.colab import drive
    except ImportError:
        print("WARNING: google.colab is not available — using local paths; nothing will persist.")
        return False
    try:
        drive.mount("/content/drive")
    except Exception as error:   # mount refused / auth failed
        print(f"WARNING: Drive mount failed ({error}) — using local paths; nothing will persist.")
        return False
    if not DRIVE_ROOT.is_dir():
        print("WARNING: Drive mounted but MyDrive not found — using local paths; nothing will persist.")
        return False
    return True

def resolve_paths(use_drive: bool) -> Paths:
    """Drive paths when mounted, else local paths under /content (or the working folder)."""
    base = Path("/content") if Path("/content").is_dir() else Path.cwd()
    work = base / (PROJECT_DIRNAME + "_work")
    if mount_drive(use_drive):
        root = DRIVE_ROOT / PROJECT_DIRNAME
        paths = Paths(root, root / "corpus", root / "models", root / "artifacts",
                      DRIVE_ROOT / "lexical_holdout_labels", work, True)
    else:
        root = base / PROJECT_DIRNAME
        paths = Paths(root, root / "corpus", root / "models", root / "artifacts", root / "labels", work, False)
    for folder in (paths.corpus, paths.models, paths.artifacts, paths.labels, paths.work):
        folder.mkdir(parents=True, exist_ok=True)
    return paths

def to_store(local_file: Path, folder: Path) -> Path:
    """Copy a finished local file to the persistent folder (no-op when it is already there)."""
    target = folder / local_file.name
    if local_file.resolve() != target.resolve():
        shutil.copyfile(local_file, target)
    return target

def from_store(name: str, folder: Path) -> Path:
    """Local copy of a stored file (copied once per session; streaming from Drive is slow)."""
    source, local = folder / name, PATHS.work / name
    if not local.is_file() or local.stat().st_size != source.stat().st_size:
        shutil.copyfile(source, local)
    return local

def file_mb(path: Path) -> float:
    """File or folder size in MB."""
    path = Path(path)
    if path.is_dir():
        return round(sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) / 1e6, 1)
    return round(path.stat().st_size / 1e6, 1) if path.exists() else 0.0

PATHS = resolve_paths(USE_DRIVE)
PERSISTENT = PATHS.persistent
LABEL_SEARCH_DIRS = [PATHS.labels] + ([Path("/content")] if Path("/content").is_dir() else [])
print(f"Paths root: {PATHS.root} | work: {PATHS.work} | PERSISTENT = {PERSISTENT}")
log_section("0. Setup", RUN_START)

## 1. Data registry
Pure data, no model calls: domains, anchors, vocabularies, the calibration / holdout / diagnostic sets, game stages and holdout labels. These are **copied verbatim from the original notebook** so the results compare directly with MiniLM. None of this data is ever written to the training corpus (Section 4 uses it only to measure coverage).
- **1.1** Domains, anchor sentences, example phrases, noise vocabulary and term pools.
- **1.2** Calibration cases, fresh gate vocabulary, holdout v2 and the old holdout (diagnostic only).
- **1.3** Game stages, boss names and the inline holdout annotations.

In [ ]:
# 1.1 Domains, anchors and vocabularies (pure data, no model calls)
# What this cell does:
#   - Defines the 4 domains and their anchor sentences (used by the domain gate).
#   - Example phrases, out-of-domain cases and the original 19 test pairs.
#   - Noise vocabulary (everyday nouns) and curated term pools per domain.
_t_s1 = time.perf_counter()
DOMAINS = ["Business", "Technology", "Science", "Arts"]

# Curated phrase bank: example guesses per domain (gate data and term pools).
# Each entry is an example guess associated with one domain.
# These examples support demonstration and evaluation; expand them with human-reviewed data.
DOMAIN_EXAMPLES = {
    "Business": [
        "executive", "ceo", "director", "manager", "revenue", "profit",
        "business logic", "market strategy", "financial analysis",
        "shareholder value", "corporate leadership", "sales forecast"
    ],
    "Technology": [
        "algorithm", "software", "database", "network", "compiler",
        "logic gate", "machine learning", "source code", "cloud computing",
        "data structure", "neural network", "operating system"
    ],
    "Science": [
        "catalyst", "molecule", "quantum mechanics", "chemical reaction",
        "cell biology", "gravity", "atomic structure", "genetic mutation",
        "laboratory experiment", "thermodynamics", "electron", "ecosystem"
    ],
    "Arts": [
        "painting", "sculpture", "abstract painting", "classical music",
        "stage performance", "poetry", "portrait art", "modern dance",
        "cinematography", "watercolor technique", "orchestra", "theater"
    ]
}


# Explicit out-of-domain examples: plausible English phrases that do not belong to
# the stated domain. They help test whether the soft domain gate blocks irrelevant inputs.
OUT_OF_DOMAIN_CASES = [
    ("banana", "Business"), ("sonata", "Business"), ("quantum mechanics", "Business"),
    ("quarterly earnings", "Technology"), ("abstract painting", "Technology"),
    ("shareholder value", "Science"), ("database", "Arts"), ("chemical reaction", "Arts"),
    ("market strategy", "Science"), ("sculpture", "Technology")
]

# Original 19 pair-level tests; kept only so fine-tuning (Appendix) excludes them.
TEST_CASES = [
    # Business target: executive
    ("executive", "ceo", "Business", "in_domain", "strong"),
    ("executive", "director", "Business", "in_domain", "strong"),
    ("executive", "business logic", "Business", "in_domain", "medium"),
    ("executive", "logic", "Business", "ambiguous", "medium"),
    ("executive", "logic gate", "Business", "out_of_domain", "blocked"),
    ("executive", "banana", "Business", "out_of_domain", "blocked"),
    # Technology target: algorithm
    ("algorithm", "data structure", "Technology", "in_domain", "strong"),
    ("algorithm", "source code", "Technology", "in_domain", "medium"),
    ("algorithm", "logic gate", "Technology", "in_domain", "strong"),
    ("algorithm", "business logic", "Technology", "ambiguous", "medium"),
    ("algorithm", "logic", "Technology", "ambiguous", "medium"),
    ("algorithm", "abstract painting", "Technology", "out_of_domain", "blocked"),
    # Science target: catalyst
    ("catalyst", "chemical reaction", "Science", "in_domain", "strong"),
    ("catalyst", "molecule", "Science", "in_domain", "medium"),
    ("catalyst", "market strategy", "Science", "out_of_domain", "blocked"),
    ("catalyst", "quantum mechanics", "Science", "in_domain", "medium"),
    # Arts target: painting (evaluation only; game stages below use three domains)
    ("painting", "abstract painting", "Arts", "in_domain", "strong"),
    ("painting", "sculpture", "Arts", "in_domain", "medium"),
    ("painting", "database", "Arts", "out_of_domain", "blocked")
]

# Rich descriptions provide context beyond isolated category labels.
# V2 anchor bank: exactly 8 anchors per domain. The old bank had 8/10/4/3
# (Business/Technology/Science/Arts); with max-over-anchors scoring, domains
# with more anchors won unrelated words more often (banana -> Technology 40%).
# New coverage: digital logic (logic gate), software architecture (business
# logic layer), chemical kinetics (catalyst), and a full Arts set.
DOMAIN_ANCHORS = {
    "Business": [
        "Business finance accounting financial reporting quarterly report annual report balance sheet income statement cash flow revenue profit budget audit forecast",
        "Business management leadership executive chief executive officer CEO president director manager supervisor board of directors chairman team organization",
        "Business operations strategy planning human resources payroll hiring project management sales marketing procurement supply chain logistics customer service",
        "Commerce and markets include companies, corporations, startups, entrepreneurs, investors, shareholders, the stock market, trade, competition, mergers and acquisitions.",
        "Management roles include director, managing director, chief executive officer, chief financial officer, chief operating officer, department manager and team leader.",
        "Corporate governance covers the board of directors, executives, stakeholders, company policy, compliance, decision-making and business performance.",
        "Financial reporting covers quarterly reports, annual reports, financial statements, balance sheets, income statements, audits and revenue forecasts.",
        "Economics and finance include investment, banking, loans, interest, assets, liabilities, pricing, costs and profit margins."
    ],
    "Technology": [
        "Technology computing computer science software development programming source code debugging compiler API application database",
        "Algorithms and data structures array linked list stack queue tree graph hash table heap sorting searching recursion time complexity pseudocode",
        "Artificial intelligence AI machine learning deep learning neural network model training inference natural language processing computer vision",
        "Computer hardware motherboard CPU processor GPU RAM memory storage hard drive SSD circuit board peripherals input output devices",
        "Digital logic and electronics logic gate AND OR NOT XOR boolean logic binary circuit transistor flip-flop microcontroller embedded system",
        "Cybersecurity and networks encryption cryptography firewall router switch protocol IP address authentication malware vulnerability",
        "Software architecture includes the backend, the frontend, the business logic layer, the data access layer, microservices, APIs, cloud computing, servers and deployment.",
        "An algorithm is a finite sequence of computational instructions used to solve a problem or process data in a software system."
    ],
    "Science": [
        "Science scientific research physics chemistry biology experiments hypotheses theories measurements natural phenomena",
        "Chemistry studies molecules, atoms, elements, compounds, chemical reactions, chemical bonds, acids, bases and the periodic table.",
        "Chemical kinetics covers catalysts, enzymes, reaction rates, activation energy, reagents, oxidation, reduction, equilibrium and thermodynamics.",
        "Physics includes mechanics, gravity, electricity, magnetism, quantum mechanics, particles, electrons, forces, energy and motion.",
        "Biology includes genetics, DNA, cells, proteins, organisms, evolution, ecosystems, photosynthesis and molecular biology.",
        "Laboratory science uses microscopes, test tubes, beakers, samples, data collection, the scientific method, observation and peer review.",
        "Earth and space science covers geology, astronomy, planets, stars, climate, the atmosphere, oceans and natural forces.",
        "Science keywords atom molecule electron proton neutron catalyst enzyme gene cell energy force gravity laboratory experiment"
    ],
    "Arts": [
        "Arts visual art painting sculpture drawing illustration design creativity aesthetics artist gallery museum",
        "Painting is a visual art form that uses pigment, color, composition, canvas, brushwork and artistic expression.",
        "Music includes melody, harmony, rhythm, songs, composers, orchestras, symphonies, opera, concerts and instruments such as guitar and piano.",
        "The performing arts include theater, drama, acting, stage performance, ballet, modern dance, choreography and musicals.",
        "Literature and writing include poetry, novels, short stories, poems, authors, storytelling, fiction and literary style.",
        "Film and photography include cinema, movies, cinematography, the film director, screenplays, camera shots and photography.",
        "Art history and styles include the renaissance, impressionism, abstract art, portraits, landscapes, still life and modern art.",
        "Crafts and design include graphic design, fashion design, pottery, ceramics, calligraphy, printmaking and textile art."
    ]
}


# Everyday nouns with no game-domain sense. Words with a common technical,
# business, science or art meaning (apple, cookie, cloud, tree, window, plate,
# bucket, forest, lake, candle, zipper, ...) are deliberately left out.
NOISE_VOCAB = [
    "banana", "grape", "lemon", "mango", "peach", "cherry", "watermelon", "pineapple",
    "strawberry", "potato", "carrot", "onion", "tomato", "cabbage", "pumpkin", "rice",
    "bread", "cheese", "butter", "soup", "sandwich", "pizza", "noodle", "pancake",
    "coffee", "tea", "juice", "milk", "honey", "popcorn", "cupcake", "lollipop",
    "dog", "cat", "horse", "cow", "sheep", "rabbit", "duck", "frog", "tiger",
    "elephant", "giraffe", "monkey", "squirrel", "butterfly", "penguin", "dolphin",
    "turtle", "parrot", "chair", "sofa", "pillow", "blanket", "curtain", "carpet",
    "cup", "spoon", "fork", "kettle", "broom", "towel", "soap", "shampoo",
    "toothbrush", "shoe", "sock", "hat", "scarf", "glove", "jacket", "umbrella",
    "backpack", "bicycle", "skateboard", "kite", "doll", "teddy bear", "beach",
    "river", "mountain", "meadow", "desert", "island", "rain", "snow", "sunset",
    "rainbow", "garden", "flower", "grass", "sand", "pebble", "door", "roof",
    "staircase", "kitchen", "bathroom", "bedroom", "picnic", "holiday", "birthday",
    "wedding", "breakfast", "lunch", "dinner", "nap", "sneeze", "yawn", "hug",
    "playground", "sandcastle", "snowman", "balloon", "ladder", "fence", "mailbox",
    "doorbell", "slipper", "raincoat",
]

# Curated in-domain terms. Used for the per-target ceiling and for hints.
# Holdout guesses are intentionally NOT listed here.
DOMAIN_VOCAB_EXTRA = {
    "Business": [
        "chief executive officer", "president", "vice president", "chairman",
        "managing director", "board of directors", "executive officer", "boss",
        "leadership", "management", "administrator", "entrepreneur", "investor",
        "stakeholder", "budget", "salary", "merger", "acquisition", "marketing",
        "accounting", "invoice", "governance", "corporation", "headquarters",
    ],
    "Technology": [
        "program", "programming", "code", "sorting", "searching", "recursion",
        "pseudocode", "flowchart", "heuristic", "computation", "software engineering",
        "function", "loop", "binary search", "optimization", "processor", "router",
        "server", "artificial intelligence", "deep learning", "data mining",
        "automation", "cryptography", "debugging",
    ],
    "Science": [
        "enzyme", "reaction rate", "activation energy", "chemical compound",
        "oxidation", "reagent", "chemical bond", "acid", "hydrogen", "oxygen",
        "element", "photosynthesis", "evolution", "velocity", "energy", "dna",
        "hypothesis", "microscope", "periodic table", "accelerant", "laboratory",
        "compound",
    ],
    "Arts": [
        "canvas", "brushstroke", "oil painting", "watercolor", "fresco", "mural",
        "sketch", "drawing", "illustration", "artist", "painter", "gallery",
        "masterpiece", "color palette", "portrait", "landscape painting",
        "still life", "impressionism", "renaissance art", "ballet", "opera",
        "symphony", "novel", "photography", "composer",
    ],
}
# Broader general coverage (round 3). The old pools were built around the
# stage targets only, so new targets (manager, encryption, enzyme, melody) got
# a low ceiling and good guesses saturated at r = 1. These are general domain
# terms; holdout guesses are excluded automatically below.
DOMAIN_VOCAB_COVERAGE = {
    "Business": ["staff", "personnel", "recruitment", "workplace", "department", "organization",
                 "customer", "client", "sales", "logistics", "supply chain", "procurement",
                 "human resources", "performance review", "business plan"],
    "Technology": ["software development", "hardware", "network security", "authentication",
                   "malware", "firewall", "web development", "mobile app", "user interface",
                   "data analysis", "computer vision", "cybersecurity", "internet protocol",
                   "microchip", "operating system"],
    "Science": ["biochemistry", "molecular biology", "organic chemistry", "physiology",
                "microbiology", "cell", "gene", "organism", "atom", "particle", "magnetism",
                "nucleus", "chemical process", "glucose", "nutrient"],
    "Arts": ["musical note", "chord", "lyrics", "singer", "piano", "concert", "musician",
             "choreography", "poet", "sculptor", "film director", "sonata", "drama",
             "aesthetic", "pottery"],
}

# Broad field labels. Correct but lazy guesses -> generic penalty.
# The old R2 list also contained DOMAIN_SEEDS (director, manager, molecule,
# database, neural network, ...), which wrongly penalised the best answers.
GENERIC_TERMS_V2 = {
    "business", "commerce", "economy", "economics", "company", "industry", "finance",
    "money", "work", "job", "office", "technology", "tech", "computing", "computer",
    "computer science", "engineering", "digital", "internet", "it", "science",
    "scientific", "research", "chemistry", "physics", "biology", "nature",
    "art", "arts", "creativity", "culture", "music", "entertainment", "design",
}

# One target per domain: sets the median span floor and the damaging-leak check.
CALIBRATION_TARGETS = {"executive": "Business", "algorithm": "Technology",
                       "catalyst": "Science", "painting": "Arts"}

In [ ]:
# 1.2 Calibration, holdout and diagnostic sets (pure data)
# What this cell does:
#   - Calibration cases: (target, guess, domain, gate label, damage tier).
#   - Fresh gate vocabulary that is NOT written in the anchors (avoids circular tuning).
#   - Holdout v2 (all-new targets) and the old holdout (diagnostic use only).
#   - Label overrides, cross-domain ambiguous pairs and reference plays for pacing.
# ------------------------------------------------------------
# 1. Pair-level calibration cases (target, guess, domain, gate label, tier)
# ------------------------------------------------------------
# Tiers: strong | medium | weak | blocked.
TEST_CASES_V2 = [
    ("executive", "ceo", "Business", "in_domain", "strong"),
    ("executive", "director", "Business", "in_domain", "strong"),
    ("executive", "business logic", "Business", "in_domain", "weak"),
    ("executive", "logic", "Business", "out_of_domain", "blocked"),
    ("executive", "logic gate", "Business", "out_of_domain", "blocked"),
    ("executive", "banana", "Business", "out_of_domain", "blocked"),
    ("algorithm", "data structure", "Technology", "in_domain", "strong"),
    ("algorithm", "source code", "Technology", "in_domain", "medium"),
    ("algorithm", "logic gate", "Technology", "in_domain", "medium"),
    ("algorithm", "business logic", "Technology", "ambiguous", "weak"),
    ("algorithm", "logic", "Technology", "ambiguous", "weak"),
    ("algorithm", "abstract painting", "Technology", "out_of_domain", "blocked"),
    ("catalyst", "chemical reaction", "Science", "in_domain", "strong"),
    ("catalyst", "molecule", "Science", "in_domain", "medium"),
    ("catalyst", "market strategy", "Science", "out_of_domain", "blocked"),
    ("catalyst", "quantum mechanics", "Science", "in_domain", "weak"),
    ("painting", "artwork", "Arts", "in_domain", "strong"),
    ("painting", "sculpture", "Arts", "in_domain", "medium"),
    ("painting", "database", "Arts", "out_of_domain", "blocked"),
]

# Extra banded cases so every calibration target has >= 8 cases (damage calibration, LOTO-CV).
# Not in the term pools, the holdout or the reference plays.
DAMAGE_CASES_EXTRA = [
    ("executive", "senior management", "Business", "in_domain", "strong"),
    ("executive", "company boss", "Business", "in_domain", "strong"),
    ("executive", "board member", "Business", "in_domain", "medium"),
    ("executive", "senior official", "Business", "in_domain", "medium"),
    ("executive", "office worker", "Business", "in_domain", "weak"),
    ("executive", "sales team", "Business", "in_domain", "weak"),
    ("executive", "sunflower", "Business", "out_of_domain", "blocked"),
    ("executive", "symphony", "Business", "out_of_domain", "blocked"),
    ("algorithm", "computer instructions", "Technology", "in_domain", "strong"),
    ("algorithm", "problem solving", "Technology", "in_domain", "medium"),
    ("algorithm", "graph theory", "Technology", "in_domain", "medium"),
    ("algorithm", "calculator", "Technology", "in_domain", "weak"),
    ("algorithm", "computer screen", "Technology", "in_domain", "weak"),
    ("algorithm", "dinosaur", "Technology", "out_of_domain", "blocked"),
    ("algorithm", "ballet", "Technology", "out_of_domain", "blocked"),
    ("catalyst", "reaction speed", "Science", "in_domain", "strong"),
    ("catalyst", "chemical agent", "Science", "in_domain", "medium"),
    ("catalyst", "platinum", "Science", "in_domain", "medium"),
    ("catalyst", "chemistry lab", "Science", "in_domain", "medium"),
    ("catalyst", "thermodynamics", "Science", "in_domain", "weak"),
    ("catalyst", "electron", "Science", "in_domain", "weak"),
    ("catalyst", "teacup", "Science", "out_of_domain", "blocked"),
    ("catalyst", "stock price", "Science", "out_of_domain", "blocked"),
    ("painting", "brush stroke", "Arts", "in_domain", "strong"),
    ("painting", "picture", "Arts", "in_domain", "medium"),
    ("painting", "art gallery", "Arts", "in_domain", "medium"),
    ("painting", "photograph", "Arts", "in_domain", "medium"),
    ("painting", "poem", "Arts", "in_domain", "weak"),
    ("painting", "dance", "Arts", "in_domain", "weak"),
    ("painting", "tractor", "Arts", "out_of_domain", "blocked"),
    ("painting", "algebra", "Arts", "out_of_domain", "blocked"),
]

# ------------------------------------------------------------
# 2. Fresh gate-calibration vocabulary (not written in DOMAIN_ANCHORS,
#    DOMAIN_EXAMPLES, the term pools or the holdout; V2 re-checks this)
# ------------------------------------------------------------
FRESH_GATE_IN_DOMAIN = {
    "Business": ["bankruptcy", "franchise", "wholesale", "retail", "bookkeeping", "inflation",
                 "recession", "advertising", "negotiation", "contract", "partnership",
                 "consultant", "tariff", "commission", "workforce", "pension", "inventory",
                 "monopoly", "layoff", "accountant"],
    "Technology": ["keyboard", "browser", "laptop", "smartphone", "bandwidth", "javascript",
                   "pixel", "bluetooth", "search engine", "website", "chatbot", "robotics",
                   "virtual reality", "blockchain", "byte", "touchscreen", "printer", "modem",
                   "antivirus"],
    "Science": ["volcano", "fossil", "telescope", "galaxy", "magnet", "bacteria", "chromosome",
                "mitochondria", "isotope", "radiation", "friction", "momentum", "wavelength",
                "solvent", "titration", "earthquake", "vaccine", "neuron", "photon",
                "black hole", "inertia"],
    "Arts": ["violin", "choir", "jazz", "novelist", "playwright", "sonnet", "comedy", "tragedy",
             "mosaic", "origami", "graffiti", "animation", "costume", "actor", "documentary",
             "ballerina", "stained glass", "woodcarving", "hip hop"],
}
# Fresh words whose cross-domain row is defensible -> 'ambiguous' instead of 'out_of_domain'.
FRESH_CROSS_AMBIGUOUS = {
    ("pixel", "Arts"), ("animation", "Technology"), ("blockchain", "Business"),
    ("robotics", "Science"), ("virtual reality", "Arts"), ("neuron", "Technology"),
    ("advertising", "Arts"), ("magnet", "Technology"),
}
# Cross-domain phrases a player could defend in the listed domain (should pass, x0.75 / x0.5).
FRESH_AMBIGUOUS_PAIRS = [
    ("film producer", "Business"), ("art auction", "Business"), ("music industry", "Business"),
    ("tech company", "Business"), ("software startup", "Business"),
    ("biotech startup", "Science"), ("data scientist", "Science"), ("rocket engine", "Science"),
    ("solar panel", "Science"), ("pharmaceutical company", "Science"),
    ("bioinformatics", "Technology"), ("medical device", "Technology"),
    ("game design", "Technology"), ("digital art", "Technology"), ("lab equipment", "Technology"),
    ("sound engineer", "Arts"), ("building architect", "Arts"), ("video game", "Arts"),
    ("web design", "Arts"), ("science museum", "Arts"), ("graphic tablet", "Arts"),
]
# Everyday words NOT in NOISE_VOCAB (that list sets the debias background), out for every domain.
FRESH_NOISE_OUT = ["lighthouse", "sailboat", "cactus", "pajamas", "volleyball", "marshmallow",
                   "waterfall", "hammock", "crocodile", "necklace", "flamingo", "haystack"]

# ------------------------------------------------------------
# 3. Holdout v2 — balanced design, 4 domains x 7 guesses, all-new targets
# ------------------------------------------------------------
# Per domain: 4 in-domain (1 strong, 2 medium, 1 weak), 1 ambiguous (weak),
# 2 out-of-domain (1 tempting distractor, 1 everyday word). Every target and
# guess is absent from DOMAIN_ANCHORS, DOMAIN_EXAMPLES, the term pools,
# NOISE_VOCAB, calibration sets, fine-tuning curated pairs and the old holdout.
# Labels below are the design intent; Section 7 replaces them with human labels.
HOLDOUT_CASES_DRAFT = [
    ("brand", "trademark", "Business", "in_domain", "strong"),
    ("brand", "logo", "Business", "in_domain", "medium"),
    ("brand", "customer loyalty", "Business", "in_domain", "medium"),
    ("brand", "packaging", "Business", "in_domain", "weak"),
    ("brand", "fashion label", "Business", "ambiguous", "weak"),
    ("brand", "cowboy", "Business", "out_of_domain", "blocked"),
    ("brand", "jellyfish", "Business", "out_of_domain", "blocked"),
    ("email", "inbox", "Technology", "in_domain", "strong"),
    ("email", "spam filter", "Technology", "in_domain", "medium"),
    ("email", "mail server", "Technology", "in_domain", "medium"),
    ("email", "chat app", "Technology", "in_domain", "weak"),
    ("email", "newsletter", "Technology", "ambiguous", "weak"),
    ("email", "postcard", "Technology", "out_of_domain", "blocked"),
    ("email", "seashell", "Technology", "out_of_domain", "blocked"),
    ("comet", "asteroid", "Science", "in_domain", "strong"),
    ("comet", "orbit", "Science", "in_domain", "medium"),
    ("comet", "solar system", "Science", "in_domain", "medium"),
    ("comet", "light year", "Science", "in_domain", "weak"),
    ("comet", "space shuttle", "Science", "ambiguous", "weak"),
    ("comet", "horoscope", "Science", "out_of_domain", "blocked"),
    ("comet", "doorknob", "Science", "out_of_domain", "blocked"),
    ("cartoon", "comic strip", "Arts", "in_domain", "strong"),
    ("cartoon", "caricature", "Arts", "in_domain", "medium"),
    ("cartoon", "animated film", "Arts", "in_domain", "medium"),
    ("cartoon", "puppet show", "Arts", "in_domain", "weak"),
    ("cartoon", "television channel", "Arts", "ambiguous", "weak"),
    ("cartoon", "stock chart", "Arts", "out_of_domain", "blocked"),
    ("cartoon", "toothpaste", "Arts", "out_of_domain", "blocked"),
]
HOLDOUT_SELECTION_NOTES = pd.DataFrame([
    ("brand", "target", "Not in any anchor/pool; has a second sense (cattle brand) that tests polysemy."),
    ("brand", "trademark", "Near-synonym: checks that a true synonym reaches the strong band without r saturation."),
    ("brand", "logo / customer loyalty", "Clearly related, not synonyms: one visual, one marketing outcome."),
    ("brand", "packaging", "Loose marketing link: should only graze."),
    ("brand", "fashion label", "Arts/fashion phrase with a real business sense: tests ambiguous -> secondary."),
    ("brand", "cowboy", "Tempting distractor via the cattle-brand sense: tests damaging leak."),
    ("brand", "jellyfish", "Everyday word: plain negative control."),
    ("email", "target", "Common tech word outside the anchors; tests a target with an everyday sense."),
    ("email", "inbox", "Near-synonym in use."),
    ("email", "spam filter / mail server", "Related components of an email system."),
    ("email", "chat app", "Another messaging technology: loose link."),
    ("email", "newsletter", "Business/writing sense, delivered by email: ambiguous."),
    ("email", "postcard", "Tempting distractor: physical mail, not technology."),
    ("email", "seashell", "Everyday word: plain negative control."),
    ("comet", "target", "Astronomy is the thinnest Science area in pools and anchors: tests coverage."),
    ("comet", "asteroid", "Sibling object: closest valid answer."),
    ("comet", "orbit / solar system", "Related concepts at two levels (motion, system)."),
    ("comet", "light year", "Astronomy unit: weak link to comets."),
    ("comet", "space shuttle", "Technology/engineering with a science sense: ambiguous."),
    ("comet", "horoscope", "Tempting distractor: astrology is not science."),
    ("comet", "doorknob", "Everyday word: plain negative control."),
    ("cartoon", "target", "Visual Arts outside painting/music, which dominate the calibration data."),
    ("cartoon", "comic strip", "Near-synonym form."),
    ("cartoon", "caricature / animated film", "Related art forms: drawing style and moving image."),
    ("cartoon", "puppet show", "Children's entertainment: loose link."),
    ("cartoon", "television channel", "Technology/business phrase with an entertainment sense: ambiguous."),
    ("cartoon", "stock chart", "Tempting distractor: a drawn chart, but finance."),
    ("cartoon", "toothpaste", "Everyday word: plain negative control."),
], columns=["target", "guess", "why chosen"])

# Old holdout (manager / encryption / enzyme / melody). It was inspected while
# fixing the gate, so it is now a DIAGNOSTIC set (Section 5.2), not a holdout.
OLD_HOLDOUT_DIAG_CASES = [
    ("manager", "supervisor", "Business", "in_domain", "strong"),
    ("manager", "team leader", "Business", "in_domain", "strong"),
    ("manager", "employee", "Business", "in_domain", "medium"),
    ("manager", "payroll", "Business", "in_domain", "weak"),
    ("manager", "spreadsheet", "Business", "ambiguous", "weak"),
    ("manager", "orchestra conductor", "Business", "out_of_domain", "blocked"),
    ("manager", "giraffe", "Business", "out_of_domain", "blocked"),
    ("manager", "firmware", "Business", "out_of_domain", "blocked"),
    ("encryption", "cipher", "Technology", "in_domain", "strong"),
    ("encryption", "private key", "Technology", "in_domain", "strong"),
    ("encryption", "password", "Technology", "in_domain", "medium"),
    ("encryption", "hacker", "Technology", "in_domain", "medium"),
    ("encryption", "secret message", "Technology", "ambiguous", "weak"),
    ("encryption", "tulip", "Technology", "out_of_domain", "blocked"),
    ("encryption", "stock market", "Technology", "out_of_domain", "blocked"),
    ("enzyme", "protein", "Science", "in_domain", "strong"),
    ("enzyme", "metabolism", "Science", "in_domain", "medium"),
    ("enzyme", "amino acid", "Science", "in_domain", "medium"),
    ("enzyme", "digestion", "Science", "in_domain", "medium"),
    ("enzyme", "fermentation", "Science", "in_domain", "weak"),
    ("enzyme", "guitar", "Science", "out_of_domain", "blocked"),
    ("enzyme", "dividend", "Science", "out_of_domain", "blocked"),
    ("melody", "harmony", "Arts", "in_domain", "strong"),
    ("melody", "tune", "Arts", "in_domain", "strong"),
    ("melody", "rhythm", "Arts", "in_domain", "medium"),
    ("melody", "song", "Arts", "in_domain", "medium"),
    ("melody", "frequency", "Arts", "ambiguous", "weak"),
    ("melody", "spreadsheet", "Arts", "out_of_domain", "blocked"),
]


RELABEL_NOTES = pd.DataFrame([
    ("executive / logic", "ambiguous · medium", "out_of_domain · blocked",
     "Bare 'logic' has no business sense; a player would not expect it to hurt a Business boss."),
    ("executive / business logic", "in_domain · medium", "in_domain · weak",
     "Business-flavoured but not about executives: graze, not a solid hit."),
    ("algorithm / logic gate", "in_domain · strong", "in_domain · medium",
     "Hardware building block, related to but not a synonym of 'algorithm'."),
    ("algorithm / logic, business logic", "ambiguous · medium", "ambiguous · weak",
     "Loose overlap only; should chip, not hit."),
    ("catalyst / quantum mechanics", "in_domain · medium", "in_domain · weak",
     "Correct domain, wrong sub-field: a graze is the fair outcome."),
    ("painting / abstract painting", "in_domain · strong", "painting / artwork",
     "'abstract painting' contains the target word and is now rejected as input."),
], columns=["Case", "Old label", "New label", "Reason"])

# Round 3: review of CROSS_DOMAIN_AMBIGUOUS. Policy: 'ambiguous' = a player can
# name a real, common sense in the stage domain -> should PASS as secondary.
RELABEL_NOTES_V3 = pd.DataFrame([
    ("director / Arts", "ambiguous", "ambiguous (keep, should pass)",
     "'film director' is a common Arts sense; secondary x0.75 is fair."),
    ("neural network / Science", "ambiguous", "ambiguous (keep)",
     "Computational neuroscience is a real science sense."),
    ("machine learning / Science", "ambiguous", "ambiguous (keep)",
     "Data science / scientific ML is a real sense."),
    ("electron / Technology", "ambiguous", "ambiguous (keep)", "Electronics."),
    ("algorithm / Science", "ambiguous", "ambiguous (keep)", "Computer science / computational science."),
    ("thermodynamics / Technology", "ambiguous", "ambiguous (keep)", "Engine and heat-system engineering."),
    ("network / Science", "ambiguous", "out_of_domain",
     "Bare 'network' points to computing; the science senses need a modifier."),
    ("cinematography / Technology", "ambiguous", "out_of_domain",
     "Camera craft is an Arts skill; the technology link is indirect."),
    ("ecosystem / Business", "ambiguous", "out_of_domain",
     "'business ecosystem' is jargon; bare 'ecosystem' is biology."),
], columns=["Case", "Old label", "New label", "Reason"])


# Per-hit damage bands in percent of boss HP. Tiers: Critical >=38, Strong >=26,
# Solid >=14, Graze <14. "strong" therefore accepts Strong or Critical.
TIER_BANDS_V2 = {"strong": (30, 45), "medium": (14, 30), "weak": (4, 14), "blocked": (0, 0)}


GATE_LABEL_OVERRIDES = {
    ("logic", "Technology"): "ambiguous", ("logic", "Business"): "out_of_domain",
    ("business logic", "Business"): "in_domain", ("business logic", "Technology"): "ambiguous",
    ("logic gate", "Technology"): "in_domain", ("logic gate", "Business"): "out_of_domain",
    ("strategy", "Business"): "in_domain", ("strategy", "Technology"): "ambiguous",
    ("model", "Science"): "ambiguous", ("model", "Arts"): "ambiguous",
}
CROSS_DOMAIN_AMBIGUOUS = {
    ("director", "Arts"), ("neural network", "Science"), ("machine learning", "Science"),
    ("electron", "Technology"), ("algorithm", "Science"), ("thermodynamics", "Technology"),
}
CROSS_DOMAIN_RELABELLED_OUT = {("network", "Science"), ("cinematography", "Technology"),
                               ("ecosystem", "Business")}


REFERENCE_PLAYS = {
    "executive": {
        "expert": (["ceo", "president", "managing director", "chairman"], (3, 4)),
        "typical": (["banana", "logic", "business logic", "market strategy", "director",
                     "manager", "leadership", "revenue", "ceo", "accounting"], (5, 8)),
    },
    "algorithm": {
        "expert": (["pseudocode", "sorting", "binary search", "recursion"], (3, 4)),
        "typical": (["banana", "logic", "programming", "source code", "software", "database",
                     "data structure", "sorting", "machine learning", "neural network"], (5, 8)),
    },
    "catalyst": {
        "expert": (["enzyme", "chemical reaction", "reaction rate", "activation energy"], (3, 4)),
        "typical": (["banana", "market strategy", "quantum mechanics", "molecule", "electron",
                     "thermodynamics", "chemistry", "chemical reaction", "gravity",
                     "reaction rate"], (5, 8)),
    },
}
PLAYER_LOG_GUESSES_V2 = {
    "executive": ["watermelon", "captain", "marketing", "management", "accounting",
                  "governor", "managing director", "leadership", "corporate officer", "ceo"],
    "algorithm": ["sorting", "programming", "hard drive", "symphony", "blueprint",
                  "software", "database", "neural network"],
    "catalyst": ["chemistry", "molecule", "electron", "thermodynamics", "chemical reaction"],
}

In [ ]:
# 1.3 Game data and holdout annotations (pure data)
# What this cell does:
#   - The 3 fixed stages of classic mode and the boss names.
#   - Holdout annotations used as human labels in Section 10.

# The three fixed stages of classic mode.
GAME_STAGES = [
    {"stage": 1, "enemy": "Corporate Titan", "target": "executive", "domain": "Business", "hp": 180},
    {"stage": 2, "enemy": "Cyber Sentinel", "target": "algorithm", "domain": "Technology", "hp": 200},
    {"stage": 3, "enemy": "Alchemist Golem", "target": "catalyst", "domain": "Science", "hp": 200},
]

# Boss names per topic for random mode.
BOSS_NAMES = {
    "Business": ["Corporate Titan", "Boardroom Baron", "Market Hydra"],
    "Technology": ["Cyber Sentinel", "Kernel Wraith", "Firewall Golem"],
    "Science": ["Alchemist Golem", "Quantum Specter", "Nebula Warden"],
    "Arts": ["Muse Phantom", "Canvas Chimera", "Symphony Siren"],
}

# Holdout labels: (gate_label, tier) per item, same order as HOLDOUT_CASES_DRAFT.
# Files named annotator_<n>.xlsx / .csv (Section 7) override entries with the same name.
GATE_CHOICES = ["in_domain", "ambiguous", "out_of_domain"]
TIER_CHOICES = ["strong", "medium", "weak", "blocked"]
HOLDOUT_ANNOTATIONS = {
    # Annotator 1 (project owner), labelled independently on 2026-11-17 via annotator_1.xlsx.
    "annotator_1": [
        ("in_domain", "strong"),        #  0 brand / trademark
        ("in_domain", "strong"),        #  1 brand / logo
        ("in_domain", "medium"),        #  2 brand / customer loyalty
        ("in_domain", "medium"),        #  3 brand / packaging
        ("ambiguous", "weak"),          #  4 brand / fashion label
        ("out_of_domain", "blocked"),   #  5 brand / cowboy
        ("out_of_domain", "blocked"),   #  6 brand / jellyfish
        ("in_domain", "strong"),        #  7 email / inbox
        ("in_domain", "strong"),        #  8 email / spam filter
        ("in_domain", "strong"),        #  9 email / mail server
        ("in_domain", "medium"),        # 10 email / chat app
        ("in_domain", "medium"),        # 11 email / newsletter
        ("out_of_domain", "blocked"),   # 12 email / postcard
        ("out_of_domain", "blocked"),   # 13 email / seashell
        ("in_domain", "strong"),        # 14 comet / asteroid
        ("in_domain", "strong"),        # 15 comet / orbit
        ("in_domain", "medium"),        # 16 comet / solar system
        ("in_domain", "medium"),        # 17 comet / light year
        ("ambiguous", "weak"),          # 18 comet / space shuttle
        ("out_of_domain", "blocked"),   # 19 comet / horoscope
        ("out_of_domain", "blocked"),   # 20 comet / doorknob
        ("in_domain", "strong"),        # 21 cartoon / comic strip
        ("in_domain", "strong"),        # 22 cartoon / caricature
        ("in_domain", "strong"),        # 23 cartoon / animated film
        ("in_domain", "medium"),        # 24 cartoon / puppet show
        ("ambiguous", "weak"),          # 25 cartoon / television channel
        ("out_of_domain", "blocked"),   # 26 cartoon / stock chart
        ("out_of_domain", "blocked"),   # 27 cartoon / toothpaste
    ],
    "annotator_2": [None] * len(HOLDOUT_CASES_DRAFT),
    "annotator_3": [None] * len(HOLDOUT_CASES_DRAFT),
}
log_section("1. Data registry", _t_s1)

## 2. Text utilities and data checks
- **2.1** `clean_text()` — the same function the game uses (lowercase, non-letters → spaces). Section 4 also uses it to preprocess the corpus, which prevents **train/serve skew**. Plus small helpers for input validation, term pools and anchor matching.
- **2.2** Derived phrase sets, the gate calibration set (built so it is not circular with the anchors) and leakage checks, as in the original notebook.

In [ ]:
# 2.1 Text utilities (pure functions, used by the engine and the data checks)
# What this cell does:
#   - clean_text(): the single text normaliser shared by the game and the corpus.
#   - Helpers for duplicate checks, generic terms, term pools and input validation.
#   - anchor_domains_of(): finds which domains' anchors contain a phrase.
_t_s2 = time.perf_counter()

def clean_text(text: str) -> str:
    """Lowercase, replace non-letters with spaces, collapse spaces ('data-base' -> 'data base')."""
    text = str(text).lower().strip()
    text = "".join(ch if (ch.isalpha() or ch.isspace()) else " " for ch in text)
    return " ".join(text.split())

def duplicate_key(text: str) -> str:
    """Key that ignores word order and simple plurals ('business logics' == 'logic business')."""
    words = []
    for word in clean_text(text).split():
        if len(word) > 3 and word.endswith("s") and not word.endswith("ss"):
            word = word[:-1]
        words.append(word)
    return " ".join(sorted(words))

def word_keys(text: str) -> set:
    """Normalised word keys ('ceos' -> 'ceo') used for target-word checks."""
    return {duplicate_key(w) for w in clean_text(text).split()}

def is_generic_term(cleaned_guess: str, generic_terms) -> bool:
    """True for broad field labels such as 'science' or 'business'."""
    return (cleaned_guess in generic_terms
            or duplicate_key(cleaned_guess) in {duplicate_key(t) for t in generic_terms})

def domain_term_pool(domain: str, examples: dict, vocab: dict, generic_terms) -> list:
    """De-duplicated, non-generic in-domain terms for one domain."""
    out, seen = [], set()
    for term in list(examples.get(domain, [])) + vocab.get(domain, []):
        cleaned = clean_text(term)
        if cleaned and cleaned not in seen and not is_generic_term(cleaned, generic_terms):
            seen.add(cleaned)
            out.append(cleaned)
    return out

def validate_player_input(guess: str, target: str, used_guesses, block_target_word: bool = True) -> tuple:
    """Clean and validate a 1–3 word guess. Returns (is_valid, cleaned, reason)."""
    cleaned = clean_text(guess)
    target_clean = clean_text(target)
    if not cleaned:
        return False, cleaned, "Please enter 1–3 English words."
    if not 1 <= len(cleaned.split()) <= 3:
        return False, cleaned, "Your guess must contain between 1 and 3 words."
    if duplicate_key(cleaned) == duplicate_key(target_clean):
        return False, cleaned, "Your guess cannot be the target word itself."
    if block_target_word and (word_keys(cleaned) & word_keys(target_clean)):
        return False, cleaned, "Your guess cannot contain the target word."
    if cleaned in used_guesses:
        return False, cleaned, f"'{cleaned}' was already used in this battle."
    if duplicate_key(cleaned) in {duplicate_key(u) for u in used_guesses}:
        return False, cleaned, f"'{cleaned}' is too similar to a guess already used in this battle."
    return True, cleaned, ""

def norm_tokens(text: str) -> list:
    """Lowercase tokens with simple plural stripping ('reactions' -> 'reaction')."""
    out = []
    for word in clean_text(text).split():
        if len(word) > 3 and word.endswith("s") and not word.endswith("ss"):
            word = word[:-1]
        out.append(word)
    return out

ANCHOR_TOKEN_SEQS = {d: [norm_tokens(a) for a in DOMAIN_ANCHORS[d]] for d in DOMAINS}

def anchor_domains_of(phrase: str) -> list:
    """Domains whose anchor text contains the phrase as a contiguous token run."""
    toks = norm_tokens(phrase)
    hits = []
    for d, seqs in ANCHOR_TOKEN_SEQS.items():
        if any(seq[i:i + len(toks)] == toks for seq in seqs for i in range(len(seq) - len(toks) + 1)):
            hits.append(d)
    return hits

In [ ]:
# 2.2 Derived phrase sets, gate calibration set and leakage checks (no model calls)
# What this cell does:
#   - Builds term pools without any holdout phrase.
#   - Builds the gate calibration set (in-domain / ambiguous / out-of-domain phrases).
#   - Runs leakage checks and prints the set sizes.

# Term pools: curated extras + general coverage, minus every holdout phrase.
HOLDOUT_PHRASES = {clean_text(p) for case in HOLDOUT_CASES_DRAFT + OLD_HOLDOUT_DIAG_CASES for p in case[:2]}
POOL_VOCAB = {d: DOMAIN_VOCAB_EXTRA[d] + [t for t in DOMAIN_VOCAB_COVERAGE[d] if clean_text(t) not in HOLDOUT_PHRASES]
              for d in DOMAINS}
CALIBRATION_PHRASES = (
    {clean_text(p) for case in TEST_CASES_V2 + DAMAGE_CASES_EXTRA for p in case[:2]}
    | {clean_text(p) for terms in FRESH_GATE_IN_DOMAIN.values() for p in terms}
    | {clean_text(p) for p, _ in FRESH_AMBIGUOUS_PAIRS}
    | {clean_text(p) for p in FRESH_NOISE_OUT}
)
EVAL_PHRASES = HOLDOUT_PHRASES | CALIBRATION_PHRASES   # calibration + holdout phrases (coverage report)

test_df_v2 = pd.DataFrame(TEST_CASES_V2, columns=[
    "target", "guess", "required_domain", "human_label", "human_expected_tier"])
damage_cases_df = pd.DataFrame(TEST_CASES_V2 + DAMAGE_CASES_EXTRA, columns=[
    "target", "guess", "required_domain", "human_label", "human_expected_tier"])
DOMAIN_REFERENCE_TARGET = {d: t for t, d in CALIBRATION_TARGETS.items()}

# Fresh vocabulary: drop any term that also appears in anchors, examples, holdout, pools or noise.
_examples = {clean_text(p) for ps in DOMAIN_EXAMPLES.values() for p in ps}
_pool_terms = {t for d in DOMAINS for t in domain_term_pool(d, DOMAIN_EXAMPLES, POOL_VOCAB, GENERIC_TERMS_V2)}
_noise_set = {clean_text(w) for w in NOISE_VOCAB}

def fresh_violation(phrase: str):
    """Reason a fresh calibration term overlaps another set, else None."""
    p = clean_text(phrase)
    if anchor_domains_of(p): return "in anchors"
    if p in _examples: return "in DOMAIN_EXAMPLES"
    if p in HOLDOUT_PHRASES: return "in holdout"
    if p in _pool_terms: return "in term pool"
    if p in _noise_set: return "in NOISE_VOCAB"
    return None

fresh_dropped = []
FRESH_IN_DOMAIN_CLEAN = {}
for d, terms in FRESH_GATE_IN_DOMAIN.items():
    FRESH_IN_DOMAIN_CLEAN[d] = []
    for t in terms:
        why = fresh_violation(t)
        if why:
            fresh_dropped.append((t, d, why))
        else:
            FRESH_IN_DOMAIN_CLEAN[d].append(clean_text(t))
FRESH_AMBIGUOUS_CLEAN = [(clean_text(p), d) for p, d in FRESH_AMBIGUOUS_PAIRS if not fresh_violation(p)]
FRESH_NOISE_CLEAN = [clean_text(w) for w in FRESH_NOISE_OUT if not fresh_violation(w)]
fresh_dropped += [(p, d, fresh_violation(p)) for p, d in FRESH_AMBIGUOUS_PAIRS if fresh_violation(p)]
fresh_dropped += [(w, "all", fresh_violation(w)) for w in FRESH_NOISE_OUT if fresh_violation(w)]
for d, terms in FRESH_IN_DOMAIN_CLEAN.items():
    assert len(terms) >= 15, f"{d}: only {len(terms)} fresh terms left; add more in Section 1.2."

def build_gate_calibration_set() -> pd.DataFrame:
    """One label per (phrase, domain); later sources override earlier ones."""
    labels, sources = {}, {}
    def put(phrase, domain, label, source):
        labels[(clean_text(phrase), domain)] = label
        sources[(clean_text(phrase), domain)] = source
    for domain, phrases in DOMAIN_EXAMPLES.items():          # lowest priority
        for phrase in phrases:
            for other in DOMAINS:
                put(phrase, other, "in_domain" if other == domain else "out_of_domain",
                    "DOMAIN_EXAMPLES")
    for phrase, domain in OUT_OF_DOMAIN_CASES:
        put(phrase, domain, "out_of_domain", "OUT_OF_DOMAIN_CASES")
    for phrase, domain in CROSS_DOMAIN_AMBIGUOUS:
        put(phrase, domain, "ambiguous", "CROSS_DOMAIN_AMBIGUOUS")
    for phrase, domain in CROSS_DOMAIN_RELABELLED_OUT:
        put(phrase, domain, "out_of_domain", "CROSS_DOMAIN_RELABELLED")
    for (phrase, domain), label in GATE_LABEL_OVERRIDES.items():
        put(phrase, domain, label, "GATE_LABEL_OVERRIDES")
    for domain, terms in FRESH_IN_DOMAIN_CLEAN.items():
        for term in terms:
            for other in DOMAINS:
                if other == domain:
                    put(term, other, "in_domain", "FRESH_IN_DOMAIN")
                elif (term, other) in FRESH_CROSS_AMBIGUOUS:
                    put(term, other, "ambiguous", "FRESH_CROSS_AMBIGUOUS")
                else:
                    put(term, other, "out_of_domain", "FRESH_CROSS_OUT")
    for phrase, domain in FRESH_AMBIGUOUS_CLEAN:
        put(phrase, domain, "ambiguous", "FRESH_AMBIGUOUS")
    for word in FRESH_NOISE_CLEAN:
        for domain in DOMAINS:
            put(word, domain, "out_of_domain", "FRESH_NOISE_OUT")
    for _, guess, domain, label, _ in TEST_CASES_V2 + DAMAGE_CASES_EXTRA:   # highest priority
        put(guess, domain, label, "BANDED_CASES")
    frame = pd.DataFrame([{"phrase": p, "domain": d, "label": l, "source": sources[(p, d)]}
                          for (p, d), l in labels.items()])
    frame["anchor_domains"] = frame["phrase"].map(lambda p: ",".join(anchor_domains_of(p)))
    frame["anchor_echo"] = frame["anchor_domains"] != ""
    return frame

gate_set_all_df = build_gate_calibration_set()
gate_set_df = gate_set_all_df[~gate_set_all_df["anchor_echo"]].reset_index(drop=True)   # tuning
anchor_echo_df = gate_set_all_df[gate_set_all_df["anchor_echo"]].reset_index(drop=True)  # sanity only

print(f"Registry: {len(TEST_CASES_V2) + len(DAMAGE_CASES_EXTRA)} banded cases | "
      f"{sum(map(len, FRESH_GATE_IN_DOMAIN.values()))} fresh in-domain terms | "
      f"{len(FRESH_AMBIGUOUS_PAIRS)} fresh ambiguous pairs | {len(HOLDOUT_CASES_DRAFT)} holdout cases")
print("Holdout per domain:", pd.Series([d for _, _, d, _, _ in HOLDOUT_CASES_DRAFT]).value_counts().to_dict())
_leaks = sorted({g for _, g, _, _, _ in HOLDOUT_CASES_DRAFT
                 if clean_text(g) in _pool_terms or clean_text(g) in set(gate_set_all_df["phrase"])})
print("Relabelled TEST_CASES (round 2):")
display(RELABEL_NOTES)
print("CROSS_DOMAIN_AMBIGUOUS review (round 3):")
display(RELABEL_NOTES_V3)
print("Banded cases per target (need >= 8):")
display(damage_cases_df.groupby(["target", "human_expected_tier"]).size().unstack(fill_value=0)
        .assign(total=lambda f: f.sum(axis=1)))
print(f"Gate set: {len(gate_set_all_df)} rows -> tuning {len(gate_set_df)} | anchor-echo (sanity only) "
      f"{len(anchor_echo_df)}")
display(gate_set_df.groupby(["label", "source"]).size().rename("rows").reset_index())
print("Fresh in-domain terms per domain:", {d: len(t) for d, t in FRESH_IN_DOMAIN_CLEAN.items()})
if fresh_dropped:
    print("Fresh terms dropped by the overlap check:", fresh_dropped)
_anchor_holdout = [g for _, g, _, _, _ in HOLDOUT_CASES_DRAFT if anchor_domains_of(g)]
print(f"Holdout leakage into pools / gate set: {_leaks if _leaks else 'none'} | "
      f"holdout guesses written in anchors: {_anchor_holdout if _anchor_holdout else 'none'}")
print("Holdout design (why each word was chosen):")
display(HOLDOUT_SELECTION_NOTES)
log_section("2. Text utilities and checks", _t_s2)

## 3. Engine and calibration functions
**3.1** `EngineConfig` + `LexicalCombatEngine` from the original notebook. The engine only calls the model through `self.model.encode(...)`, so our adapter can replace MiniLM directly. Two changes:
1. It detects **words the model does not know** (OOV) and tells the player "Unknown word" without counting the turn.
2. It drops noise-vocab / term-pool words the model cannot represent (zero vectors would distort the background and the percentiles).

**3.2a–c** Gate diagnosis, gate grid, damage grid, plateau selection, grouped CV and LOTO-CV from the original notebook. They now take the `grid` as a parameter, and the grid is **expanded automatically** when a chosen value sits on its edge. Cosine values of word embeddings are distributed very differently from MiniLM's, so the old thresholds do not transfer: every model is calibrated from scratch.

In [ ]:
# 3.1 Engine: EngineConfig (every tunable parameter) + LexicalCombatEngine (all scoring)
# What this cell does:
#   - EngineConfig: every tunable parameter (gate thresholds, damage curve, tiers).
#   - LexicalCombatEngine: domain gate, target relevance, damage and battle simulation.
#   - Handles unknown words (OOV) and drops terms the model cannot represent.
# Copied from the original notebook. Changes: (1) unknown-word check in evaluate_attack (the
# self-trained encoders expose is_known), (2) refresh() drops noise / pool terms the model cannot
# represent, so zero vectors never enter a background, percentile or ceiling.
_t_s3 = time.perf_counter()

@dataclass(frozen=True)
class EngineConfig:
    """Tunable parameters. Defaults are the pre-calibration values; Section 5 replaces some."""
    # Domain gate (debiased score = top-k anchor cosine minus the domain's noise background)
    anchor_topk: int = 2                     # mean of the k best anchors per domain
    gate_primary_min: float = 0.20           # primary: required domain is #1 and >= this
    gate_secondary_min: float = 0.08         # secondary / weak: required domain >= this
    gate_secondary_max_gap: float = 0.04     # secondary: may trail the #1 domain by <= this
    gate_weak_max_gap: float = 0.10          # weak secondary: may trail by <= this
    domain_primary_bonus: float = 1.00       # damage multiplier per gate result
    domain_secondary_multiplier: float = 0.75
    domain_weak_secondary_multiplier: float = 0.50
    display_softmax_temperature: float = 0.05  # display only, not used for decisions
    # Target relevance r = (cos - noise) / (ceiling - noise)
    noise_percentile: int = 90               # noise floor = percentile of cos(target, everyday nouns)
    ceiling_topk: int = 3                    # ceiling = mean of the top-k pool cosines
    min_calibration_span: float = 0.15       # fixed span floor (min_span_mode == "fixed")
    min_span_mode: str = "median"            # "median": floor = median span of calibration targets
    primary_graze_floor: bool = True         # primary guess above median noise gets chip damage
    primary_graze_noise_pct: int = 50
    # Damage curve on r in [0, 1]
    relevance_min: float = 0.10              # r below this -> no effect
    damage_gamma: float = 1.0                # >1 stricter mid-range, <1 more generous
    graze_min_percent: float = 5             # damage at r == relevance_min
    max_hit_percent: float = 45              # damage at r == 1 (per-hit cap)
    chip_percent: float = 4                  # minimum damage of an allowed, relevant hit
    # Penalties and input rules
    generic_penalty: float = 0.60            # broad field labels ("science")
    echo_relative: float = 0.60              # echo if cos to an earlier hit is this far from noise to 1.0
    echo_penalty: float = 0.50
    block_target_word_in_guess: bool = True  # a guess may not contain any word of the target
    tier_cuts: tuple = ((38, "CRITICAL STRIKE"), (26, "STRONG HIT"), (14, "SOLID HIT"))

@dataclass(frozen=True)
class EngineData:
    """Vocabularies the engine encodes. Built once from Sections 1–2."""
    domains: tuple
    anchors: dict
    noise_vocab: tuple
    examples: dict
    pool_vocab: dict
    generic_terms: frozenset
    calibration_targets: dict

GATE_LABEL_OF_KIND = {"primary": "in_domain", "secondary": "ambiguous",
                      "weak_secondary": "ambiguous", "out_of_domain": "out_of_domain"}

def kinds_to_labels(kinds) -> np.ndarray:
    """Map gate kinds to the human label vocabulary."""
    kinds = np.asarray(kinds)
    return np.where(kinds == "primary", "in_domain",
                    np.where(kinds == "out_of_domain", "out_of_domain", "ambiguous"))

def topk_mean(similarities, k: int):
    """Mean of the k largest values along the last axis."""
    sims = np.asarray(similarities, dtype=float)
    k = max(1, min(int(k), sims.shape[-1]))
    return np.sort(sims, axis=-1)[..., -k:].mean(axis=-1)

def normalized_relevance(similarity, noise, ceiling):
    """r = (cos - noise) / (ceiling - noise), clipped to [0, 1]."""
    return np.clip((np.asarray(similarity, dtype=float) - noise) / (ceiling - noise), 0.0, 1.0)

def gate_kind_arrays(required, lead, primary_min, secondary_min, secondary_gap, weak_gap) -> np.ndarray:
    """Vectorised gate. lead = required score - best other domain score."""
    required = np.asarray(required, dtype=float)
    lead = np.asarray(lead, dtype=float)
    is_primary = (required >= primary_min) & (lead >= 0)
    passes = required >= secondary_min
    is_secondary = ~is_primary & passes & (lead >= -secondary_gap)
    is_weak = ~is_primary & ~is_secondary & passes & (lead >= -weak_gap)
    return np.select([is_primary, is_secondary, is_weak],
                     ["primary", "secondary", "weak_secondary"], "out_of_domain")

def timed_call(function, *args, **kwargs):
    """Run a function; return (result, elapsed milliseconds)."""
    start = time.perf_counter()
    result = function(*args, **kwargs)
    return result, (time.perf_counter() - start) * 1000

class LexicalCombatEngine:
    """Owns the model, anchor embeddings, backgrounds, pools, target cache and span floor."""

    def __init__(self, model, data: EngineData, config: EngineConfig = EngineConfig()):
        self.data = data
        self._config = config
        self.refresh(model)

    # ---- configuration -------------------------------------------------
    @property
    def config(self) -> EngineConfig:
        return self._config

    @config.setter
    def config(self, new_config: EngineConfig) -> None:
        """Apply new parameters; the span floor depends on them, so recompute it."""
        self._config = new_config
        self.span_floor = self.resolve_span_floor()

    # ---- encoding and engine state --------------------------------------
    def encode(self, texts) -> np.ndarray:
        """L2-normalised sentence vectors."""
        return self.model.encode(texts, convert_to_numpy=True, normalize_embeddings=True,
                                 show_progress_bar=False)

    def refresh(self, model=None) -> None:
        """(Re)encode anchors, noise words and term pools with the given model."""
        if model is not None:
            self.model = model
        d = self.data
        texts, owners = [], []
        for domain, descriptions in d.anchors.items():
            for description in descriptions:
                texts.append(description)
                owners.append(domain)
        vectors = self.encode(texts)
        self.anchor_vectors = {dom: vectors[[i for i, o in enumerate(owners) if o == dom]] for dom in d.domains}
        known = getattr(self.model, "is_known", None)
        self.noise_terms = [w for w in d.noise_vocab if known is None or known(w)]
        self.dropped_terms = {"noise": [w for w in d.noise_vocab if w not in self.noise_terms]}
        self.noise_vectors = self.encode(self.noise_terms)
        self.background_by_k = {
            k: {dom: float(topk_mean(self.noise_vectors @ self.anchor_vectors[dom].T, k).mean())
                for dom in d.domains}
            for k in (1, 2, 3, 4, 5)
        }
        self.pools = {}
        for dom in d.domains:
            pool = domain_term_pool(dom, d.examples, d.pool_vocab, d.generic_terms)
            self.dropped_terms[dom] = [t for t in pool if known is not None and not known(t)]
            pool = [t for t in pool if known is None or known(t)]
            self.pools[dom] = (pool, self.encode(pool))
        self.targets = {}
        pair_sims = self.noise_vectors @ self.noise_vectors.T
        upper = pair_sims[np.triu_indices(len(self.noise_terms), k=1)]
        self.noise_pair_mean = float(upper.mean())     # anisotropy check
        self.noise_pair_p90 = float(np.percentile(upper, 90))
        self.span_floor = self.resolve_span_floor()

    @property
    def background(self) -> dict:
        return self.background_by_k[self.config.anchor_topk]

    # ---- domain gate ------------------------------------------------------
    def domain_scores(self, vectors, k: int = None):
        """Debiased (n, 4) and raw (n, 4) domain scores at top-k."""
        k = self.config.anchor_topk if k is None else int(k)
        vectors = np.atleast_2d(vectors)
        raw = np.column_stack([topk_mean(vectors @ self.anchor_vectors[dom].T, k) for dom in self.data.domains])
        background = np.array([self.background_by_k[k][dom] for dom in self.data.domains])
        return raw - background, raw

    def domain_scores_from_vector(self, vector, k: int = None):
        """(debiased scores, raw scores, display probabilities, nearest anchors) for one vector."""
        k = self.config.anchor_topk if k is None else int(k)
        raw, adjusted, nearest = {}, {}, {}
        for dom in self.data.domains:
            sims = self.anchor_vectors[dom] @ vector
            raw[dom] = float(topk_mean(sims, k))
            adjusted[dom] = raw[dom] - self.background_by_k[k][dom]
            nearest[dom] = self.data.anchors[dom][int(np.argmax(sims))]
        values = np.array([adjusted[dom] for dom in self.data.domains])
        weights = np.exp((values - values.max()) / self.config.display_softmax_temperature)
        probabilities = dict(zip(self.data.domains, (weights / weights.sum()).astype(float)))
        return adjusted, raw, probabilities, nearest

    def gate_multipliers(self) -> dict:
        c = self.config
        return {"primary": c.domain_primary_bonus, "secondary": c.domain_secondary_multiplier,
                "weak_secondary": c.domain_weak_secondary_multiplier, "out_of_domain": 0.0}

    def gate_kinds(self, required, lead) -> np.ndarray:
        """Gate kinds with the current thresholds."""
        c = self.config
        return gate_kind_arrays(required, lead, c.gate_primary_min, c.gate_secondary_min,
                                c.gate_secondary_max_gap, c.gate_weak_max_gap)

    def gate(self, domain_scores: dict, required_domain: str) -> tuple:
        """Soft gate on debiased scores. Returns (allowed, multiplier, kind)."""
        if required_domain not in domain_scores:
            raise ValueError(f"Unknown domain: {required_domain}")
        required = float(domain_scores[required_domain])
        lead = required - max(v for dom, v in domain_scores.items() if dom != required_domain)
        kind = str(self.gate_kinds([required], [lead])[0])
        return kind != "out_of_domain", self.gate_multipliers()[kind], kind

    # ---- target relevance ---------------------------------------------------
    def target_calibration(self, target: str, domain: str) -> dict:
        """Cached target vector plus its cosines to noise words and the domain pool."""
        key = (clean_text(target), domain)
        if key not in self.targets:
            target_clean = clean_text(target)
            vector = self.encode([target_clean])[0]
            pool, pool_vectors = self.pools[domain]
            target_keys = word_keys(target_clean)
            keep = [i for i, term in enumerate(pool) if not (word_keys(term) & target_keys)]
            self.targets[key] = {
                "vector": vector,
                "noise_sims": self.noise_vectors @ vector,
                "pool_sims": pool_vectors[keep] @ vector,
                "pool_terms": [pool[i] for i in keep],
            }
        return self.targets[key]

    def raw_bounds(self, cal: dict, noise_percentile=None, ceiling_topk=None) -> tuple:
        """(noise_floor, ceiling) before the span floor."""
        noise_percentile = self.config.noise_percentile if noise_percentile is None else noise_percentile
        ceiling_topk = self.config.ceiling_topk if ceiling_topk is None else ceiling_topk
        return (float(np.percentile(cal["noise_sims"], noise_percentile)),
                float(topk_mean(cal["pool_sims"], ceiling_topk)))

    def median_calibration_span(self, noise_percentile=None, ceiling_topk=None) -> float:
        """Median raw span (ceiling - noise) over the calibration targets."""
        spans = []
        for target, domain in self.data.calibration_targets.items():
            noise, ceiling = self.raw_bounds(self.target_calibration(target, domain), noise_percentile, ceiling_topk)
            spans.append(ceiling - noise)
        return float(np.median(spans))

    def resolve_span_floor(self) -> float:
        if self.config.min_span_mode == "median":
            return max(self.median_calibration_span(), 0.05)
        return self.config.min_calibration_span

    def bounds(self, cal: dict, noise_percentile=None, ceiling_topk=None, span_floor=None) -> tuple:
        """(noise_floor, ceiling) with ceiling >= noise + span floor."""
        noise, ceiling = self.raw_bounds(cal, noise_percentile, ceiling_topk)
        span_floor = self.span_floor if span_floor is None else span_floor
        return noise, max(ceiling, noise + span_floor)

    relevance = staticmethod(normalized_relevance)

    def graze_floor_mask(self, similarity, cal: dict, kind):
        """True where a primary-gate guess beats the median noise word (gets chip damage)."""
        if not self.config.primary_graze_floor:
            return np.zeros(np.shape(similarity), dtype=bool)
        floor = float(np.percentile(cal["noise_sims"], self.config.primary_graze_noise_pct))
        return (np.asarray(kind) == "primary") & (np.asarray(similarity, dtype=float) >= floor)

    # ---- damage ---------------------------------------------------------------
    def hit_percent(self, relevance, multiplier, allowed, generic_multiplier=1.0, echo_multiplier=1.0,
                    relevance_min=None, gamma=None, floor_mask=None):
        """Damage in percent of the boss's starting HP (single source of truth).

        r < r_min or r <= 0 -> 0; base = GRAZE + (MAX - GRAZE) * ((r - r_min) / (1 - r_min)) ** gamma;
        hit = base * domain * generic * echo; chip floor for allowed hits (and floor_mask rows); cap at MAX.
        """
        c = self.config
        relevance_min = c.relevance_min if relevance_min is None else relevance_min
        gamma = c.damage_gamma if gamma is None else gamma
        r = np.asarray(relevance, dtype=float)
        scaled = np.clip((r - relevance_min) / (1.0 - relevance_min), 0.0, 1.0)
        base = np.where((r < relevance_min) | (r <= 0.0), 0.0,
                        c.graze_min_percent + (c.max_hit_percent - c.graze_min_percent) * scaled ** gamma)
        allowed = np.asarray(allowed, dtype=bool)
        hit = base * np.asarray(multiplier, float) * np.asarray(generic_multiplier, float) \
            * np.asarray(echo_multiplier, float)
        chip_ok = allowed & (base > 0)
        if floor_mask is not None:
            chip_ok = chip_ok | (allowed & np.asarray(floor_mask, dtype=bool))
        hit = np.where(chip_ok, np.maximum(hit, c.chip_percent), 0.0)
        return np.minimum(hit, c.max_hit_percent)

    def base_damage(self, relevance) -> float:
        """Base damage percent for a relevance with domain multiplier 1."""
        return float(self.hit_percent(relevance, 1.0, True))

    def tier(self, damage_percent: float, allowed: bool) -> str:
        if not allowed:
            return "BLOCKED — OUT OF DOMAIN"
        if damage_percent <= 0:
            return "NO EFFECT"
        for cut, name in self.config.tier_cuts:
            if damage_percent >= cut:
                return name
        return "WEAK GRAZE"

    def is_generic(self, cleaned: str) -> bool:
        return is_generic_term(cleaned, self.data.generic_terms)

    def validate(self, guess: str, target: str, used) -> tuple:
        return validate_player_input(guess, target, used, self.config.block_target_word_in_guess)

    def unknown_reason(self, cleaned: str):
        """Message when the encoder has no vector for any word of the guess, else None."""
        known = getattr(self.model, "is_known", None)
        if known is not None and not known(cleaned):
            return (f"Unknown word: the model does not know '{cleaned}' — "
                    "this turn is not counted, try another word.")
        return None

    def echo_threshold(self) -> float:
        """Raw cosine above which a guess counts as a semantic repeat (model-relative)."""
        floor = self.noise_pair_p90
        return floor + self.config.echo_relative * (1.0 - floor)

    def echo_similarity(self, guess_vector, previous_vectors, cal: dict, required_domain: str) -> float:
        """Max cosine to earlier guesses that would have dealt damage."""
        best = 0.0
        noise, ceiling = self.bounds(cal)
        for vector in previous_vectors:
            adjusted, _, _, _ = self.domain_scores_from_vector(vector)
            allowed, _, _ = self.gate(adjusted, required_domain)
            relevance = float(normalized_relevance(vector @ cal["vector"], noise, ceiling))
            if allowed and relevance >= self.config.relevance_min:
                best = max(best, float(vector @ guess_vector))
        return best

    # ---- attacks, battles, hints ----------------------------------------------
    def evaluate_attack(self, guess: str, target: str, required_domain: str, used_guesses, enemy_max_hp: int) -> dict:
        """Validate, score and compute damage for one guess."""
        c = self.config
        is_valid, cleaned, reason = self.validate(guess, target, used_guesses)
        if not is_valid:
            return {"valid": False, "reason": reason}
        unknown = self.unknown_reason(cleaned)
        if unknown:
            return {"valid": False, "reason": unknown, "unknown": True, "word": cleaned}
        cal = self.target_calibration(target, required_domain)
        vectors, latency_ms = timed_call(self.encode, [cleaned] + sorted(used_guesses))
        guess_vector = vectors[0]

        similarity = float(guess_vector @ cal["vector"])
        noise, ceiling = self.bounds(cal)
        relevance = float(normalized_relevance(similarity, noise, ceiling))

        adjusted, raw, probabilities, nearest = self.domain_scores_from_vector(guess_vector)
        allowed, multiplier, match_type = self.gate(adjusted, required_domain)
        lead = adjusted[required_domain] - max(v for dom, v in adjusted.items() if dom != required_domain)

        generic = self.is_generic(cleaned)
        generic_multiplier = c.generic_penalty if generic else 1.0
        echo_similarity = self.echo_similarity(guess_vector, vectors[1:], cal, required_domain)
        echo = echo_similarity >= self.echo_threshold()
        echo_multiplier = c.echo_penalty if echo else 1.0

        base_percent = self.base_damage(relevance) if allowed else 0.0
        floor_mask = bool(self.graze_floor_mask(similarity, cal, match_type))
        hit_percent = float(self.hit_percent(relevance, multiplier, allowed, generic_multiplier,
                                             echo_multiplier, floor_mask=floor_mask))
        damage = int(round(enemy_max_hp * hit_percent / 100))
        if hit_percent > 0:
            damage = max(damage, 1)
        return {
            "valid": True, "word": cleaned, "similarity": similarity,
            "noise_floor": noise, "ceiling": ceiling, "relevance": relevance,
            "domain_scores": adjusted, "domain_raw_scores": raw,
            "domain_background": dict(self.background), "domain_lead": lead,
            "graze_floor": floor_mask,
            "domain_probs": probabilities, "nearest_anchors": nearest,
            "domain_allowed": allowed, "domain_match_type": match_type, "multiplier": multiplier,
            "is_generic": generic, "generic_multiplier": generic_multiplier,
            "echo_similarity": echo_similarity, "is_echo": echo, "echo_multiplier": echo_multiplier,
            "base_percent": base_percent, "damage_percent": round(hit_percent, 1),
            "hit_percent": hit_percent, "damage": damage,
            "tier": self.tier(hit_percent, allowed), "latency_ms": latency_ms,
        }

    @staticmethod
    def resolve_damage(result: dict, enemy_max_hp: int, current_hp: int) -> dict:
        """Add overkill bookkeeping (the per-hit cap is already applied)."""
        if not result["valid"]:
            return result
        calculated = result["damage"] if result["domain_allowed"] else 0
        actual = max(0, min(calculated, current_hp))
        result.update({"calculated_damage": calculated, "actual_damage": actual,
                       "overkill": calculated - actual, "battle_tier": result["tier"]})
        return result

    def simulate_battle(self, target: str, domain: str, initial_hp: int, guesses, on_turn=None) -> pd.DataFrame:
        """Non-interactive battle; one row per guess. on_turn(result, hp, domain) renders a turn."""
        hp, used, rows = initial_hp, set(), []
        for phrase in guesses:
            if hp <= 0:
                break
            result = self.evaluate_attack(phrase, target, domain, used, initial_hp)
            if not result["valid"]:
                rows.append({"Guess": clean_text(phrase), "Tier": "INVALID", "Note": result["reason"]})
                continue
            result = self.resolve_damage(result, initial_hp, hp)
            used.add(result["word"])
            hp_before, hp = hp, hp - result["actual_damage"]
            if on_turn is not None:
                on_turn(result, hp, domain)
            rows.append({
                "Guess": result["word"], "Cosine": round(result["similarity"], 3),
                "Relevance r": round(result["relevance"], 2), "Gate": result["domain_match_type"],
                "Generic": result["is_generic"], "Echo": result["is_echo"],
                "Hit %": round(result["hit_percent"], 1), "Tier": result["tier"],
                "Damage": result["actual_damage"], "HP before": hp_before, "HP left": max(hp, 0),
            })
        return pd.DataFrame(rows)

    @staticmethod
    def summarize_battle(battle_df: pd.DataFrame, initial_hp: int) -> dict:
        """Turns, outcome and tier counts for one battle table."""
        valid = battle_df[battle_df["Tier"] != "INVALID"]
        tiers = valid["Tier"]
        defeated = bool(len(valid) and valid["HP left"].iloc[-1] <= 0)
        return {
            "Valid turns": len(valid), "Defeated": defeated,
            "Turns to defeat": len(valid) if defeated else None,
            "HP left": int(valid["HP left"].iloc[-1]) if len(valid) else initial_hp,
            "Critical": int((tiers == "CRITICAL STRIKE").sum()),
            "Strong": int((tiers == "STRONG HIT").sum()),
            "Solid": int((tiers == "SOLID HIT").sum()),
            "Graze": int((tiers == "WEAK GRAZE").sum()),
            "No effect / blocked": int(tiers.isin(["NO EFFECT", "BLOCKED — OUT OF DOMAIN"]).sum()),
        }

    def hint(self, target: str, domain: str, used_guesses) -> str:
        """A useful but not optimal pool term (relevance 0.35–0.75)."""
        cal = self.target_calibration(target, domain)
        noise, ceiling = self.bounds(cal)
        relevance = normalized_relevance(cal["pool_sims"], noise, ceiling)
        used = {duplicate_key(u) for u in used_guesses}
        options = [(abs(r - 0.55), t) for t, r in zip(cal["pool_terms"], relevance)
                   if duplicate_key(t) not in used and 0.35 <= r <= 0.75]
        if not options:
            options = [(abs(r - 0.55), t) for t, r in zip(cal["pool_terms"], relevance)
                       if duplicate_key(t) not in used]
        return min(options)[1] if options else "No unused hint is available."

ENGINE_DATA = EngineData(
    domains=tuple(DOMAINS), anchors=DOMAIN_ANCHORS, noise_vocab=tuple(NOISE_VOCAB),
    examples=DOMAIN_EXAMPLES, pool_vocab=POOL_VOCAB, generic_terms=frozenset(GENERIC_TERMS_V2),
    calibration_targets=dict(CALIBRATION_TARGETS),
)

def evaluate_test_set(engine: LexicalCombatEngine, test_cases: pd.DataFrame) -> pd.DataFrame:
    """Run labelled (target, guess, domain) rows through evaluate_attack; same columns as before."""
    rows = []
    for row in test_cases.itertuples(index=False):
        result = engine.evaluate_attack(row.guess, row.target, row.required_domain, set(), 200)
        if not result["valid"]:
            rows.append({"Target": row.target, "Guess": row.guess,
                         "Required Domain": row.required_domain, "Predicted Tier": "INVALID",
                         "Human Label": row.human_label, "Predicted Label": "invalid",
                         "Label Aligned": False})
            continue
        predicted = GATE_LABEL_OF_KIND[result["domain_match_type"]]
        rows.append({
            "Target": row.target, "Guess": row.guess, "Required Domain": row.required_domain,
            "Cosine Similarity": round(result["similarity"], 3),
            "Relevance r": round(result["relevance"], 2),
            "Domain Scores": " | ".join(f"{d}: {result['domain_scores'][d]:+.2f}" for d in engine.data.domains),
            "Domain Match": result["domain_match_type"], "Predicted Tier": result["tier"],
            "Hit %": round(result["hit_percent"], 1), "Damage Dealt": result["damage"],
            "Inference Latency (ms)": round(result["latency_ms"], 2),
            "Human Label": row.human_label, "Predicted Label": predicted,
            "Label Aligned": predicted == row.human_label,
        })
    return pd.DataFrame(rows)
log_section("3.1 Engine", _t_s3)

In [ ]:
# 3.2a Gate diagnosis (from the original 5.2; function only, Section 8.1 runs it on the old holdout)
# What this cell does:
#   - diagnose_gate_rows(): explains why an in-domain phrase missed PRIMARY.
#   - Gate grid search with plateau selection and 5-fold grouped CV (calibrate_gate).
#   - Metrics: macro F1, pass / block rates, leak breakdown.
# Cause rules for an in-domain row that did not reach PRIMARY:
#   negative lead      : another domain scores higher (lead < 0)
#   top-k dilution     : passes at top-1 but the top-k mean drags it below the cut
#   threshold too high : required domain is #1 but below the primary cut at any k
# The new holdout (Section 10) is never looked at before Section 10.

def diagnose_gate_rows(engine: LexicalCombatEngine, frame: pd.DataFrame, params: dict) -> pd.DataFrame:
    """One row per (guess, domain): debiased/raw scores at k=1..3, nearest anchors, cause."""
    phrases = frame["guess"].map(clean_text).tolist()
    vectors = engine.encode(phrases)
    rows = []
    for i, row in enumerate(frame.itertuples(index=False)):
        d = row.required_domain
        info = {"target": row.target, "guess": row.guess, "domain": d, "human": row.human_label}
        for k in (1, 2, 3):
            adj, raw = engine.domain_scores(vectors[i], k)
            adj, raw = dict(zip(DOMAINS, adj[0])), dict(zip(DOMAINS, raw[0]))
            others = {x: v for x, v in adj.items() if x != d}
            best_other = max(others, key=others.get)
            info[f"req_k{k}"] = adj[d]
            info[f"lead_k{k}"] = adj[d] - others[best_other]
            if k == params["k"]:
                info["raw"] = raw[d]
                info["background"] = engine.background_by_k[k][d]
                info["top_domain"] = max(adj, key=adj.get)
                info["rival"] = best_other
        req, lead = info["req_k" + str(params["k"])], info["lead_k" + str(params["k"])]
        kind = str(gate_kind_arrays([req], [lead],
                                    params["primary"], params["secondary"], params["gap"],
                                    params["weak_gap"])[0])
        info["gate"] = kind
        sims = engine.anchor_vectors[d] @ vectors[i]
        info["nearest_anchor_cos"] = float(sims.max())
        info["nearest_anchor"] = DOMAIN_ANCHORS[d][int(np.argmax(sims))][:70]
        rival_sims = engine.anchor_vectors[info["rival"]] @ vectors[i]
        info["rival_anchor"] = DOMAIN_ANCHORS[info["rival"]][int(np.argmax(rival_sims))][:50]
        if kind == "primary":
            cause = "ok (primary)"
        elif lead < 0:
            cause = f"negative lead ({info['rival']} wins)"
        elif info["req_k1"] >= params["primary"] and info["lead_k1"] >= 0:
            cause = "top-k dilution"
        else:
            cause = "threshold too high"
        info["cause"] = cause if row.human_label == "in_domain" else ""
        rows.append(info)
    return pd.DataFrame(rows)


# Gate calibration (from the original 5.3; every function now takes the grid as an argument)
# Tunes on gate_set_df (anchor-echo rows excluded), ANCHOR_TOPK in {1,2,3}, plateau selection,
# 5-fold grouped CV. Leaks are split into gate leak (0 damage) and damaging leak.

LABELS = ["in_domain", "ambiguous", "out_of_domain"]
LABEL_CODE = {label: i for i, label in enumerate(LABELS)}
KIND_CODE = {"primary": 0, "secondary": 1, "weak_secondary": 1, "out_of_domain": 2}
MAX_OUT_OF_DOMAIN_LEAK = 0.03     # at most 3% of out-of-domain phrases may pass
MIN_IN_DOMAIN_PRIMARY = 0.80      # at least 80% of in-domain phrases must be PRIMARY
GATE_CV_FOLDS = 5
SELECTION_TOL = 0.005             # plateau scores within this of the best count as ties
GATE_GRID = {
    "k": [1, 2, 3],
    "primary": [float(x) for x in np.round(np.arange(0.02, 0.401, 0.02), 2)],
    "secondary": [float(x) for x in np.round(np.arange(0.00, 0.201, 0.02), 2)],
    "gap": [0.0, 0.02, 0.04, 0.06, 0.08, 0.10, 0.12],
    "weak_extra": [0.02, 0.05, 0.08, 0.12, 0.16, 0.20],
}
GATE_NUMERIC_DIMS = ["primary", "secondary", "gap", "weak_extra"]

def classification_metrics(y_true, y_pred):
    """Accuracy, balanced accuracy and macro F1 over the three gate labels."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    recalls, f1s = [], []
    for label in LABELS:
        tp = np.sum((y_true == label) & (y_pred == label))
        fp = np.sum((y_true != label) & (y_pred == label))
        fn = np.sum((y_true == label) & (y_pred != label))
        recalls.append(tp / (tp + fn) if tp + fn else 0.0)
        f1s.append(2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0)
    return {"accuracy": float(np.mean(y_true == y_pred)),
            "balanced_accuracy": float(np.mean(recalls)), "macro_f1": float(np.mean(f1s))}

def pass_block_metrics(y_true, kinds):
    """Game-feel metrics on gate kinds (primary / secondary / weak_secondary / out_of_domain)."""
    y_true, kinds = np.asarray(y_true), np.asarray(kinds)
    passed = kinds != "out_of_domain"
    def rate(mask, values):
        return float(np.mean(values[mask])) if mask.any() else 1.0
    in_mask, amb_mask, out_mask = (y_true == "in_domain"), (y_true == "ambiguous"), (y_true == "out_of_domain")
    out_block = rate(out_mask, ~passed)
    return {"in_domain_primary": rate(in_mask, kinds == "primary"), "in_domain_pass": rate(in_mask, passed),
            "ambiguous_pass": rate(amb_mask, passed), "out_block": out_block, "out_leak": 1 - out_block}

def gate_inputs(engine: LexicalCombatEngine, frame: pd.DataFrame, ks=(1, 2, 3)) -> dict:
    """Required-domain score and lead for every row at each top-k."""
    unique = sorted(set(frame["phrase"]))
    vectors = engine.encode(unique)
    index = {p: i for i, p in enumerate(unique)}
    rows = np.array([index[p] for p in frame["phrase"]])
    cols = np.array([DOMAINS.index(d) for d in frame["domain"]])
    out = {}
    for k in ks:
        adjusted, _ = engine.domain_scores(vectors, k)
        adj = adjusted[rows]
        required = adj[np.arange(len(rows)), cols]
        masked = adj.copy()
        masked[np.arange(len(rows)), cols] = -np.inf
        out[k] = {"required": required, "lead": required - masked.max(axis=1),
                  "top_domain": np.array(DOMAINS)[adj.argmax(axis=1)]}
    return out

def _grid_axes(grid) -> list:
    """Threshold axes in grid order."""
    return [grid[name] for name in ["primary", "secondary", "gap", "weak_extra"]]

def grid_predictions(required, lead, grid):
    """(combos, n) gate codes for every primary/secondary/gap/weak_extra combo."""
    p, s, g, e = [a.reshape(-1, 1) for a in np.meshgrid(*_grid_axes(grid), indexing="ij")]
    is_primary = (required >= p) & (lead >= 0)
    passes = required >= s
    is_sec = ~is_primary & passes & (lead >= -g)
    is_weak = ~is_primary & ~is_sec & passes & (lead >= -(g + e))
    return np.where(is_primary, 0, np.where(is_sec | is_weak, 1, 2)).astype(np.int8)

def grid_metrics(pred, y, mask):
    """Vectorised metrics over combos for the rows in mask."""
    pred, y = pred[:, mask], y[mask]
    def class_rate(cls_mask, cond):
        return cond[:, cls_mask].mean(axis=1) if cls_mask.any() else np.ones(len(pred))
    in_primary = class_rate(y == 0, pred == 0)
    amb_pass = class_rate(y == 1, pred != 2)
    out_block = class_rate(y == 2, pred == 2)
    f1 = []
    for c in range(3):
        tp = ((pred == c) & (y == c)).sum(axis=1)
        fp = ((pred == c) & (y != c)).sum(axis=1)
        fn = ((pred != c) & (y == c)).sum(axis=1)
        denom = 2 * tp + fp + fn
        f1.append(np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0))
    macro_f1 = np.mean(f1, axis=0)
    leak = 1 - out_block
    score = ((in_primary + amb_pass + out_block) / 3
             - 3 * np.clip(leak - MAX_OUT_OF_DOMAIN_LEAK, 0, None)
             - np.clip(MIN_IN_DOMAIN_PRIMARY - in_primary, 0, None)
             + 0.1 * macro_f1)
    feasible = (leak <= MAX_OUT_OF_DOMAIN_LEAK) & (in_primary >= MIN_IN_DOMAIN_PRIMARY)
    return {"score": score, "feasible": feasible, "in_domain_primary": in_primary,
            "ambiguous_pass": amb_pass, "out_leak": leak, "macro_f1": macro_f1}

def select_plateau(score_nd, numeric_axes, tol=SELECTION_TOL):
    """Index of the best neighbourhood-mean score; ties broken toward the grid centre."""
    size = [3 if axis in numeric_axes else 1 for axis in range(score_nd.ndim)]
    smooth = uniform_filter(score_nd, size=size, mode="nearest")
    candidates = np.argwhere(smooth >= smooth.max() - tol)
    shape = np.array(score_nd.shape)
    def centre_distance(idx):
        return sum(((idx[a] / max(shape[a] - 1, 1)) - 0.5) ** 2 for a in numeric_axes)
    best = min(candidates, key=lambda idx: (centre_distance(idx), -score_nd[tuple(idx)]))
    return tuple(int(i) for i in best), smooth

def edge_report(chosen, grid, numeric_names):
    """Which chosen values sit on the first/last grid value."""
    return {name: (chosen[name] in (grid[name][0], grid[name][-1])) for name in numeric_names}

def _phrase_folds(phrases, n_folds, seed=42) -> np.ndarray:
    """Grouped folds: every row of a phrase lands in the same fold."""
    unique = sorted(set(phrases))
    rng = np.random.default_rng(seed)
    rng.shuffle(unique)
    fold_of = {p: i % n_folds for i, p in enumerate(unique)}
    return np.array([fold_of[p] for p in phrases])

def damage_against_reference(engine: LexicalCombatEngine, phrases, domains) -> np.ndarray:
    """Hit % if each phrase were played against its domain's calibration target."""
    out = []
    for phrase, domain in zip(phrases, domains):
        result = engine.evaluate_attack(phrase, DOMAIN_REFERENCE_TARGET[domain], domain, set(), 200)
        out.append(result["hit_percent"] if result["valid"] else 0.0)
    return np.array(out)

def leak_breakdown(labels, kinds, hit_percents):
    """Gate leak = out-of-domain passes gate with 0 damage; damaging leak = passes AND damages."""
    labels, kinds, hits = np.asarray(labels), np.asarray(kinds), np.asarray(hit_percents, float)
    out = labels == "out_of_domain"
    passed = kinds != "out_of_domain"
    n = max(int(out.sum()), 1)
    gate_leak = int((out & passed & (hits <= 0)).sum())
    damaging = int((out & passed & (hits > 0)).sum())
    return {"out_of_domain rows": int(out.sum()), "gate leak (0 dmg)": gate_leak,
            "damaging leak": damaging, "gate leak %": gate_leak / n, "damaging leak %": damaging / n}

def gate_config_params(grid_params: dict) -> dict:
    """Grid choice -> EngineConfig fields (weak gap = gap + weak_extra, as before)."""
    return {"anchor_topk": int(grid_params["k"]),
            "gate_primary_min": float(grid_params["primary"]),
            "gate_secondary_min": float(grid_params["secondary"]),
            "gate_secondary_max_gap": float(grid_params["gap"]),
            "gate_weak_max_gap": float(grid_params["gap"] + grid_params["weak_extra"])}

def calibrate_gate(engine: LexicalCombatEngine, frame: pd.DataFrame, verbose: bool = True, grid: dict = None) -> dict:
    """Grid + plateau selection + grouped CV. Returns a summary dict; result["params"] uses EngineConfig names."""
    grid = GATE_GRID if grid is None else grid
    y = frame["label"].map(LABEL_CODE).to_numpy()
    inputs = gate_inputs(engine, frame, grid["k"])
    preds = {k: grid_predictions(inputs[k]["required"], inputs[k]["lead"], grid) for k in grid["k"]}
    grid_shape = [len(a) for a in _grid_axes(grid)]
    numeric_axes = [1, 2, 3, 4]                       # axis 0 = k (categorical)

    def choose(mask):
        metrics = {k: grid_metrics(preds[k], y, mask) for k in grid["k"]}
        score_nd = np.stack([metrics[k]["score"].reshape(grid_shape) for k in grid["k"]])
        idx, smooth = select_plateau(score_nd, numeric_axes)
        k = grid["k"][idx[0]]
        flat = int(np.ravel_multi_index(idx[1:], grid_shape))
        params = {"k": k, **{n: grid[n][i] for n, i in zip(["primary", "secondary", "gap", "weak_extra"], idx[1:])}}
        return params, flat, metrics, smooth, score_nd

    full_mask = np.ones(len(frame), dtype=bool)
    params, flat, metrics, smooth, score_nd = choose(full_mask)
    tuned_codes = preds[params["k"]][flat]

    folds = _phrase_folds(frame["phrase"].tolist(), GATE_CV_FOLDS)
    oof_codes, fold_rows = np.zeros(len(frame), dtype=np.int8), []
    for f in range(GATE_CV_FOLDS):
        train, test = folds != f, folds == f
        p_f, flat_f, _, _, _ = choose(train)
        oof_codes[test] = preds[p_f["k"]][flat_f][test]
        fold_rows.append({"fold": f, **p_f, "test rows": int(test.sum())})

    def code_to_kind(codes):
        return np.array(["primary", "secondary", "out_of_domain"])[codes]

    def summarise(codes):
        kinds = code_to_kind(codes)
        return {**classification_metrics(frame["label"], kinds_to_labels(kinds)),
                **pass_block_metrics(frame["label"], kinds)}

    tuned, cv = summarise(tuned_codes), summarise(oof_codes)
    edges = edge_report(params, grid, GATE_NUMERIC_DIMS)
    feasible = bool(metrics[params["k"]]["feasible"][flat])

    scored = frame.copy()
    k = params["k"]
    scored["required"], scored["lead"] = inputs[k]["required"], inputs[k]["lead"]
    scored["top_domain"] = inputs[k]["top_domain"]
    scored["gate"] = code_to_kind(tuned_codes)
    scored["predicted"] = kinds_to_labels(scored["gate"])

    # Top settings table (raw and plateau score) for the chosen k.
    axes = [a.ravel() for a in np.meshgrid(*_grid_axes(grid), indexing="ij")]
    top = pd.DataFrame(dict(zip(["primary", "secondary", "gap", "weak_extra"], axes)))
    for name in ["score", "feasible", "in_domain_primary", "ambiguous_pass", "out_leak", "macro_f1"]:
        top[name] = metrics[k][name]
    top["plateau"] = smooth[grid["k"].index(k)].ravel()
    top = top.sort_values(["plateau", "score"], ascending=False).head(10)

    result = {"params": gate_config_params(params), "grid_params": params, "feasible": feasible, "tuned": tuned, "cv": cv, "edges": edges,
              "folds": pd.DataFrame(fold_rows), "scored": scored, "top": top,
              "grid_size": {n: [min(grid[n]), max(grid[n]), len(grid[n])] for n in GATE_NUMERIC_DIMS}}
    if verbose:
        print_gate_result(result)
    return result

def print_gate_result(result: dict) -> None:
    """Report of one gate calibration (top settings, applied gate, edges, tuned vs CV, folds)."""
    params = result["grid_params"]
    if not result["feasible"]:
        print(f"WARNING: chosen setting misses leak <= {MAX_OUT_OF_DOMAIN_LEAK:.0%} or "
              f"in-domain primary >= {MIN_IN_DOMAIN_PRIMARY:.0%}; best trade-off used.")
    print(f"Top plateau settings at k={params['k']}:")
    display(result["top"].round(3))
    print(f"Applied gate: k={params['k']} | primary>={params['primary']:.2f} (and #1) | secondary>={params['secondary']:.2f} "
          f"gap<={params['gap']:.2f} | weak gap<={params['gap'] + params['weak_extra']:.2f}")
    print("Grid ranges [min, max, n]:", result.get("grid_size"))
    print("Edge check (True = on the grid edge):", result["edges"])
    print("Tuned set vs 5-fold grouped CV:")
    display(pd.DataFrame({"tuned": result["tuned"], "cv": result["cv"]}).round(3))
    print("Per-fold choices (stability):")
    display(result["folds"])

In [ ]:
# 3.2b Damage-curve calibration (from the original 5.4; the grid is an argument)
# What this cell does:
#   - Damage-curve grid search (noise percentile, ceiling top-k, relevance min, gamma).
#   - Loss = per-target band distance + pacing penalty; plateau selection.
#   - Leave-one-target-out CV (calibrate_damage).
# Wide grid, per-target loss (mean + 0.5·std across targets), plateau selection,
# leave-one-target-out CV. Echo flags in battles use the parameters active when
# build_damage_data() runs (pre-calibration defaults), as before.

PACING_WEIGHT = 1.0
STD_WEIGHT = 0.5
DAMAGE_GRID = {
    "noise_percentile": [60, 65, 70, 75, 80, 85, 90, 95],
    "ceiling_topk": [1, 2, 3, 4, 5],
    "relevance_min": [float(x) for x in np.round(np.arange(0.0, 0.2001, 0.025), 3)],
    "gamma": [float(x) for x in np.round(np.arange(0.5, 1.501, 0.1), 2)],
    "span_mode": ["fixed", "median"],
}
DAMAGE_NUMERIC_DIMS = ["noise_percentile", "ceiling_topk", "relevance_min", "gamma"]
DAMAGE_DIM_ORDER = ["noise_percentile", "ceiling_topk", "relevance_min", "gamma", "span_mode"]

def band_distance(percent, band) -> float:
    """Distance from a damage percent to a (low, high) band; 0 inside."""
    low, high = band
    return float(max(low - percent, 0.0, percent - high))

def score_guess(engine: LexicalCombatEngine, target: str, domain: str, guess: str, used=()):
    """Everything the grid needs for one guess; None if the input is invalid."""
    ok, cleaned, _ = engine.validate(guess, target, set(used))
    if not ok or engine.unknown_reason(cleaned):
        return None
    cal = engine.target_calibration(target, domain)
    vectors = engine.encode([cleaned] + list(used))
    adjusted, _, _, _ = engine.domain_scores_from_vector(vectors[0])
    allowed, multiplier, kind = engine.gate(adjusted, domain)
    echo_sim = engine.echo_similarity(vectors[0], vectors[1:], cal, domain)
    similarity = float(vectors[0] @ cal["vector"])
    return {"target": clean_text(target), "domain": domain, "guess": cleaned,
            "similarity": similarity, "allowed": allowed, "multiplier": multiplier, "kind": kind,
            "floor_mask": bool(engine.graze_floor_mask(similarity, cal, kind)),
            "generic_multiplier": engine.config.generic_penalty if engine.is_generic(cleaned) else 1.0,
            "echo_multiplier": engine.config.echo_penalty if echo_sim >= engine.echo_threshold() else 1.0}

def build_damage_data(engine: LexicalCombatEngine) -> tuple:
    """Score banded hits and reference battles with the engine's current gate and parameters."""
    hits = []
    for row in damage_cases_df.itertuples(index=False):
        scored = score_guess(engine, row.target, row.required_domain, row.guess)
        if scored:
            hits.append({**scored, "band": TIER_BANDS_V2[row.human_expected_tier],
                         "expected": row.human_expected_tier})
    stages = {s["target"]: s for s in GAME_STAGES}
    battles = []
    for target, plays in REFERENCE_PLAYS.items():
        for style, (guesses, turn_range) in plays.items():
            used, rows = [], []
            for guess in guesses:
                scored = score_guess(engine, target, stages[target]["domain"], guess, used)
                if scored:
                    rows.append(scored)
                    used.append(scored["guess"])
            battles.append({"target": target, "style": style, "range": turn_range,
                            "hp": stages[target]["hp"], "rows": pd.DataFrame(rows)})
    return pd.DataFrame(hits), battles

def _bounds_table(engine: LexicalCombatEngine, targets_domains, grid) -> tuple:
    """noise[t][pi] and raw ceiling[t][ki] for every grid value."""
    noise, ceil = {}, {}
    for target, domain in targets_domains:
        cal = engine.target_calibration(target, domain)
        noise[target] = np.array([np.percentile(cal["noise_sims"], p) for p in grid["noise_percentile"]])
        ceil[target] = np.array([topk_mean(cal["pool_sims"], k) for k in grid["ceiling_topk"]])
    return noise, ceil

def _percents(engine: LexicalCombatEngine, frame, noise_tab, ceil_tab, pi, ki, span_floor, grid):
    """(R, G, rows) hit % for one noise percentile / top-k / span floor."""
    targets = frame["target"].to_numpy()
    noise = np.array([noise_tab[t][pi] for t in targets])
    ceiling = np.maximum(np.array([ceil_tab[t][ki] for t in targets]), noise + span_floor)
    relevance = normalized_relevance(frame["similarity"].to_numpy(), noise, ceiling)
    r_min = np.array(grid["relevance_min"]).reshape(-1, 1, 1)
    gamma = np.array(grid["gamma"]).reshape(1, -1, 1)
    return engine.hit_percent(relevance, frame["multiplier"].to_numpy(), frame["allowed"].to_numpy(bool),
                             frame["generic_multiplier"].to_numpy(), frame["echo_multiplier"].to_numpy(),
                             r_min, gamma, frame["floor_mask"].to_numpy(bool))

def _turns_vectorised(percents, hp):
    """percents (..., n) -> turns (..., ) as float (nan = never) and remaining HP."""
    damage = np.where(percents > 0, np.maximum(np.round(hp * percents / 100), 1), 0)
    cumulative = np.cumsum(damage, axis=-1)
    won = cumulative >= hp
    turns = np.where(won.any(axis=-1), won.argmax(axis=-1) + 1, np.nan)
    remaining = np.maximum(hp - cumulative[..., -1], 0)
    return turns, remaining

def calibrate_damage(engine: LexicalCombatEngine, verbose: bool = True, grid: dict = None) -> dict:
    """Full grid, plateau selection on all targets, LOTO-CV. result["params"] uses EngineConfig names."""
    grid = DAMAGE_GRID if grid is None else grid
    hits, battles = build_damage_data(engine)
    targets = sorted(hits["target"].unique())
    pairs = set(zip(hits["target"], hits["domain"])) | {(t, d) for t, d in CALIBRATION_TARGETS.items()}
    noise_tab, ceil_tab = _bounds_table(engine, pairs, grid)
    P, K, R, G, S = (len(grid[n]) for n in DAMAGE_DIM_ORDER)
    T = len(targets)
    band_loss = np.zeros((P, K, R, G, S, T))
    pace_loss = np.zeros((P, K, R, G, S, T))
    hit_pct = np.zeros((P, K, R, G, S, len(hits)))
    battle_turns = np.full((P, K, R, G, S, len(battles)), np.nan)
    lows = np.array([b[0] for b in hits["band"]])
    highs = np.array([b[1] for b in hits["band"]])
    target_idx = np.array([targets.index(t) for t in hits["target"]])
    for pi, ki, si in itertools.product(range(P), range(K), range(S)):
        if grid["span_mode"][si] == "median":
            spans = [ceil_tab[t][ki] - noise_tab[t][pi] for t in CALIBRATION_TARGETS]
            span_floor = max(float(np.median(spans)), 0.05)
        else:
            span_floor = engine.config.min_calibration_span
        pct = _percents(engine, hits, noise_tab, ceil_tab, pi, ki, span_floor, grid)      # (R, G, n)
        hit_pct[pi, ki, :, :, si] = pct
        dist = np.maximum(np.maximum(lows - pct, 0), pct - highs)
        for t in range(T):
            band_loss[pi, ki, :, :, si, t] = dist[..., target_idx == t].mean(axis=-1)
        penalties = {t: [] for t in range(T)}
        for bi, battle in enumerate(battles):
            if battle["rows"].empty:
                continue
            bp = _percents(engine, battle["rows"], noise_tab, ceil_tab, pi, ki, span_floor, grid)
            turns, remaining = _turns_vectorised(bp, battle["hp"])
            battle_turns[pi, ki, :, :, si, bi] = turns
            low, high = battle["range"]
            pen = np.where(np.isnan(turns), 6.0 + 10.0 * remaining / battle["hp"],
                           3.0 * np.maximum(np.maximum(low - np.nan_to_num(turns), 0), np.nan_to_num(turns) - high))
            penalties[targets.index(battle["target"])].append(pen)
        for t in range(T):
            if penalties[t]:
                pace_loss[pi, ki, :, :, si, t] = np.mean(penalties[t], axis=0)
    per_target = band_loss + PACING_WEIGHT * pace_loss
    numeric_axes = [0, 1, 2, 3]

    def choose_index(target_mask):
        sub = per_target[..., target_mask]
        score = -(sub.mean(axis=-1) + STD_WEIGHT * sub.std(axis=-1))   # higher = better
        idx, _ = select_plateau(score, numeric_axes, tol=SELECTION_TOL)
        return idx

    def params_of(idx):
        return {n: grid[n][i] for n, i in zip(DAMAGE_DIM_ORDER, idx)}

    def eval_rows(idx, row_mask, battle_ids):
        pct = hit_pct[idx][row_mask]
        in_band = ((pct >= lows[row_mask]) & (pct <= highs[row_mask]))
        pace = []
        for bi in battle_ids:
            turns = battle_turns[idx][bi]
            low, high = battles[bi]["range"]
            pace.append({"battle": f"{battles[bi]['target']}/{battles[bi]['style']}",
                         "turns": None if np.isnan(turns) else int(turns), "goal": f"{low}–{high}",
                         "met": bool(not np.isnan(turns) and low <= turns <= high)})
        return in_band, pace

    all_mask = np.ones(T, dtype=bool)
    final_idx = choose_index(all_mask)
    final_params = params_of(final_idx)
    tuned_in_band, tuned_pace = eval_rows(final_idx, np.ones(len(hits), bool), range(len(battles)))

    fold_rows, cv_in_band, cv_pace = [], [], []
    for t, name in enumerate(targets):
        mask = all_mask.copy()
        mask[t] = False
        idx = choose_index(mask)
        in_band, pace = eval_rows(idx, target_idx == t,
                                  [bi for bi, b in enumerate(battles) if b["target"] == name])
        cv_in_band.extend(in_band.tolist())
        cv_pace.extend(pace)
        fold_rows.append({"held-out target": name, **params_of(idx), "cases": int(in_band.size),
                          "held-out in-band": float(in_band.mean()) if in_band.size else np.nan,
                          "held-out pacing met": f"{sum(p['met'] for p in pace)}/{len(pace)}" if pace else "—"})

    edges = edge_report(final_params, grid, DAMAGE_NUMERIC_DIMS)
    summary = pd.DataFrame({
        "tuned (all targets)": {"in-band": float(np.mean(tuned_in_band)),
                                "pacing met": f"{sum(p['met'] for p in tuned_pace)}/{len(tuned_pace)}"},
        "LOTO-CV (held-out)": {"in-band": float(np.mean(cv_in_band)),
                               "pacing met": f"{sum(p['met'] for p in cv_pace)}/{len(cv_pace)}"},
    })
    result = {"params": damage_config_params(final_params), "grid_params": final_params, "edges": edges, "summary": summary,
              "folds": pd.DataFrame(fold_rows), "tuned_pace": pd.DataFrame(tuned_pace),
              "cv_pace": pd.DataFrame(cv_pace), "tuned_in_band": float(np.mean(tuned_in_band)),
              "cv_in_band": float(np.mean(cv_in_band)),
              "all_pacing_met": all(p["met"] for p in tuned_pace),
              "cv_all_pacing_met": all(p["met"] for p in cv_pace),
              "grid_size": {n: [min(grid[n]), max(grid[n]), len(grid[n])] for n in DAMAGE_NUMERIC_DIMS}}
    if verbose:
        print_damage_result(result)
    return result

def print_damage_result(result: dict) -> None:
    """Report of one damage calibration."""
    print("Applied:", result["grid_params"])
    print("Grid ranges [min, max, n]:", result.get("grid_size"))
    print("Edge check (True = on the grid edge):", result["edges"])
    print("Tuned vs leave-one-target-out CV:")
    display(result["summary"])
    print("Per-fold choices:")
    display(result["folds"].round(3))
    print("Pacing with the final parameters:")
    display(result["tuned_pace"])

def damage_config_params(grid_params: dict) -> dict:
    """Grid choice -> EngineConfig fields."""
    return {"noise_percentile": int(grid_params["noise_percentile"]),
            "ceiling_topk": int(grid_params["ceiling_topk"]),
            "relevance_min": float(grid_params["relevance_min"]),
            "damage_gamma": float(grid_params["gamma"]),
            "min_span_mode": grid_params["span_mode"]}

In [ ]:
# 3.2c Grid expansion, calibration files per model, calibrate_engine() (used by Sections 6, 8 and 9)
# What this cell does:
#   - Expands a grid automatically when the best value sits on its edge.
#   - Saves / loads tuned_params_<model>.json, keyed by a fingerprint.
#   - calibrate_engine(): runs gate + damage calibration and applies it to the engine.
_t_s32 = time.perf_counter()

# (lowest, highest, step) a grid axis may grow to. A choice on an edge that is also a limit is a
# "hard edge" (e.g. gap = 0, relevance_min = 0, ceiling_topk = 1): the parameter cannot go further.
GATE_LIMITS = {"primary": (-0.30, 1.00, 0.02), "secondary": (-0.30, 0.80, 0.02),
               "gap": (0.0, 0.50, 0.02), "weak_extra": (0.0, 0.60, 0.02)}
DAMAGE_LIMITS = {"noise_percentile": (10, 99, 5), "ceiling_topk": (1, 10, 1),
                 "relevance_min": (0.0, 0.40, 0.025), "gamma": (0.2, 3.0, 0.1)}
MAX_AXIS_LEN = {"primary": 40, "secondary": 25, "gap": 16, "weak_extra": 14,
                "noise_percentile": 18, "ceiling_topk": 10, "relevance_min": 17, "gamma": 22}
INTEGER_DIMS = {"noise_percentile", "ceiling_topk"}
MAX_GRID_ROUNDS = 6        # expansion rounds per calibration
CALIBRATION_CACHE = {}     # (model choice, fingerprint) -> (gate, damage); avoids repeats in one session

def expand_axis(values, side: str, step: float, lo: float, hi: float, name: str, n_new: int = 3) -> list:
    """Add n_new values beyond one edge (clamped to the limit); trim the far side if the axis gets too long."""
    vals = sorted(values)
    if side == "high":
        new = [min(hi, vals[-1] + step * i) for i in range(1, n_new + 1)]
    else:
        new = [max(lo, vals[0] - step * i) for i in range(1, n_new + 1)]
    cast = (lambda v: int(round(v))) if name in INTEGER_DIMS else (lambda v: float(round(v, 4)))
    merged = sorted({cast(v) for v in vals + new})
    limit = MAX_AXIS_LEN.get(name)
    if limit and len(merged) > limit:
        merged = merged[-limit:] if side == "high" else merged[:limit]
    return merged

def edge_moves(chosen: dict, grid: dict, limits: dict) -> tuple:
    """(expandable edges, hard edges) as lists of (name, side)."""
    moves, hard = [], []
    for name, (lo, hi, _) in limits.items():
        values = sorted(grid[name])
        if len(values) < 2:
            continue
        if chosen[name] == values[0]:
            (moves if values[0] > lo + 1e-9 else hard).append((name, "low"))
        elif chosen[name] == values[-1]:
            (moves if values[-1] < hi - 1e-9 else hard).append((name, "high"))
    return moves, hard

def calibrate_with_expansion(kind: str, engine: LexicalCombatEngine, frame: pd.DataFrame = None,
                             max_rounds: int = MAX_GRID_ROUNDS, verbose: bool = True) -> dict:
    """Run the gate or damage grid; while the choice sits on an expandable edge, widen that axis and re-run."""
    limits = GATE_LIMITS if kind == "gate" else DAMAGE_LIMITS
    grid = {k: list(v) for k, v in (GATE_GRID if kind == "gate" else DAMAGE_GRID).items()}
    log = []
    for round_no in range(max_rounds + 1):
        if kind == "gate":
            result = calibrate_gate(engine, frame, verbose=False, grid=grid)
        else:
            result = calibrate_damage(engine, verbose=False, grid=grid)
        moves, hard = edge_moves(result["grid_params"], grid, limits)
        last = not moves or round_no == max_rounds
        log.append({"round": round_no, "chosen": dict(result["grid_params"]),
                    "expanded": "" if last else ", ".join(f"{n} {s}" for n, s in moves),
                    "hard edges": ", ".join(f"{n} {s}" for n, s in hard)})
        if last:
            break
        for name, side in moves:
            lo, hi, step = limits[name]
            grid[name] = expand_axis(grid[name], side, step, lo, hi, name)
    result["grid"] = grid
    result["expansion_log"] = pd.DataFrame(log)
    result["soft_edges"] = {name: any(n == name for n, _ in moves) for name in limits}
    result["hard_edges"] = {name: any(n == name for n, _ in hard) for name in limits}
    if verbose:
        print(f"{kind} grid expansion rounds:")
        display(result["expansion_log"])
        (print_gate_result if kind == "gate" else print_damage_result)(result)
        print("Edges after expansion — soft (still expandable, a real problem):",
              [n for n, v in result["soft_edges"].items() if v] or "none",
              "| hard (natural limit):", [n for n, v in result["hard_edges"].items() if v] or "none")
    return result

def to_jsonable(value):
    """Convert numpy / pandas values for json.dump."""
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]
    if isinstance(value, pd.DataFrame):
        return {"__frame__": to_jsonable(value.to_dict(orient="split"))}
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.ndarray):
        return to_jsonable(value.tolist())
    if isinstance(value, float) and value != value:
        return None
    return value

def from_jsonable(value):
    """Inverse of to_jsonable for the frames we store."""
    if isinstance(value, dict):
        if "__frame__" in value:
            f = value["__frame__"]
            return pd.DataFrame(f["data"], index=f["index"], columns=f["columns"])
        return {k: from_jsonable(v) for k, v in value.items()}
    if isinstance(value, list):
        return [from_jsonable(v) for v in value]
    return value

def calibration_fingerprint(model_identity: str) -> str:
    """Hash of everything the grid searches depend on; a stale file is never reused."""
    payload = {
        "model": model_identity, "defaults": asdict(EngineConfig()),
        "gate_set": gate_set_df.to_dict(orient="list"), "cases": damage_cases_df.to_dict(orient="list"),
        "plays": REFERENCE_PLAYS, "gate_grid": GATE_GRID, "damage_grid": DAMAGE_GRID,
        "expansion": [GATE_LIMITS, DAMAGE_LIMITS, MAX_AXIS_LEN, MAX_GRID_ROUNDS],
        "limits": [MAX_OUT_OF_DOMAIN_LEAK, MIN_IN_DOMAIN_PRIMARY, GATE_CV_FOLDS, SELECTION_TOL,
                   PACING_WEIGHT, STD_WEIGHT],
    }
    return hashlib.sha1(json.dumps(to_jsonable(payload), sort_keys=True).encode()).hexdigest()[:16]

def load_calibration(path: Path, fingerprint: str, verbose: bool = True):
    """(gate_cal, damage_cal) from a saved file, or None."""
    if not path.is_file():
        if verbose:
            print(f"{path.name} does not exist yet — running the grid searches.")
        return None
    stored = json.loads(path.read_text())
    if stored.get("fingerprint") != fingerprint:
        if verbose:
            print(f"WARNING: {path.name} was made with another model/data/settings — running the grid searches.")
        return None
    return from_jsonable(stored["gate"]), from_jsonable(stored["damage"])

def save_calibration(path: Path, fingerprint: str, gate_cal: dict, damage_cal: dict) -> None:
    """Write the calibration file (only when PERSISTENT)."""
    if not PERSISTENT:
        print(f"Calibration not saved ({path.name}): PERSISTENT = False (Drive not mounted).")
        return
    gate = {k: v for k, v in gate_cal.items() if k != "scored"}
    path.write_text(json.dumps(to_jsonable({"fingerprint": fingerprint, "gate": gate, "damage": damage_cal}), indent=1))
    print(f"Calibration saved: {path}")

def score_gate_set(engine: LexicalCombatEngine, frame: pd.DataFrame) -> pd.DataFrame:
    """Gate result per row with the engine's thresholds (weak secondary shown as secondary, as in tuning)."""
    k = engine.config.anchor_topk
    inputs = gate_inputs(engine, frame, [k])[k]
    kinds = engine.gate_kinds(inputs["required"], inputs["lead"])
    scored = frame.copy()
    scored["required"], scored["lead"], scored["top_domain"] = inputs["required"], inputs["lead"], inputs["top_domain"]
    scored["gate"] = np.where(kinds == "weak_secondary", "secondary", kinds)
    scored["predicted"] = kinds_to_labels(scored["gate"])
    return scored

def calibrate_engine(engine: LexicalCombatEngine, choice: str, identity: str, verbose: bool = True,
                     save: bool = True) -> tuple:
    """Gate grid then damage grid (both with edge expansion), or load a matching file. Returns (gate, damage, source)."""
    path = PATHS.artifacts / f"tuned_params_{choice}.json"
    fingerprint = calibration_fingerprint(identity)
    engine.config = EngineConfig()          # every model starts from the same pre-calibration defaults
    cached = CALIBRATION_CACHE.get((choice, fingerprint))
    loaded = cached or (load_calibration(path, fingerprint, verbose)
                        if LOAD_TUNED_PARAMS and not FORCE_REBUILD else None)
    if loaded is None:
        source = "grid search"
        gate = calibrate_with_expansion("gate", engine, gate_set_df, verbose=verbose)
        engine.config = replace(engine.config, **gate["params"])
        if verbose:   # preliminary leak split: damage still uses the pre-calibration curve (Section 9 repeats it)
            gs = gate["scored"]
            need = ((gs["label"] == "out_of_domain") & (gs["gate"] != "out_of_domain")).to_numpy()
            hits = np.zeros(len(gs))
            if need.any():
                hits[need] = damage_against_reference(engine, gs.loc[need, "phrase"], gs.loc[need, "domain"])
            print("Leak split on the tuning set (vs each domain's calibration target):",
                  leak_breakdown(gs["label"], gs["gate"], hits))
        damage = calibrate_with_expansion("damage", engine, verbose=verbose)
        engine.config = replace(engine.config, **damage["params"])
        if save:
            save_calibration(path, fingerprint, gate, damage)
    else:
        source = "session cache" if cached else f"loaded from {path.name}"
        gate, damage = loaded
        engine.config = replace(engine.config, **gate["params"], **damage["params"])
        gate["scored"] = score_gate_set(engine, gate_set_df)
        if verbose:
            print(f"Calibration {source} (set LOAD_TUNED_PARAMS = False or FORCE_REBUILD = True to re-run).")
            print("Gate:", gate["grid_params"], "| soft edges:", [n for n, v in gate.get("soft_edges", {}).items() if v] or "none")
            display(pd.DataFrame({"tuned": gate["tuned"], "cv": gate["cv"]}).round(3))
            print("Damage:", damage["grid_params"], "| soft edges:", [n for n, v in damage.get("soft_edges", {}).items() if v] or "none")
            display(damage["summary"])
    CALIBRATION_CACHE[(choice, fingerprint)] = (gate, damage)
    return gate, damage, source

def damage_curve_table(engine: LexicalCombatEngine) -> pd.DataFrame:
    """Base damage per relevance with the active parameters."""
    curve = pd.DataFrame({"relevance r": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]})
    curve["base hit %"] = [round(engine.base_damage(r), 1) for r in curve["relevance r"]]
    curve["tier (primary)"] = [engine.tier(p, True) for p in curve["base hit %"]]
    curve["secondary ×0.75 %"] = [round(float(engine.hit_percent(r, engine.config.domain_secondary_multiplier, True)), 1)
                                  for r in curve["relevance r"]]
    return curve

log_section("3. Engine + calibration functions", _t_s32)

## 4. Corpus: collection and preparation (Phase 2)

**Main source:** Simple English Wikipedia through Hugging Face `datasets` (`wikimedia/wikipedia`, config `20231101.simple`) — plain language, covers every topic, and fits free Colab.

**Supplement (used only when coverage is below target):** English Wikipedia (`20231101.en`) in streaming mode, **keeping only articles that contain game words that are still missing or rare** (at most 40 articles per word, with limits on articles scanned and time). The word list comes from anchors, term pools, noise vocab and the calibration set only — **never from the holdout** (selecting training data with holdout words would be leakage). Only natural Wikipedia text is stored; no game text is written to the corpus.

**License:** Wikipedia (Simple and English) is **CC BY-SA 3.0 and GFDL** (per the Hugging Face dataset card; current Wikipedia text is published under CC BY-SA 4.0) — cite the source in the report. Last-resort fallback when Hugging Face is unreachable: NLTK corpora (Brown, Reuters-21578, Gutenberg, Webtext, ABC), which are for research/educational use and much smaller (lower quality).

**Preprocessing (matches the game):** sentence split (regex on `. ! ?` and newlines) → the game's `clean_text()` (with a regex fast path for ASCII sentences that gives identical output, checked by an assert) → whitespace tokenization → drop sentences shorter than 3 tokens → two `Phrases` passes (bigram, then trigram, with connector words, e.g. `board_of_directors`) → write one sentence per line so gensim can stream it without loading everything into RAM.

**Pass criterion:** coverage of game words ≥ 95% (a word counts as covered if it appears ≥ `min_count` = 5 times in the corpus).

- **4.1** Corpus configuration, sentence splitting / cleaning, data sources with fallbacks, coverage tools.
- **4.2** Build (or load) the processed corpus.
- **4.3** Corpus statistics, coverage report and leakage checks.

In [ ]:
# 4.1 Corpus configuration, sentence splitting / cleaning, sources (with checks and fallbacks), coverage tools
# What this cell does:
#   - Corpus settings (sources, limits, min_count, coverage target).
#   - Sentence splitting and a fast, equivalent version of clean_text().
#   - Data sources: Simple Wikipedia, English Wikipedia (streaming), NLTK fallback.
#   - Coverage tools: how many game words the corpus knows (never written to it).
_t_s4 = time.perf_counter()

WIKI_DATASET = "wikimedia/wikipedia"
CORPUS_CONFIG = {
    "primary_config": "20231101.simple",       # Simple English Wikipedia (CC BY-SA 3.0 / GFDL)
    "supplement_config": "20231101.en",        # English Wikipedia, streamed and filtered (CC BY-SA 3.0 / GFDL)
    "max_primary_articles": 15_000 if QUICK_RUN else None,     # None = every article
    "min_sentence_tokens": 3,
    "min_count": 5,                            # vocabulary threshold (also used by Word2Vec / FastText)
    "coverage_target": 0.95,
    "supplement_min_freq": 20,                 # game words rarer than this get targeted articles
    "supplement_per_word": 40,                 # at most this many articles per targeted word
    "supplement_max_scan": 5_000 if QUICK_RUN else 300_000,    # articles scanned at most
    "supplement_time_limit_s": 120 if QUICK_RUN else 900,
    "phrases": {"min_count": 5, "threshold": 10.0, "max_vocab_size": 20_000_000},
}
CORPUS_FILES = ["corpus.txt", "phrases_bigram.pkl", "phrases_trigram.pkl", "counts.json", "corpus_stats.json"]

# Function words only. sklearn's ENGLISH_STOP_WORDS is not used: it also lists content words
# that matter in this game (computer, system, interest, fire, bill, ...).
STOPWORDS = frozenset("""
a an the and or but nor so yet if then than that this these those there here of to in on at by for with
from into onto upon about above below over under between among through during before after as is are was
were be been being am do does did done have has had having it its it's itself he him his she her hers they
them their theirs we us our ours you your yours i me my mine who whom whose which what when where why how
not no can could will would shall should may might must also such each other some any all both either neither
very more most much many few less least own same too just only even still while whereas although though
because since until unless via per etc
""".split())

_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+|\n+")
_NON_ALPHA_ASCII = re.compile(r"[^a-z]+")

def fast_clean_text(text: str) -> str:
    """clean_text() with a regex fast path for ASCII text (same output, checked below)."""
    if text.isascii():
        return " ".join(_NON_ALPHA_ASCII.sub(" ", text.lower()).split())
    return clean_text(text)

_rng_check = random.Random(SEED)
_samples = ["Data-base & AI: 3 CEOs' (2024) plans!", "Café — naïve façade", "x\ty\nz", "", "  CO2 and H2O  ",
            "".join(_rng_check.choice("abcXYZ019 -_.,'\t\né") for _ in range(400))]
assert all(fast_clean_text(s) == clean_text(s) for s in _samples), "fast_clean_text must equal clean_text"

def article_sentences(text: str, min_tokens: int = CORPUS_CONFIG["min_sentence_tokens"]) -> list:
    """Split an article into sentences, clean them like the game, keep sentences with >= min_tokens."""
    out = []
    for piece in _SENT_SPLIT.split(text):
        tokens = fast_clean_text(piece).split()
        if len(tokens) >= min_tokens:
            out.append(tokens)
    return out

def resolve_wiki_config(wanted: str):
    """Return a config name that exists (wanted, else the newest with the same language suffix), or None."""
    try:
        from datasets import get_dataset_config_names
        names = get_dataset_config_names(WIKI_DATASET)
    except Exception as error:   # offline / hub error: try the wanted name directly
        print(f"WARNING: could not list {WIKI_DATASET} configs ({type(error).__name__}); trying '{wanted}' directly.")
        return wanted
    if wanted in names:
        print(f"Config check: {WIKI_DATASET} / {wanted} exists.")
        return wanted
    suffix = wanted.split(".", 1)[1]
    same = sorted(n for n in names if n.endswith("." + suffix))
    if same:
        print(f"WARNING: {wanted} not found; using the newest '{suffix}' config: {same[-1]}")
        return same[-1]
    print(f"WARNING: no '{suffix}' config in {WIKI_DATASET}.")
    return None

def wiki_articles(config: str, limit: int = None):
    """Stream (title, text) from Hugging Face Wikipedia; raises if the config cannot be opened."""
    from datasets import load_dataset
    stream = load_dataset(WIKI_DATASET, config, split="train", streaming=True)
    for i, row in enumerate(stream):
        if limit is not None and i >= limit:
            break
        yield {"title": row.get("title", ""), "text": row["text"], "source": f"{WIKI_DATASET}/{config}"}

def nltk_articles(limit: int = None):
    """Last-resort fallback: NLTK corpora (small; research / educational use)."""
    import nltk
    names = ["brown", "reuters", "gutenberg", "webtext", "abc"]
    for name in names:
        try:
            nltk.download(name, quiet=True)
        except Exception:
            pass
    from nltk import corpus as nltk_corpus
    count = 0
    for name in names:
        try:
            reader = getattr(nltk_corpus, name)
            fileids = reader.fileids()
        except Exception as error:
            print(f"WARNING: NLTK corpus {name} unavailable ({type(error).__name__}).")
            continue
        for fid in fileids:
            if limit is not None and count >= limit:
                return
            count += 1
            yield {"title": f"{name}/{fid}", "text": " ".join(reader.words(fid)), "source": f"nltk/{name}"}

SOURCE_LICENSES = {
    WIKI_DATASET: "Wikipedia text, CC BY-SA 3.0 and GFDL (per the dataset card); https://huggingface.co/datasets/wikimedia/wikipedia",
    "nltk": "NLTK corpora (Brown, Reuters-21578, Gutenberg, Webtext, ABC): research / educational use; see nltk.org/nltk_data",
}

# ---- game vocabulary: used ONLY to measure coverage (never written to the corpus) ----
def game_phrase_sets() -> dict:
    """Set name -> list of (phrase, domain). Anchors contribute their content words."""
    sets = {"anchors": [], "term pools": [], "noise vocab": [], "calibration": [], "old diagnostic": [], "holdout": []}
    for d in DOMAINS:
        for text in DOMAIN_ANCHORS[d]:
            sets["anchors"] += [(w, d) for w in clean_text(text).split() if w not in STOPWORDS]
        sets["term pools"] += [(t, d) for t in domain_term_pool(d, DOMAIN_EXAMPLES, POOL_VOCAB, GENERIC_TERMS_V2)]
    sets["noise vocab"] = [(w, "noise") for w in NOISE_VOCAB]
    for t, g, d, _, _ in TEST_CASES_V2 + DAMAGE_CASES_EXTRA:
        sets["calibration"] += [(t, d), (g, d)]
    sets["calibration"] += [(p, d) for d, ps in FRESH_GATE_IN_DOMAIN.items() for p in ps]
    sets["calibration"] += list(FRESH_AMBIGUOUS_PAIRS) + [(w, "noise") for w in FRESH_NOISE_OUT]
    sets["calibration"] += [(p, d) for d, ps in DOMAIN_EXAMPLES.items() for p in ps] + list(OUT_OF_DOMAIN_CASES)
    for target, plays in REFERENCE_PLAYS.items():
        d = CALIBRATION_TARGETS[target]
        sets["calibration"] += [(g, d) for guesses, _ in plays.values() for g in guesses]
        sets["calibration"] += [(g, d) for g in PLAYER_LOG_GUESSES_V2.get(target, [])]
    for t, g, d, _, _ in OLD_HOLDOUT_DIAG_CASES:
        sets["old diagnostic"] += [(t, d), (g, d)]
    for t, g, d, _, _ in HOLDOUT_CASES_DRAFT:
        sets["holdout"] += [(t, d), (g, d)]
    return {name: sorted({(clean_text(p), d) for p, d in pairs if clean_text(p)}) for name, pairs in sets.items()}

GAME_PHRASE_SETS = game_phrase_sets()
NON_HOLDOUT_SETS = ["anchors", "term pools", "noise vocab", "calibration", "old diagnostic"]

def coverage_report(word_known, phrase_unit=None, label: str = "") -> tuple:
    """(summary per set/domain, missing words per set). word_known(w) -> bool; phrase_unit(p) -> bool."""
    rows, missing = [], {}
    for name, pairs in GAME_PHRASE_SETS.items():
        for domain in sorted({d for _, d in pairs}) + ["all"]:
            sel = [p for p, d in pairs if domain == "all" or d == domain]
            words = sorted({w for p in sel for w in p.split()})
            known = [w for w in words if word_known(w)]
            full = [p for p in sel if all(word_known(w) for w in p.split())]
            units = [p for p in sel if len(p.split()) > 1 and phrase_unit is not None and phrase_unit(p)]
            rows.append({"set": name, "domain": domain, "words": len(words), "word coverage": len(known) / max(len(words), 1),
                         "phrases": len(sel), "phrases fully covered": len(full) / max(len(sel), 1),
                         "multi-word as one unit": len(units)})
            if domain == "all":
                missing[name] = sorted(set(words) - set(known))
    all_words = sorted({w for pairs in GAME_PHRASE_SETS.values() for p, _ in pairs for w in p.split()})
    nonhold = sorted({w for n in NON_HOLDOUT_SETS for p, _ in GAME_PHRASE_SETS[n] for w in p.split()})
    totals = {"all game words": sum(map(word_known, all_words)) / len(all_words),
              "non-holdout game words": sum(map(word_known, nonhold)) / len(nonhold),
              "unique game words": len(all_words)}
    if label:
        print(f"Coverage ({label}): " + " | ".join(f"{k}: {v:.1%}" if isinstance(v, float) else f"{k}: {v}"
                                                    for k, v in totals.items()))
    return pd.DataFrame(rows), missing, totals

print("Game phrase sets (coverage only):", {k: len(v) for k, v in GAME_PHRASE_SETS.items()})

In [ ]:
# 4.2 Build (or load) the processed corpus: articles -> sentences -> clean_text -> Phrases -> corpus.txt
# What this cell does:
#   - Builds the corpus once: articles -> sentences -> clean_text -> Phrases -> corpus.txt.
#   - Adds a targeted supplement from English Wikipedia if coverage is too low.
#   - Loads the cached corpus from Drive on later runs.
def corpus_cached() -> bool:
    return not FORCE_REBUILD and all((PATHS.corpus / f).is_file() for f in CORPUS_FILES)

def domain_keyword_sets() -> dict:
    """Single content words of each domain's term pool (only for the rough topic estimate)."""
    return {d: {w for t in domain_term_pool(d, DOMAIN_EXAMPLES, POOL_VOCAB, GENERIC_TERMS_V2)
                for w in t.split() if w not in STOPWORDS and len(w) > 3} for d in DOMAINS}

def write_articles(articles, out, counts: Counter, stats: dict, keywords: dict, label: str, keep=None) -> int:
    """Append cleaned sentences of each article to `out`; returns the number of articles written."""
    written, start = 0, time.perf_counter()
    for i, art in enumerate(articles):
        sents = article_sentences(art["text"])
        if not sents or (keep is not None and not keep(sents)):
            continue
        for tokens in sents:
            out.write(" ".join(tokens) + "\n")
            counts.update(tokens)
        written += 1
        src = stats["sources"].setdefault(art["source"], {"articles": 0, "sentences": 0, "tokens": 0})
        src["articles"] += 1
        src["sentences"] += len(sents)
        src["tokens"] += sum(map(len, sents))
        vocab = {t for s in sents for t in s}
        hits = {d: len(vocab & kw) for d, kw in keywords.items()}
        best = max(hits, key=hits.get)
        top_two = sorted(hits.values())[-2:]
        stats["domain_estimate"][best if hits[best] >= 3 and top_two[0] < top_two[1] else "general / mixed"] += 1
        if written % 20_000 == 0:
            print(f"  {label}: {written:,} articles | {sum(counts.values()):,} tokens | "
                  f"{time.perf_counter() - start:.0f}s")
    return written

def targeted_supplement(needed: set, out, counts: Counter, stats: dict, keywords: dict) -> dict:
    """Stream English Wikipedia; keep articles that contain still-needed game words (holdout words excluded)."""
    cfg = CORPUS_CONFIG
    config = resolve_wiki_config(cfg["supplement_config"])
    if config is None or not needed:
        return {"scanned": 0, "kept": 0, "still_needed": sorted(needed)}
    quota = {w: cfg["supplement_per_word"] for w in needed}
    def pattern_for(words):
        return re.compile(r"\b(?:" + "|".join(sorted(map(re.escape, words), key=len, reverse=True)) + r")\b")
    pattern, scanned, kept, start = pattern_for(quota), 0, 0, time.perf_counter()
    state = {"matched": []}
    def keep(sents):
        tokens = {t for s in sents for t in s}
        state["matched"] = [w for w in quota if w in tokens]
        return bool(state["matched"])
    try:
        from datasets import load_dataset
        stream = load_dataset(WIKI_DATASET, config, split="train", streaming=True)
        for row in stream:
            if scanned >= cfg["supplement_max_scan"] or not quota \
                    or time.perf_counter() - start > cfg["supplement_time_limit_s"]:
                break
            scanned += 1
            if not pattern.search(row["text"].lower()):
                continue
            art = {"title": row.get("title", ""), "text": row["text"], "source": f"{WIKI_DATASET}/{config} (targeted)"}
            if write_articles([art], out, counts, stats, keywords, "supplement", keep=keep):
                kept += 1
                done = False
                for w in state["matched"]:
                    quota[w] -= 1
                    if quota[w] <= 0:
                        del quota[w]
                        done = True
                if done and quota:
                    pattern = pattern_for(quota)
            if scanned % 50_000 == 0:
                print(f"  supplement: scanned {scanned:,} | kept {kept:,} | words still short {len(quota)}")
    except Exception as error:
        print(f"WARNING: supplement stopped ({type(error).__name__}: {str(error)[:120]})")
    return {"config": config, "scanned": scanned, "kept": kept, "seconds": round(time.perf_counter() - start, 1),
            "still_needed": sorted(w for w in quota if counts[w] < cfg["supplement_min_freq"])}

def build_corpus() -> dict:
    """Download, clean, (supplement), learn phrases, write corpus.txt + counts + stats; persist them."""
    cfg, start = CORPUS_CONFIG, time.perf_counter()
    raw_path, corpus_path = PATHS.work / "raw_corpus.txt", PATHS.work / "corpus.txt"
    stats = {"sources": {}, "domain_estimate": Counter(), "config": cfg, "quick_run": QUICK_RUN,
             "licenses": {}, "seed": SEED}
    raw_counts, keywords = Counter(), domain_keyword_sets()
    with open(raw_path, "w") as out:
        config = resolve_wiki_config(cfg["primary_config"])
        if config:
            try:
                write_articles(wiki_articles(config, cfg["max_primary_articles"]), out, raw_counts, stats,
                               keywords, config)
            except Exception as error:
                print(f"WARNING: could not read {WIKI_DATASET}/{config} ({type(error).__name__}: {str(error)[:150]}).")
                if stats["sources"]:
                    print("  Keeping the articles read before the error (partial corpus).")
            if stats["sources"]:
                stats["licenses"][WIKI_DATASET] = SOURCE_LICENSES[WIKI_DATASET]
        if not stats["sources"]:
            print("WARNING: Hugging Face Wikipedia unavailable — falling back to NLTK corpora (much smaller).")
            write_articles(nltk_articles(), out, raw_counts, stats, keywords, "nltk")
            stats["licenses"]["nltk"] = SOURCE_LICENSES["nltk"]
        if not stats["sources"]:
            raise RuntimeError("No corpus source could be read. Check the internet connection and re-run.")

        # Coverage before the supplement (holdout words are reported, never targeted).
        mc = cfg["min_count"]
        _, _, before = coverage_report(lambda w: raw_counts[w] >= mc, label="primary source only")
        target_words = {w for n in NON_HOLDOUT_SETS for p, _ in GAME_PHRASE_SETS[n] for w in p.split()}
        needed = {w for w in target_words if raw_counts[w] < cfg["supplement_min_freq"]}
        stats["coverage_before_supplement"] = before
        print(f"Game words (non-holdout) rarer than {cfg['supplement_min_freq']}: {len(needed)}")
        if needed:
            print("Targeted supplement from English Wikipedia for:", sorted(needed)[:60], "..." if len(needed) > 60 else "")
            stats["supplement"] = targeted_supplement(needed, out, raw_counts, stats, keywords)
            if stats["supplement"]["kept"]:
                stats["licenses"][WIKI_DATASET] = SOURCE_LICENSES[WIKI_DATASET]
            print("Supplement:", {k: v for k, v in stats["supplement"].items() if k != "still_needed"},
                  "| still rare:", stats["supplement"]["still_needed"][:40])
        else:
            stats["supplement"] = {"scanned": 0, "kept": 0, "still_needed": []}

    # Phrases: bigrams, then trigrams on the bigram output (connector words allow 'board_of_directors').
    pcfg = cfg["phrases"]
    t_ph = time.perf_counter()
    sentences = LineSentence(str(raw_path))
    bigram = Phrases(sentences, min_count=pcfg["min_count"], threshold=pcfg["threshold"],
                     max_vocab_size=pcfg["max_vocab_size"], connector_words=ENGLISH_CONNECTOR_WORDS).freeze()
    trigram = Phrases(bigram[sentences], min_count=pcfg["min_count"], threshold=pcfg["threshold"],
                      max_vocab_size=pcfg["max_vocab_size"], connector_words=ENGLISH_CONNECTOR_WORDS).freeze()
    stats["phrases_seconds"] = round(time.perf_counter() - t_ph, 1)

    # Apply phrases, count tokens, drop any line equal to an anchor (leakage guard; expected 0).
    anchor_lines = {clean_text(a) for d in DOMAINS for a in DOMAIN_ANCHORS[d]}
    counts, n_lines, anchor_hits = Counter(), 0, 0
    with open(raw_path) as src, open(corpus_path, "w") as out:
        for line in src:
            if line.strip() in anchor_lines:
                anchor_hits += 1
                continue
            tokens = trigram[bigram[line.split()]]
            out.write(" ".join(tokens) + "\n")
            counts.update(tokens)
            n_lines += 1
    raw_path.unlink()
    stats.update({
        "articles": sum(s["articles"] for s in stats["sources"].values()),
        "sentences": n_lines, "tokens_before_phrases": sum(raw_counts.values()), "tokens": sum(counts.values()),
        "vocab_min_count": sum(1 for c in counts.values() if c >= mc),
        "phrase_tokens_min_count": sum(1 for w, c in counts.items() if c >= mc and "_" in w),
        "top_phrases": [w for w, _ in sorted(((w, c) for w, c in counts.items() if "_" in w),
                                              key=lambda x: -x[1])[:40]],
        "anchor_lines_found_in_corpus": anchor_hits, "build_seconds": round(time.perf_counter() - start, 1),
        "domain_estimate": dict(stats["domain_estimate"]),
    })
    bigram.save(str(PATHS.work / "phrases_bigram.pkl"))
    trigram.save(str(PATHS.work / "phrases_trigram.pkl"))
    (PATHS.work / "counts.json").write_text(json.dumps({w: c for w, c in counts.items() if c >= 2}))
    (PATHS.work / "corpus_stats.json").write_text(json.dumps(to_jsonable(stats), indent=1))
    for name in CORPUS_FILES:
        to_store(PATHS.work / name, PATHS.corpus)
    print(f"Corpus built in {stats['build_seconds']:.0f}s and stored in {PATHS.corpus}")
    return stats

with section("4.2 Corpus build / load") as _s:
    if corpus_cached():
        print(f"Loading the cached corpus from {PATHS.corpus} (FORCE_REBUILD = True rebuilds it).")
    elif RUN_DATA_PREP:
        build_corpus()
    else:
        raise RuntimeError("No cached corpus and RUN_DATA_PREP = False. Set RUN_DATA_PREP = True.")
    CORPUS_PATH = from_store("corpus.txt", PATHS.corpus)
    BIGRAM = FrozenPhrases.load(str(from_store("phrases_bigram.pkl", PATHS.corpus)))
    TRIGRAM = FrozenPhrases.load(str(from_store("phrases_trigram.pkl", PATHS.corpus)))
    COUNTS = Counter(json.loads(from_store("counts.json", PATHS.corpus).read_text()))
    CORPUS_STATS = json.loads(from_store("corpus_stats.json", PATHS.corpus).read_text())
    CORPUS_HASH = hashlib.sha1(json.dumps({k: CORPUS_STATS[k] for k in ("sources", "tokens", "sentences", "config")},
                                          sort_keys=True).encode()).hexdigest()[:12]
    print(f"Corpus: {CORPUS_STATS['articles']:,} articles | {CORPUS_STATS['sentences']:,} sentences | "
          f"{CORPUS_STATS['tokens']:,} tokens | vocab (count >= {CORPUS_CONFIG['min_count']}) "
          f"{CORPUS_STATS['vocab_min_count']:,} | phrase tokens {CORPUS_STATS['phrase_tokens_min_count']:,} | "
          f"file {file_mb(CORPUS_PATH)} MB | id {CORPUS_HASH}")

In [ ]:
# 4.3 Corpus statistics, coverage report (pass criterion >= 95%) and leakage checks
# What this cell does:
#   - Prints corpus sources, licenses and size statistics.
#   - Coverage report per set and domain (pass criterion >= 95%).
#   - Leakage checks: no game data or holdout words used to build the corpus.
with section("4.3 Corpus report"):
    print("Sources and licenses:")
    display(pd.DataFrame(CORPUS_STATS["sources"]).T)
    for source, text in CORPUS_STATS["licenses"].items():
        print(f"  {source}: {text}")
    print(f"\nArticles {CORPUS_STATS['articles']:,} | sentences {CORPUS_STATS['sentences']:,} | tokens before phrases "
          f"{CORPUS_STATS['tokens_before_phrases']:,} | tokens after phrases {CORPUS_STATS['tokens']:,} | vocab "
          f"(>= {CORPUS_CONFIG['min_count']}) {CORPUS_STATS['vocab_min_count']:,} | phrase tokens "
          f"{CORPUS_STATS['phrase_tokens_min_count']:,} | build {CORPUS_STATS.get('build_seconds', 0):.0f}s")
    print("Rough topic estimate per article (keyword overlap with each domain's term pool; estimate only):")
    display(pd.Series(CORPUS_STATS["domain_estimate"]).sort_values(ascending=False).to_frame("articles"))
    print("Most frequent learned phrases:", ", ".join(CORPUS_STATS["top_phrases"][:30]))
    sup = CORPUS_STATS.get("supplement", {})
    print(f"Targeted supplement: scanned {sup.get('scanned', 0):,} articles, kept {sup.get('kept', 0):,}; "
          f"still rare after it: {sup.get('still_needed', [])[:30]}")

    # Coverage: a word is covered if it occurs >= min_count times; a phrase is one unit if its
    # underscore form is a token (Phrases merged it).
    mc = CORPUS_CONFIG["min_count"]
    CORPUS_COVERAGE, CORPUS_MISSING, CORPUS_COVERAGE_TOTALS = coverage_report(
        lambda w: COUNTS.get(w, 0) >= mc, lambda p: COUNTS.get("_".join(p.split()), 0) >= mc, label="final corpus")
    print("\nCoverage per set and domain:")
    display(CORPUS_COVERAGE.round(3))
    print("Missing words per set (count < min_count):")
    for name, words in CORPUS_MISSING.items():
        print(f"  {name:<15} {len(words):>3} missing: {words[:40]}")
    game_units = sorted({p for pairs in GAME_PHRASE_SETS.values() for p, _ in pairs
                         if len(p.split()) > 1 and COUNTS.get('_'.join(p.split()), 0) >= mc})
    print(f"Game phrases that Phrases turned into one token ({len(game_units)}):", game_units[:40])
    COVERAGE_PASS = CORPUS_COVERAGE_TOTALS["all game words"] >= CORPUS_CONFIG["coverage_target"]
    print(f"\nPASS criterion (coverage of game words >= {CORPUS_CONFIG['coverage_target']:.0%}): "
          f"{CORPUS_COVERAGE_TOTALS['all game words']:.1%} -> {'PASS' if COVERAGE_PASS else 'FAIL'}")
    if not COVERAGE_PASS:
        print("  Add sources (Section 4.1 CORPUS_CONFIG: raise supplement_max_scan / supplement_time_limit_s) "
              "and rebuild with FORCE_REBUILD = True.")

    # Leakage checks: the builder only reads corpus sources; anchor sentences are not in the corpus;
    # holdout words were never used to select articles.
    holdout_words = {w for p, _ in GAME_PHRASE_SETS["holdout"] for w in p.split()}
    print("\nLeakage checks:")
    print(f"  anchor sentences found (and dropped) in the corpus: {CORPUS_STATS['anchor_lines_found_in_corpus']} "
          "(expected 0)")
    print("  game data written to the corpus: none — build_corpus() writes only text read from the sources")
    print(f"  holdout words used to select supplement articles: none (targets come from {NON_HOLDOUT_SETS})")
    print(f"  holdout words that occur naturally in the corpus: "
          f"{sum(COUNTS.get(w, 0) >= mc for w in holdout_words)}/{len(holdout_words)} (natural text, not leakage)")

## 5. Encoders: an adapter that plugs straight into `LexicalCombatEngine`

`SelfTrainedEncoder` has an `encode(texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)` method with the same signature as `SentenceTransformer.encode` and returns L2-normalised vectors, so it replaces MiniLM without engine changes.

**Representing 1–3 word phrases:** `clean_text()` → if the whole phrase is one token in the vocab (e.g. `machine_learning`), use that vector → otherwise apply the corpus-trained `Phrases` (bigram → trigram) → split tokens that are still unknown into words → combine the word vectors.

**Composition — two options, chosen per model on the calibration set:**
- `mean` — plain average of the (unit-normalised) word vectors, stopwords included
- `sif` — **SIF** (Arora et al., 2017): drop stopwords, weight each word by `a / (a + p(w))` (a = 10⁻³, p(w) from corpus frequency), then **remove the first principal component**, computed once from 20,000 random corpus sentences (no game data, so no leakage, and independent of the batch passed to `encode`). This stops common words from dominating the vectors of long anchor sentences.

**OOV:** Word2Vec / PPMI skip unknown words; if no word is left they return a **zero vector** and `is_known()` is `False` → the engine shows "Unknown word" (turn not counted, no crash). FastText can always build a vector from character n-grams (handles misspellings).

**PPMI + Truncated SVD:** count co-occurrences inside a window (word2vec-style weight `(W−d+1)/W`) over the 50k most frequent words (+ non-holdout game words) → `PPMI(w,c) = max(0, log P(w,c) / (P(w)·P_α(c)))` with context-distribution smoothing α = 0.75 (Levy et al., 2015) → 300-dim randomized SVD → `W = U·√Σ`.

**Intrinsic evaluation:** WordSim-353 (relatedness) and SimLex-999 (similarity) shipped with gensim — **evaluation only**. Spearman on the pairs the model knows, plus the OOV rate.

**Selection score for hyper-parameters / composition / model (calibration set + intrinsic only, holdout never touched):**
`score = 0.35 × gate macro F1 (5-fold grouped CV) + 0.35 × damage in-band (LOTO-CV) + 0.10 × pacing met (LOTO-CV) + 0.20 × mean Spearman (WordSim, SimLex)`. It favours game (extrinsic) results because that is the real goal, and uses **CV** values rather than tuned values to limit overfitting to the calibration set.

In [ ]:
# 5.1 SelfTrainedEncoder (adapter), MiniLM wrapper, PPMI-SVD builder, intrinsic evaluation, selection score
# What this cell does:
#   - SelfTrainedEncoder: turns word vectors into phrase vectors (mean or SIF).
#   - Same encode() signature as SentenceTransformer, so the engine works unchanged.
#   - ReferenceEncoder: wraps MiniLM with the same interface.
#   - PPMI + SVD builder, WordSim / SimLex evaluation and the selection score.
_t_s5 = time.perf_counter()

class SelfTrainedEncoder:
    """Word vectors -> phrase / sentence vectors with the SentenceTransformer.encode() signature."""

    def __init__(self, name: str, family: str, words, vectors, counts: Counter, phrasers=(),
                 composition: str = "sif", oov_vector=None, sif_a: float = 1e-3, info: dict = None):
        self.name, self.family, self.composition = name, family, composition
        self.words = list(words)
        self.key_to_index = {w: i for i, w in enumerate(self.words)}
        vectors = np.asarray(vectors, dtype=np.float32)
        self.unit = vectors / np.maximum(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-12)
        self.dim = int(self.unit.shape[1])
        self.counts, self.total = counts, float(sum(counts.values())) or 1.0
        self.phrasers, self.oov_vector, self.sif_a = tuple(phrasers), oov_vector, sif_a
        self.common_component = None
        self.info = dict(info or {})
        self._cache = {}

    # ---- identity / variants ------------------------------------------------------------------
    def identity(self) -> str:
        return f"{self.family}|{self.info.get('model_hash', self.name)}|{self.composition}"

    def with_composition(self, composition: str) -> "SelfTrainedEncoder":
        """Same vectors, other composition (arrays are shared, cache is not)."""
        other = copy.copy(self)
        other.composition, other._cache = composition, {}
        return other

    # ---- tokens and word vectors --------------------------------------------------------------
    def tokens_for(self, cleaned: str) -> list:
        """Whole phrase as one token if known, else Phrases merges, else single words."""
        words = cleaned.split()
        if not words:
            return []
        joined = "_".join(words)
        if len(words) > 1 and joined in self.key_to_index:
            return [joined]
        tokens = words
        for phraser in self.phrasers:
            tokens = phraser[tokens]
        out = []
        for t in tokens:
            if t in self.key_to_index or "_" not in t:
                out.append(t)
            else:
                out.extend(t.split("_"))
        return out

    def word_vector(self, token: str):
        """Unit vector of one token (FastText: subword vector for OOV), or None."""
        i = self.key_to_index.get(token)
        if i is not None:
            return self.unit[i]
        if self.oov_vector is not None:
            v = self.oov_vector(token)
            if v is not None:
                n = float(np.linalg.norm(v))
                if n > 0:
                    return (v / n).astype(np.float32)
        return None

    def weight(self, token: str) -> float:
        """SIF weight a / (a + p(w)); 1.0 for the plain mean."""
        if self.composition != "sif":
            return 1.0
        return self.sif_a / (self.sif_a + self.counts.get(token, 0) / self.total)

    def _compose(self, tokens: list):
        """(vector before normalisation, known tokens, content tokens)."""
        content = ([t for t in tokens if t not in STOPWORDS] or tokens) if self.composition == "sif" else tokens
        vecs, weights = [], []
        for t in content:
            v = self.word_vector(t)
            if v is not None:
                vecs.append(v)
                weights.append(self.weight(t))
        if not vecs:
            return np.zeros(self.dim, dtype=np.float32), 0, len(content)
        v = np.average(np.vstack(vecs), axis=0, weights=np.asarray(weights, dtype=np.float64))
        if self.composition == "sif" and self.common_component is not None:
            v = v - float(v @ self.common_component) * self.common_component
        return v.astype(np.float32), len(vecs), len(content)

    def _vector(self, text: str):
        hit = self._cache.get(text)
        if hit is None:
            hit = self._compose(self.tokens_for(clean_text(text)))
            if len(self._cache) < 200_000:
                self._cache[text] = hit
        return hit

    def fit_common_component(self, sentences, max_n: int = 20_000) -> None:
        """First singular vector of SIF-weighted corpus sentence vectors (Arora et al., 2017)."""
        saved, self.common_component = self.composition, None
        self.composition = "sif"
        rows = [self._compose(list(toks))[0] for toks in sentences[:max_n]]
        self.composition = saved
        matrix = np.vstack([r for r in rows if np.any(r)])
        _, _, vt = randomized_svd(matrix, n_components=1, n_iter=5, random_state=SEED)
        self.common_component = (vt[0] / np.linalg.norm(vt[0])).astype(np.float32)
        self._cache = {}

    # ---- SentenceTransformer-compatible API --------------------------------------------------
    def encode(self, sentences, convert_to_numpy: bool = True, normalize_embeddings: bool = True,
               show_progress_bar: bool = False, batch_size: int = None, **kwargs):
        """L2-normalised vectors, shape (n, dim) (or (dim,) for one string). Unknown text -> zero vector."""
        single = isinstance(sentences, str)
        texts = [sentences] if single else list(sentences)
        out = np.zeros((len(texts), self.dim), dtype=np.float32)
        for i, text in enumerate(texts):
            out[i] = self._vector(str(text))[0]
        if normalize_embeddings:
            norms = np.linalg.norm(out, axis=1, keepdims=True)
            out = np.divide(out, norms, out=np.zeros_like(out), where=norms > 0)
        return out[0] if single else out

    def is_known(self, text: str) -> bool:
        """True if at least one word of the text has a vector."""
        return self._vector(str(text))[1] > 0

    def in_vocab(self, word: str) -> bool:
        return clean_text(word).replace(" ", "_") in self.key_to_index

    def coverage_kind(self, text: str) -> str:
        """'phrase unit' | 'in vocab' | 'subword (FastText)' | 'partial' | 'unknown'."""
        words = clean_text(text).split()
        tokens = self.tokens_for(" ".join(words))
        known = [t for t in tokens if self.word_vector(t) is not None]
        if not known:
            return "unknown"
        if len(known) < len(tokens):
            return "partial"
        if any(t not in self.key_to_index for t in tokens):
            return "subword (FastText)"
        return "phrase unit" if len(tokens) < len(words) else "in vocab"

    def word_matrix(self, words) -> tuple:
        """Raw unit word vectors (no SIF component removal) and a known mask, for intrinsic tests."""
        out, known = np.zeros((len(words), self.dim), dtype=np.float32), np.zeros(len(words), dtype=bool)
        for i, w in enumerate(words):
            toks = self.tokens_for(clean_text(w))
            vecs = [v for v in (self.word_vector(t) for t in toks) if v is not None]
            if vecs:
                v = np.mean(vecs, axis=0)
                out[i], known[i] = v / max(np.linalg.norm(v), 1e-12), True
        return out, known

    def neighbours(self, word: str, topn: int = 8, search_top: int = 100_000) -> list:
        """Nearest vocabulary tokens (raw word vectors) among the most frequent search_top words."""
        matrix, known = self.word_matrix([word])
        if not known[0]:
            return []
        limit = min(search_top, len(self.words))
        order = self._freq_order()[:limit]
        if getattr(self, "_nn_limit", None) != limit:
            self._nn_matrix, self._nn_limit = self.unit[order], limit
        sims = self._nn_matrix @ matrix[0]
        skip = set(clean_text(word).split()) | {clean_text(word).replace(" ", "_")}
        best = []
        for j in np.argsort(-sims):
            token = self.words[order[j]]
            if token not in skip and not (set(token.split("_")) & skip):
                best.append((token, round(float(sims[j]), 3)))
            if len(best) >= topn:
                break
        return best

    def _freq_order(self) -> np.ndarray:
        if not hasattr(self, "_order"):
            self._order = np.argsort([-self.counts.get(w, 0) for w in self.words], kind="stable")
        return self._order

class ReferenceEncoder:
    """Thin wrapper around a SentenceTransformer (MiniLM reference baseline only)."""
    family, composition = "minilm_reference", "transformer"

    def __init__(self, model, name: str):
        self.model, self.name, self.info = model, name, {"source": name, "pretrained": True}
        self.dim = int(model.get_sentence_embedding_dimension())

    def identity(self) -> str:
        return f"minilm|{self.name}"

    def encode(self, sentences, **kwargs):
        options = {"convert_to_numpy": True, "normalize_embeddings": True, "show_progress_bar": False, **kwargs}
        return self.model.encode(sentences, **options)

    def is_known(self, text: str) -> bool:
        return bool(clean_text(text))   # sub-word tokenizer: every non-empty input has a vector

    def coverage_kind(self, text: str) -> str:
        return "subword (WordPiece)"

    def word_matrix(self, words) -> tuple:
        return self.encode(list(words)), np.ones(len(words), dtype=bool)

    def neighbours(self, word: str, topn: int = 8, search_top: int = 0) -> list:
        return []

# ---- PPMI + truncated SVD (count-based, built from scratch) ----------------------------------
def cooccurrence_matrices(corpus_path: Path, vocab: list, windows=(2, 5), chunk_tokens: int = 4_000_000) -> dict:
    """Weighted symmetric co-occurrence counts for each window (weight (W - d + 1) / W, same sentence only)."""
    index = {w: i for i, w in enumerate(vocab)}
    V, W = len(vocab), max(windows)
    mats = {win: sp.csr_matrix((V, V), dtype=np.float32) for win in windows}
    ids_buf, sid_buf = [], []

    def flush():
        if not ids_buf:
            return
        ids, sid = np.asarray(ids_buf, dtype=np.int32), np.asarray(sid_buf, dtype=np.int32)
        pairs = {}
        for d in range(1, W + 1):
            a, b = ids[:-d], ids[d:]
            m = (sid[:-d] == sid[d:]) & (a >= 0) & (b >= 0)
            pairs[d] = (np.concatenate([a[m], b[m]]), np.concatenate([b[m], a[m]]))
        for win in windows:
            ds = [d for d in pairs if d <= win]
            rows = np.concatenate([pairs[d][0] for d in ds])
            cols = np.concatenate([pairs[d][1] for d in ds])
            data = np.concatenate([np.full(len(pairs[d][0]), (win - d + 1) / win, dtype=np.float32) for d in ds])
            mats[win] = mats[win] + sp.coo_matrix((data, (rows, cols)), shape=(V, V)).tocsr()
            del rows, cols, data
        del pairs
        ids_buf.clear()
        sid_buf.clear()

    with open(corpus_path) as handle:
        for line_no, line in enumerate(handle):
            tokens = line.split()
            ids_buf.extend(index.get(t, -1) for t in tokens)
            sid_buf.extend(itertools.repeat(line_no, len(tokens)))
            if len(ids_buf) >= chunk_tokens:
                flush()
    flush()
    return mats

def ppmi_svd_vectors(counts_matrix, dim: int, alpha: float = 0.75) -> np.ndarray:
    """PPMI with context-distribution smoothing, then randomized SVD. Returns (U * sqrt(S), PPMI nnz)."""
    coo = counts_matrix.tocoo()
    total = float(coo.data.sum())
    row = np.asarray(counts_matrix.sum(axis=1)).ravel()
    col = np.asarray(counts_matrix.sum(axis=0)).ravel()
    p_w = row / total
    p_c = col ** alpha / (col ** alpha).sum()
    pmi = np.log(coo.data / total) - np.log(p_w[coo.row]) - np.log(p_c[coo.col])
    keep = pmi > 0
    ppmi = sp.csr_matrix((pmi[keep].astype(np.float32), (coo.row[keep], coo.col[keep])), shape=counts_matrix.shape)
    u, s, _ = randomized_svd(ppmi, n_components=dim, n_iter=4, random_state=SEED)
    return (u * np.sqrt(s)).astype(np.float32), int(ppmi.nnz)

# ---- intrinsic evaluation (WordSim-353 / SimLex-999; evaluation only) ------------------------
def load_word_pairs(filename: str) -> pd.DataFrame:
    rows = []
    for line in Path(datapath(filename)).read_text(encoding="utf-8").splitlines():
        if not line.strip() or line.startswith("#"):
            continue
        parts = line.split("\t")
        if len(parts) >= 3:
            rows.append((clean_text(parts[0]), clean_text(parts[1]), float(parts[2])))
    return pd.DataFrame(rows, columns=["w1", "w2", "human"])

INTRINSIC_SETS = {"WordSim-353": load_word_pairs("wordsim353.tsv"), "SimLex-999": load_word_pairs("simlex999.txt")}

def intrinsic_scores(encoder) -> dict:
    """Spearman on pairs the model knows + OOV pair rates (vocabulary OOV reported separately)."""
    out = {}
    for name, pairs in INTRINSIC_SETS.items():
        words = sorted(set(pairs["w1"]) | set(pairs["w2"]))
        matrix, known = encoder.word_matrix(words)
        pos = {w: i for i, w in enumerate(words)}
        i1, i2 = pairs["w1"].map(pos).to_numpy(), pairs["w2"].map(pos).to_numpy()
        mask = known[i1] & known[i2]
        sims = np.einsum("ij,ij->i", matrix[i1], matrix[i2])
        rho = spearmanr(pairs["human"][mask], sims[mask]).correlation if mask.sum() > 2 else float("nan")
        out[f"{name} Spearman"] = float(rho)
        out[f"{name} OOV pairs"] = float(1 - mask.mean())
        if isinstance(encoder, SelfTrainedEncoder):
            in_vocab = np.array([encoder.in_vocab(w) for w in words])
            out[f"{name} vocab-OOV pairs"] = float(1 - (in_vocab[i1] & in_vocab[i2]).mean())
    return out

def selection_metrics(encoder, max_rounds: int = 2) -> dict:
    """Calibration-set-only score (gate CV, damage LOTO-CV, pacing) + intrinsic. Never touches the holdout."""
    start = time.perf_counter()
    eng = LexicalCombatEngine(encoder, ENGINE_DATA, EngineConfig())
    gate = calibrate_with_expansion("gate", eng, gate_set_df, max_rounds=max_rounds, verbose=False)
    eng.config = replace(eng.config, **gate["params"])
    damage = calibrate_with_expansion("damage", eng, max_rounds=max_rounds, verbose=False)
    intrinsic = intrinsic_scores(encoder)
    pace = damage["cv_pace"]
    pacing_cv = float(pace["met"].mean()) if len(pace) else 0.0
    rho = np.nanmean([max(intrinsic["WordSim-353 Spearman"], 0), max(intrinsic["SimLex-999 Spearman"], 0)])
    score = (0.35 * gate["cv"]["macro_f1"] + 0.35 * damage["cv_in_band"] + 0.10 * pacing_cv
             + 0.20 * (0.0 if np.isnan(rho) else rho))
    return {"score": round(float(score), 4), "gate macro F1 (CV)": gate["cv"]["macro_f1"],
            "gate leak (CV)": gate["cv"]["out_leak"], "in-domain primary (CV)": gate["cv"]["in_domain_primary"],
            "damage in-band (CV)": damage["cv_in_band"], "pacing met (CV)": pacing_cv,
            "WordSim-353": intrinsic["WordSim-353 Spearman"], "SimLex-999": intrinsic["SimLex-999 Spearman"],
            "anisotropy": eng.noise_pair_mean, "eval seconds": round(time.perf_counter() - start, 1)}

log_section("5. Encoders", _t_s5)

## 6. Training from scratch (Phase 3)

Every model starts from **random initialization** on the same corpus (Section 4) — no pretrained weights are loaded.

| Step | Details |
|---|---|
| 6.1 Prepare | Sample 20,000 corpus sentences (for the SIF common component) and write a **search sample** file (every k-th line ≈ 15% of the corpus) |
| 6.2 PPMI-SVD | Count co-occurrences once for both window 2 and 5 → 300-dim SVD · search: window ∈ {2, 5} × dim ∈ {100, 300} (100 dims = first 100 components of the same SVD) × composition ∈ {mean, sif} on the full corpus (PPMI has no epochs, so no sample is needed) |
| 6.3 Word2Vec search | `vector_size` ∈ {100, 300} × `window` ∈ {5, 10} × `sg` ∈ {0 (CBOW), 1 (skip-gram)} = 8 configs on the search sample, 2 epochs, × composition ∈ {mean, sif} |
| 6.3 FastText search | FastText is ~2–3× slower than Word2Vec, so only `vector_size` ∈ {100, 300} is searched, with skip-gram (`sg=1`) and the best Word2Vec `window` · `min_n=3`, `max_n=6`, `bucket=500k` (down from the 2M default so the file is ~600 MB instead of 2.4 GB) |
| 6.4 Final | Train the best config of each family on the full corpus (`negative=10`, `min_count=5`, `epochs=5`). If the time estimate (from the throughput measured during the search) exceeds the budget (Word2Vec 20 min, FastText 25 min), epochs are reduced (minimum 3); if it is still too long, part of the corpus is used, and this is printed honestly · composition is chosen again on the final model · `MODEL_CHOICE = "auto"` = family with the highest selection score |

All selection values come from the **calibration set + intrinsic benchmarks** only (formula in Section 5). Search results, models and config JSON files are saved to Drive, so the next run loads them instead of retraining.

**Reproducibility:** seeds are set everywhere, and the training config, corpus size, time, loss and gensim version are saved to `config.json` next to each model. **Limitation:** multi-threaded gensim (`workers > 1`) is not 100% deterministic because the order of weight updates between threads varies (bit-exact runs need `workers=1` and `PYTHONHASHSEED` set before the interpreter starts, which is much slower), so numbers may differ slightly between runs. Training loss is only available for Word2Vec (gensim does not compute FastText loss).

In [ ]:
# 6.1 Training tools: corpus samples, gensim training with a loss / time log, saving and loading models
# What this cell does:
#   - Training settings (epochs, time budgets, vocab sizes, search grids).
#   - Writes the search sample and the SIF sentence sample.
#   - Helpers to train gensim models with a loss / time log and save / load them.
_t_s6 = time.perf_counter()
FAMILIES = ["ppmi_svd", "word2vec", "fasttext"]
SEARCH_FRACTION = 0.15                      # share of corpus lines used for the Word2Vec / FastText search
SEARCH_EPOCHS = 2
FINAL_EPOCHS = 2 if QUICK_RUN else 5
TRAIN_BUDGET_MIN = {"word2vec": 3, "fasttext": 4} if QUICK_RUN else {"word2vec": 20, "fasttext": 25}
MIN_FINAL_EPOCHS = 2 if QUICK_RUN else 3
PPMI_VOCAB = 20_000 if QUICK_RUN else 50_000
PPMI_WINDOWS, PPMI_DIMS = ((5,), (100,)) if QUICK_RUN else ((2, 5), (100, 300))
FASTTEXT_BUCKET = 200_000 if QUICK_RUN else 500_000
COMMON = {"negative": 10, "min_count": CORPUS_CONFIG["min_count"], "sample": 1e-3}
W2V_GRID = ([{"vector_size": 100, "window": 5, "sg": s} for s in (1, 0)] if QUICK_RUN else
            [{"vector_size": v, "window": w, "sg": s} for v in (100, 300) for w in (5, 10) for s in (0, 1)])
FT_VECTOR_SIZES = (100,) if QUICK_RUN else (100, 300)
COMPOSITIONS = ["mean", "sif"]
NONDETERMINISM_NOTE = ("gensim with workers > 1 is not bit-for-bit reproducible (thread scheduling changes the "
                       "update order); use workers=1 and a fixed PYTHONHASHSEED for exact repeats.")

def sample_lines(path: Path, every: int, out_path: Path = None, limit: int = None) -> list:
    """Every k-th line (deterministic). Writes them to out_path, or returns token lists."""
    rows = []
    with open(path) as src, (open(out_path, "w") if out_path else open(os.devnull, "w")) as out:
        for i, line in enumerate(src):
            if i % every:
                continue
            if out_path:
                out.write(line)
            else:
                rows.append(line.split())
                if limit and len(rows) >= limit:
                    break
    return rows

class EpochLogger(CallbackAny2Vec):
    """Per-epoch time and (Word2Vec only) loss; gensim's running loss is cumulative, so we store deltas."""
    def __init__(self):
        self.rows, self._previous, self._start = [], 0.0, time.perf_counter()

    def on_epoch_end(self, model):
        loss = float(model.get_latest_training_loss()) if getattr(model, "compute_loss", False) else float("nan")
        delta = loss - self._previous if loss == loss else float("nan")
        self._previous = loss if loss == loss else self._previous
        self.rows.append({"epoch": len(self.rows) + 1, "loss": round(delta, 1),
                          "seconds": round(time.perf_counter() - self._start, 1)})

def train_gensim(family: str, params: dict, corpus_file: Path, epochs: int) -> tuple:
    """Train Word2Vec / FastText from random init on a LineSentence file. Returns (model, info)."""
    logger = EpochLogger()
    kwargs = dict(corpus_file=str(corpus_file), epochs=epochs, workers=WORKERS, seed=SEED,
                  callbacks=[logger], **COMMON, **params)
    start = time.perf_counter()
    if family == "word2vec":
        model = Word2Vec(compute_loss=True, **kwargs)
    else:
        model = FastText(min_n=3, max_n=6, bucket=FASTTEXT_BUCKET, **kwargs)
    seconds = time.perf_counter() - start
    words = int(model.corpus_total_words) * epochs
    return model, {"family": family, "params": {**COMMON, **params, "epochs": epochs,
                                                **({"min_n": 3, "max_n": 6, "bucket": FASTTEXT_BUCKET}
                                                   if family == "fasttext" else {})},
                   "train_seconds": round(seconds, 1), "words_per_second": round(words / max(seconds, 1e-9)),
                   "corpus_words": int(model.corpus_total_words), "vocab": len(model.wv), "epoch_log": logger.rows}

def encoder_from_kv(name: str, family: str, kv, composition: str, info: dict) -> SelfTrainedEncoder:
    """Adapter around gensim KeyedVectors (FastText also gets its subword OOV function)."""
    oov = (lambda w: kv.get_vector(w)) if family == "fasttext" else None
    enc = SelfTrainedEncoder(name, family, kv.index_to_key, kv.vectors, COUNTS, (BIGRAM, TRIGRAM),
                             composition, oov_vector=oov, info=info)
    enc.fit_common_component(SIF_SAMPLE)
    return enc

def model_dir(family: str) -> Path:
    path = PATHS.models / family
    path.mkdir(parents=True, exist_ok=True)
    return path

def model_hash(info: dict) -> str:
    keys = {k: info.get(k) for k in ("family", "params", "corpus_id", "ppmi_window", "ppmi_dim", "train_fraction")}
    return hashlib.sha1(json.dumps(keys, sort_keys=True, default=str).encode()).hexdigest()[:12]

def write_model_config(family: str, info: dict) -> dict:
    info = {**info, "corpus_id": CORPUS_HASH, "seed": SEED, "workers": WORKERS, "gensim": gensim.__version__,
            "quick_run": QUICK_RUN, "nondeterminism": NONDETERMINISM_NOTE,
            "corpus": {k: CORPUS_STATS.get(k) for k in ("articles", "sentences", "tokens", "vocab_min_count",
                                                          "phrase_tokens_min_count", "sources")}}
    info["model_hash"] = model_hash(info)
    (model_dir(family) / "config.json").write_text(json.dumps(to_jsonable(info), indent=1))
    return info

def read_model_config(family: str):
    path = model_dir(family) / "config.json"
    if FORCE_REBUILD or not path.is_file():
        return None
    info = json.loads(path.read_text())
    return info if info.get("corpus_id") == CORPUS_HASH else None

with section("6.1 Training tools"):
    _every = max(1, CORPUS_STATS["sentences"] // 20_000)
    SIF_SAMPLE = sample_lines(CORPUS_PATH, _every, limit=20_000)
    SEARCH_CORPUS = PATHS.work / "corpus_search.txt"
    _search_every = max(1, round(1 / SEARCH_FRACTION))
    if not SEARCH_CORPUS.is_file() or FORCE_REBUILD:
        sample_lines(CORPUS_PATH, _search_every, SEARCH_CORPUS)
    print(f"SIF sample: {len(SIF_SAMPLE):,} sentences | search sample: every {_search_every}th line "
          f"({file_mb(SEARCH_CORPUS)} MB) | final epochs {FINAL_EPOCHS} | budget (min) {TRAIN_BUDGET_MIN} | "
          f"workers {WORKERS}")

In [ ]:
# 6.2 PPMI + truncated SVD: build (or load), variant search on the calibration set
# What this cell does:
#   - Builds (or loads) PPMI co-occurrence matrices and SVD vectors.
#   - Searches window x dim x composition on the calibration set.
def ppmi_vocabulary() -> list:
    """Most frequent PPMI_VOCAB tokens + non-holdout game words that pass min_count (holdout never used)."""
    mc = CORPUS_CONFIG["min_count"]
    top = [w for w, c in COUNTS.most_common(PPMI_VOCAB) if c >= mc]
    seen = set(top)
    game = {t for n in NON_HOLDOUT_SETS for p, _ in GAME_PHRASE_SETS[n] for t in [p.replace(" ", "_")] + p.split()}
    return top + sorted(w for w in game if COUNTS.get(w, 0) >= mc and w not in seen)

def ppmi_encoder(vectors: dict, vocab: list, info: dict, window: int, dim: int) -> SelfTrainedEncoder:
    """SIF encoder (common component fitted) for one PPMI window / dimension."""
    variant = {**info, "ppmi_window": window, "ppmi_dim": dim}
    variant["model_hash"] = model_hash(variant)
    enc = SelfTrainedEncoder(f"ppmi_svd w{window} d{dim}", "ppmi_svd", vocab, vectors[window][:, :dim], COUNTS,
                             (BIGRAM, TRIGRAM), "sif", info=variant)
    enc.fit_common_component(SIF_SAMPLE)
    return enc

with section("6.2 PPMI-SVD") as _s:
    _folder = model_dir("ppmi_svd")
    PPMI_INFO = read_model_config("ppmi_svd")
    _ready = PPMI_INFO is not None and (_folder / "vocab.json").is_file() and all(
        (_folder / f"vectors_w{w}.npy").is_file() for w in PPMI_WINDOWS)
    PPMI_VECTORS, PPMI_VOCAB_LIST = None, None
    if _ready:
        PPMI_VOCAB_LIST = json.loads((_folder / "vocab.json").read_text())
        PPMI_VECTORS = {w: np.load(_folder / f"vectors_w{w}.npy") for w in PPMI_WINDOWS}
        print(f"Loaded PPMI-SVD from {_folder} ({len(PPMI_VOCAB_LIST):,} words, windows {PPMI_WINDOWS}).")
    elif RUN_TRAINING:
        _t = time.perf_counter()
        PPMI_VOCAB_LIST = ppmi_vocabulary()
        _mats = cooccurrence_matrices(CORPUS_PATH, PPMI_VOCAB_LIST, PPMI_WINDOWS)
        _t_count = time.perf_counter() - _t
        PPMI_VECTORS, _nnz = {}, {}
        for _w in PPMI_WINDOWS:
            PPMI_VECTORS[_w], _nnz[_w] = ppmi_svd_vectors(_mats.pop(_w), max(PPMI_DIMS))   # frees the counts
            np.save(_folder / f"vectors_w{_w}.npy", PPMI_VECTORS[_w])
            gc.collect()
        del _mats
        gc.collect()
        (_folder / "vocab.json").write_text(json.dumps(PPMI_VOCAB_LIST))
        PPMI_INFO = write_model_config("ppmi_svd", {
            "family": "ppmi_svd", "params": {"windows": list(PPMI_WINDOWS), "max_dim": max(PPMI_DIMS),
                                             "vocab": len(PPMI_VOCAB_LIST), "cds_alpha": 0.75,
                                             "window_weight": "(W - d + 1) / W", "svd": "randomized, n_iter=4",
                                             "eigen_weight": 0.5},
            "count_seconds": round(_t_count, 1), "train_seconds": round(time.perf_counter() - _t, 1),
            "ppmi_nnz": _nnz, "model_mb": file_mb(_folder)})
        print(f"PPMI-SVD built in {PPMI_INFO['train_seconds']:.0f}s (counting {_t_count:.0f}s) | "
              f"vocab {len(PPMI_VOCAB_LIST):,} | PPMI nnz {_nnz} | {file_mb(_folder)} MB")
    else:
        _s.skip("no saved PPMI-SVD and RUN_TRAINING = False")

    PPMI_SEARCH = None
    if PPMI_VECTORS is not None:
        _path = PATHS.artifacts / "hp_search_ppmi_svd.csv"
        _variants = [(w, d) for w in PPMI_WINDOWS for d in PPMI_DIMS]
        if _path.is_file() and not FORCE_REBUILD:
            _cached = pd.read_csv(_path)
            if set(_cached["corpus_id"]) == {CORPUS_HASH} and len(_cached) == len(_variants) * len(COMPOSITIONS):
                PPMI_SEARCH = _cached
                print(f"Loaded PPMI variant search from {_path.name}.")
        if PPMI_SEARCH is None and RUN_HP_SEARCH:
            _rows = []
            for _w, _d in _variants:
                _enc = ppmi_encoder(PPMI_VECTORS, PPMI_VOCAB_LIST, PPMI_INFO, _w, _d)
                for _comp in COMPOSITIONS:
                    _m = selection_metrics(_enc if _comp == "sif" else _enc.with_composition(_comp))
                    _rows.append({"window": _w, "dim": _d, "composition": _comp, **_m, "corpus_id": CORPUS_HASH})
                    print(f"  PPMI w{_w} d{_d} {_comp}: score {_m['score']:.3f} ({_m['eval seconds']}s)")
            PPMI_SEARCH = pd.DataFrame(_rows)
            PPMI_SEARCH.to_csv(_path, index=False)
        if PPMI_SEARCH is not None:
            print("PPMI-SVD variants (calibration set + intrinsic only):")
            display(PPMI_SEARCH.drop(columns=["corpus_id"]).sort_values("score", ascending=False).round(3))
            _best = PPMI_SEARCH.sort_values("score", ascending=False).iloc[0]
            PPMI_CHOICE = {"window": int(_best["window"]), "dim": int(_best["dim"])}
        else:
            PPMI_CHOICE = {"window": max(PPMI_WINDOWS), "dim": max(PPMI_DIMS)}
            print("RUN_HP_SEARCH = False — using window", PPMI_CHOICE["window"], "dim", PPMI_CHOICE["dim"])
        print("PPMI-SVD choice:", PPMI_CHOICE)

In [ ]:
# 6.3 Word2Vec and FastText hyper-parameter search on the search sample (calibration set + intrinsic only)
# What this cell does:
#   - Word2Vec search: vector_size x window x sg on the search sample.
#   - FastText search: vector_size at the best Word2Vec window.
#   - Each config is scored with both compositions; results cached as CSV.
def search_family(family: str, configs: list, allow_run: bool) -> pd.DataFrame:
    """Train each config on the search sample, score both compositions; hp_search_<family>.csv is written after
    every config, so a disconnected Colab session resumes where it stopped. Returns None when nothing is cached
    and allow_run is False."""
    path = PATHS.artifacts / f"hp_search_{family}.csv"
    keys = ["vector_size", "window", "sg"]
    key_of = lambda c: tuple(int(c[k]) for k in keys)
    wanted = [key_of(c) for c in configs]
    rows = []
    if path.is_file() and not FORCE_REBUILD:
        rows = [r for r in pd.read_csv(path).to_dict(orient="records")
                if r.get("corpus_id") == CORPUS_HASH and key_of(r) in wanted]
    done = {key_of(r) for r in rows if sum(key_of(x) == key_of(r) for x in rows) == len(COMPOSITIONS)}
    if done == set(wanted):
        print(f"Loaded {family} search from {path.name}.")
        return pd.DataFrame(rows)
    if not allow_run:
        return pd.DataFrame(rows) if rows else None
    if done:
        print(f"Resuming {family} search: {len(done)}/{len(wanted)} configs already in {path.name}.")
    rows = [r for r in rows if key_of(r) in done]
    for params in configs:
        if key_of(params) in done:
            continue
        model, info = train_gensim(family, params, SEARCH_CORPUS, SEARCH_EPOCHS)
        info["model_hash"] = "search-" + hashlib.sha1(json.dumps([family, params, CORPUS_HASH]).encode()).hexdigest()[:10]
        base = encoder_from_kv(f"{family} search", family, model.wv, "sif", info)
        del model
        for comp in COMPOSITIONS:
            metrics = selection_metrics(base if comp == "sif" else base.with_composition(comp))
            rows.append({**params, "composition": comp, **metrics, "train_seconds": info["train_seconds"],
                         "words_per_second": info["words_per_second"], "vocab": info["vocab"], "corpus_id": CORPUS_HASH})
            print(f"  {family} {params} {comp}: score {metrics['score']:.3f} | train {info['train_seconds']:.0f}s "
                  f"({info['words_per_second']:,} words/s) | eval {metrics['eval seconds']}s")
        del base
        gc.collect()
        pd.DataFrame(rows).to_csv(path, index=False)    # checkpoint after every config
    return pd.DataFrame(rows)

def best_config(frame: pd.DataFrame) -> dict:
    row = frame.sort_values("score", ascending=False).iloc[0]
    return {"vector_size": int(row["vector_size"]), "window": int(row["window"]), "sg": int(row["sg"])}

DEFAULT_PARAMS = {"vector_size": 300, "window": 5, "sg": 1}   # the suggested starting values

with section("6.3 Word2Vec / FastText search") as _s:
    _allow = RUN_HP_SEARCH and RUN_TRAINING
    SEARCH_RESULTS = {"word2vec": search_family("word2vec", W2V_GRID, _allow)}
    _w2v_best = best_config(SEARCH_RESULTS["word2vec"]) if SEARCH_RESULTS["word2vec"] is not None else DEFAULT_PARAMS
    FT_GRID = [{"vector_size": v, "window": _w2v_best["window"], "sg": 1} for v in FT_VECTOR_SIZES]
    SEARCH_RESULTS["fasttext"] = search_family("fasttext", FT_GRID, _allow)
    SEARCH_RESULTS = {f: r for f, r in SEARCH_RESULTS.items() if r is not None}
    BEST_PARAMS = {f: best_config(SEARCH_RESULTS[f]) if f in SEARCH_RESULTS else dict(DEFAULT_PARAMS)
                   for f in ("word2vec", "fasttext")}
    if not SEARCH_RESULTS:
        _s.skip("no search results and RUN_HP_SEARCH / RUN_TRAINING is False — using the default configs")
    for _family, _frame in SEARCH_RESULTS.items():
        print(f"\n{_family} search ({SEARCH_EPOCHS} epochs on {SEARCH_FRACTION:.0%} of the corpus):")
        display(_frame.drop(columns=["corpus_id"]).sort_values("score", ascending=False).round(3))
    print("Best configs:", BEST_PARAMS)

In [ ]:
# 6.4 Final training on the full corpus (budget-aware), composition choice, auto model, neighbour check
# What this cell does:
#   - Plans epochs / corpus share to fit the time budget.
#   - Trains (or loads) the final Word2Vec and FastText models on the full corpus.
#   - Chooses composition per model and the best self-trained model (auto).
#   - Prints nearest neighbours as a sanity check.
def plan_final(family: str, params: dict) -> dict:
    """Epochs and corpus share that fit the time budget, from the throughput measured in the search."""
    plan = {"epochs": FINAL_EPOCHS, "fraction": 1.0, "estimate_min": None}
    frame = SEARCH_RESULTS.get(family)
    if frame is None:
        return plan
    rows = frame[(frame["vector_size"] == params["vector_size"]) & (frame["window"] == params["window"])
                 & (frame["sg"] == params["sg"])]
    if rows.empty:
        return plan
    epoch_s = CORPUS_STATS["tokens"] / float(rows["words_per_second"].iloc[0])
    budget_s = TRAIN_BUDGET_MIN[family] * 60
    epochs = int(min(FINAL_EPOCHS, max(MIN_FINAL_EPOCHS, budget_s // max(epoch_s, 1e-9))))
    fraction = 1.0 if epochs * epoch_s <= budget_s * 1.25 else max(0.3, budget_s / (epochs * epoch_s))
    plan.update(epochs=epochs, fraction=round(fraction, 3), estimate_min=round(epochs * epoch_s * fraction / 60, 1))
    return plan

def subset_corpus(fraction: float, name: str) -> Path:
    """Deterministic share of corpus lines (hash of the line number)."""
    out = PATHS.work / name
    with open(CORPUS_PATH) as src, open(out, "w") as dst:
        for i, line in enumerate(src):
            if (i * 2654435761) % 1000 < fraction * 1000:
                dst.write(line)
    return out

def train_or_load_final(family: str) -> tuple:
    """(KeyedVectors, info) — loaded when a matching saved model exists, else trained and saved."""
    folder = model_dir(family)
    info = read_model_config(family)
    if info is not None and info.get("params", {}).get("vector_size") == BEST_PARAMS[family]["vector_size"] \
            and info["params"].get("window") == BEST_PARAMS[family]["window"] \
            and info["params"].get("sg") == BEST_PARAMS[family]["sg"] and (folder / "vectors.kv").is_file():
        print(f"Loading {family} from {folder} (trained {info['train_seconds']:.0f}s, {info['params']}).")
        return KeyedVectors.load(str(folder / "vectors.kv")), info
    if not RUN_TRAINING:
        raise RuntimeError(f"No saved {family} model matching {BEST_PARAMS[family]} and RUN_TRAINING = False.")
    plan = plan_final(family, BEST_PARAMS[family])
    corpus = CORPUS_PATH if plan["fraction"] >= 1.0 else subset_corpus(plan["fraction"], f"corpus_final_{family}.txt")
    print(f"Training {family} {BEST_PARAMS[family]} | epochs {plan['epochs']} | corpus share {plan['fraction']:.0%} | "
          f"estimate {plan['estimate_min']} min")
    if plan["epochs"] < FINAL_EPOCHS or plan["fraction"] < 1.0:
        print(f"  NOTE: reduced from {FINAL_EPOCHS} epochs / full corpus to stay within ~{TRAIN_BUDGET_MIN[family]} min "
              "(Colab free); reported in config.json.")
    model, info = train_gensim(family, BEST_PARAMS[family], corpus, plan["epochs"])
    for name in ("vectors.kv",):
        for old in folder.glob(name + "*"):
            old.unlink()
    model.wv.save(str(folder / "vectors.kv"))
    del model
    gc.collect()
    info = write_model_config(family, {**info, "train_fraction": plan["fraction"], "plan": plan,
                                       "model_mb": file_mb(folder)})
    print(f"  done in {info['train_seconds']:.0f}s ({info['words_per_second']:,} words/s) | vocab {info['vocab']:,} | "
          f"{info['model_mb']} MB saved to {folder}")
    return KeyedVectors.load(str(folder / "vectors.kv")), info   # reload from disk: proves the files work

def final_selection(family: str, encoder: SelfTrainedEncoder) -> pd.DataFrame:
    """Both compositions of the final model on the calibration set (cached per model hash)."""
    path = PATHS.artifacts / f"final_selection_{family}.csv"
    if path.is_file() and not FORCE_REBUILD:
        cached = pd.read_csv(path)
        if set(cached["model_hash"]) == {encoder.info["model_hash"]} and len(cached) == len(COMPOSITIONS):
            return cached
    rows = [{"family": family, "composition": c, **selection_metrics(encoder.with_composition(c)),
             "model_hash": encoder.info["model_hash"]} for c in COMPOSITIONS]
    frame = pd.DataFrame(rows)
    frame.to_csv(path, index=False)
    return frame

with section("6.4 Final models") as _s:
    MODEL_REGISTRY, MODEL_INFO, _selection = {}, {}, []
    if PPMI_VECTORS is not None:
        _enc = ppmi_encoder(PPMI_VECTORS, PPMI_VOCAB_LIST, PPMI_INFO, PPMI_CHOICE["window"], PPMI_CHOICE["dim"])
        _enc.name = "PPMI-SVD"
        MODEL_REGISTRY["ppmi_svd"], MODEL_INFO["ppmi_svd"] = _enc, {**PPMI_INFO, **_enc.info}
    for _family in ("word2vec", "fasttext"):
        try:
            _kv, _info = train_or_load_final(_family)
        except RuntimeError as error:
            print(f"WARNING: {error}")
            continue
        MODEL_REGISTRY[_family] = encoder_from_kv({"word2vec": "Word2Vec", "fasttext": "FastText"}[_family],
                                                  _family, _kv, "sif", _info)
        MODEL_INFO[_family] = _info
    if not MODEL_REGISTRY:
        raise RuntimeError("No self-trained model is available. Set RUN_TRAINING = True.")

    for _family, _enc in MODEL_REGISTRY.items():
        _frame = final_selection(_family, _enc)
        _best = _frame.sort_values("score", ascending=False).iloc[0]
        MODEL_REGISTRY[_family] = _enc.with_composition(_best["composition"])
        MODEL_INFO[_family]["composition"] = _best["composition"]
        MODEL_INFO[_family]["selection"] = _best.drop(["family", "model_hash"]).to_dict()
        _selection.append(_frame)
    FINAL_SELECTION = pd.concat(_selection, ignore_index=True)
    print("Final models: composition choice and selection score (calibration set + intrinsic only):")
    display(FINAL_SELECTION.drop(columns=["model_hash"]).round(3))
    _scores = {f: MODEL_INFO[f]["selection"]["score"] for f in MODEL_REGISTRY}
    BEST_SELF_TRAINED = max(_scores, key=_scores.get)
    print(f"Best self-trained model (selection score): {BEST_SELF_TRAINED} {_scores}")

    print("\nTraining summary:")
    display(pd.DataFrame([{
        "model": f, "params": json.dumps(MODEL_INFO[f].get("params", {})), "composition": MODEL_INFO[f]["composition"],
        "vocab": len(MODEL_REGISTRY[f].words), "dim": MODEL_REGISTRY[f].dim,
        "train s": MODEL_INFO[f].get("train_seconds"), "words/s": MODEL_INFO[f].get("words_per_second"),
        "size MB": MODEL_INFO[f].get("model_mb"), "corpus share": MODEL_INFO[f].get("train_fraction", 1.0)}
        for f in MODEL_REGISTRY]))
    if "word2vec" in MODEL_INFO and MODEL_INFO["word2vec"].get("epoch_log"):
        print("Word2Vec training loss per epoch (gensim running loss, delta per epoch):")
        display(pd.DataFrame(MODEL_INFO["word2vec"]["epoch_log"]))

    # Pass check: models load from disk (done above) and neighbours of the main targets look sensible.
    _nn = {}
    for _target in ["executive", "algorithm", "catalyst", "painting"]:
        for _family, _enc in MODEL_REGISTRY.items():
            _nn[(_target, _family)] = ", ".join(w for w, _ in _enc.neighbours(_target, 8)) or "(unknown word)"
    print("\nNearest neighbours of the main targets (raw word vectors, 100k most frequent words):")
    display(pd.Series(_nn).unstack())

## 7. Model loading, unit tests and intrinsic evaluation (Phase 4–5)

- **7.1** `load_model(choice)` picks the model from `MODEL_CHOICE` (`"auto"` = the self-trained model with the highest selection score in 6.4). MiniLM is loaded only as a reference. Adapter unit tests: output shape, norm = 1, empty / OOV input does not crash, `evaluate_attack()` works.
- **7.2** Intrinsic evaluation of every model: Spearman on WordSim-353 / SimLex-999 with OOV rate · nearest-neighbour table · **anisotropy** (mean cosine between everyday nouns in `NOISE_VOCAB`; higher = vectors crowd into one direction and meanings are harder to separate) · t-SNE of domain words + cosine silhouette score, a number for how well the domains form separate clusters.

In [ ]:
# 7.1 Model loading (MODEL_CHOICE), MiniLM reference, adapter unit tests
# What this cell does:
#   - load_model(): returns the encoder chosen by MODEL_CHOICE.
#   - Loads MiniLM as a reference baseline only.
#   - Adapter unit tests: shape, unit norm, empty / OOV input, evaluate_attack().
BASE_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"   # reference baseline only
MODEL_CHOICES = ["ppmi_svd", "word2vec", "fasttext", "minilm_reference"]
MODEL_LABELS = {"ppmi_svd": "PPMI-SVD", "word2vec": "Word2Vec", "fasttext": "FastText", "minilm_reference": "MiniLM (ref)"}
_MINILM = {}

def load_minilm():
    """ReferenceEncoder around all-MiniLM-L6-v2, or None (with a warning) when it cannot be loaded."""
    if "encoder" not in _MINILM:
        try:
            from sentence_transformers import SentenceTransformer
            _MINILM["encoder"] = ReferenceEncoder(SentenceTransformer(BASE_MODEL_NAME), BASE_MODEL_NAME)
        except Exception as error:
            print(f"WARNING: MiniLM reference unavailable ({type(error).__name__}: {str(error)[:150]}).")
            _MINILM["encoder"] = None
    return _MINILM["encoder"]

def available_model_choices() -> list:
    """Self-trained models in the registry, plus MiniLM when it is enabled and loadable."""
    out = [c for c in MODEL_CHOICES if c in MODEL_REGISTRY]
    if RUN_MINILM_REFERENCE and load_minilm() is not None:
        out.append("minilm_reference")
    return out

def load_model(choice: str) -> tuple:
    """Return (encoder, resolved_choice, source). 'auto' = best self-trained model (calibration set only)."""
    if choice not in MODEL_CHOICES + ["auto"]:
        raise ValueError(f"MODEL_CHOICE must be one of {MODEL_CHOICES + ['auto']}.")
    if choice == "auto":
        choice = BEST_SELF_TRAINED
    if choice == "minilm_reference":
        encoder = load_minilm()
        if encoder is not None:
            return encoder, choice, BASE_MODEL_NAME
        print(f"WARNING: falling back to the best self-trained model ({BEST_SELF_TRAINED}).")
        choice = BEST_SELF_TRAINED
    if choice not in MODEL_REGISTRY:
        print(f"WARNING: {choice} is not available — using {BEST_SELF_TRAINED}.")
        choice = BEST_SELF_TRAINED
    encoder = MODEL_REGISTRY[choice]
    return encoder, choice, f"self-trained {encoder.name} ({encoder.composition}, {encoder.info.get('model_hash')})"

def adapter_unit_tests(encoder) -> pd.DataFrame:
    """Shape, norm, empty / OOV input, phrase handling and one evaluate_attack() call."""
    rows = []
    def check(test, ok, detail=""):
        rows.append({"model": encoder.family, "test": test, "pass": bool(ok), "detail": str(detail)[:90]})
    out = encoder.encode(["executive", "machine learning", "board of directors"], convert_to_numpy=True,
                         normalize_embeddings=True, show_progress_bar=False)
    check("output shape (3, dim)", out.shape == (3, encoder.dim), out.shape)
    check("dtype float32", out.dtype == np.float32, out.dtype)
    check("L2 norm = 1", np.allclose(np.linalg.norm(out, axis=1), 1.0, atol=1e-4), np.linalg.norm(out, axis=1).round(4))
    single = encoder.encode("painting")
    check("one string -> (dim,)", single.shape == (encoder.dim,), single.shape)
    empty = encoder.encode([""])
    check("empty input -> zero vector, no crash", empty.shape == (1, encoder.dim) and not np.any(empty), "")
    oov = "qzxvbnmwk"
    vec = encoder.encode([oov])[0]
    if encoder.family == "fasttext":
        check("OOV -> subword vector (FastText)", encoder.is_known(oov) and np.isclose(np.linalg.norm(vec), 1, atol=1e-4))
    else:
        check("OOV -> zero vector + is_known False", (not encoder.is_known(oov)) and not np.any(vec))
    mixed = encoder.encode([f"{oov} painting", "painting"])
    if encoder.family == "fasttext":
        check("partly unknown phrase -> unit vector", np.isclose(np.linalg.norm(mixed[0]), 1, atol=1e-4))
    else:
        check("partly unknown phrase uses the known word", float(mixed[0] @ mixed[1]) > 0.99,
              round(float(mixed[0] @ mixed[1]), 3))
    check("phrase handling", True, f"'machine learning' -> {encoder.tokens_for('machine learning')} | "
                                   f"'board of directors' -> {encoder.tokens_for('board of directors')}")
    eng = LexicalCombatEngine(encoder, ENGINE_DATA, EngineConfig())
    guess = next((g for g in ["ceo", "director", "manager", "president", "boss"] if encoder.is_known(g)), "manager")
    res = eng.evaluate_attack(guess, "executive", "Business", set(), 200)
    check(f"evaluate_attack() valid guess ('{guess}')", res["valid"] and 0 <= res["hit_percent"] <= 45,
          f"{res.get('tier')} {res.get('hit_percent', 0):.1f}%")
    res = eng.evaluate_attack(oov, "executive", "Business", set(), 200)
    check("evaluate_attack() unknown word does not crash", isinstance(res, dict),
          res.get("reason", f"valid, {res.get('tier')}"))
    check("dropped noise / pool terms", True, eng.dropped_terms)
    return pd.DataFrame(rows)

with section("7.1 Model loading + unit tests"):
    print("Available models:", available_model_choices(), "| MODEL_CHOICE =", MODEL_CHOICE,
          f"(auto -> {BEST_SELF_TRAINED})")
    UNIT_TESTS = pd.concat([adapter_unit_tests(MODEL_REGISTRY[f]) for f in MODEL_REGISTRY], ignore_index=True)
    display(UNIT_TESTS)
    assert UNIT_TESTS["pass"].all(), "An adapter unit test failed (see the table above)."
    print("All adapter unit tests passed.")

In [ ]:
# 7.2 Intrinsic evaluation: WordSim-353 / SimLex-999, neighbours, anisotropy, t-SNE + silhouette
# What this cell does:
#   - Spearman on WordSim-353 / SimLex-999 with OOV rates for every model.
#   - Nearest neighbours, anisotropy and silhouette score per model.
#   - t-SNE plot of domain words for the best self-trained model.
RUN_TSNE = True

def anisotropy(encoder) -> float:
    """Mean cosine between everyday nouns (NOISE_VOCAB) the model knows."""
    words = [w for w in NOISE_VOCAB if encoder.is_known(w)]
    vectors = encoder.encode(words)
    sims = vectors @ vectors.T
    return float(sims[np.triu_indices(len(words), k=1)].mean())

def domain_word_sample() -> tuple:
    """Single-domain terms (term pools + fresh calibration terms) for the cluster plot."""
    words, labels = [], []
    for d in DOMAINS:
        for t in domain_term_pool(d, DOMAIN_EXAMPLES, POOL_VOCAB, GENERIC_TERMS_V2) + FRESH_IN_DOMAIN_CLEAN[d]:
            if t not in words:
                words.append(t)
                labels.append(d)
    return words, labels

with section("7.2 Intrinsic evaluation"):
    INTRINSIC_ENCODERS = {c: load_model(c)[0] for c in available_model_choices()}
    _words, _labels = domain_word_sample()
    _rows = []
    for _choice, _enc in INTRINSIC_ENCODERS.items():
        _known = [i for i, w in enumerate(_words) if _enc.is_known(w)]
        _vectors = _enc.encode([_words[i] for i in _known])
        _rows.append({"model": MODEL_LABELS[_choice], **intrinsic_scores(_enc),
                      "anisotropy (noise-pair mean cos)": anisotropy(_enc),
                      "domain silhouette (cosine)": silhouette_score(_vectors, [_labels[i] for i in _known], metric="cosine"),
                      "vocab": len(_enc.words) if hasattr(_enc, "words") else "subword", "dim": _enc.dim})
    INTRINSIC_TABLE = pd.DataFrame(_rows).set_index("model")
    print("Intrinsic metrics (Spearman on known pairs; OOV = pairs with an unknown word):")
    display(INTRINSIC_TABLE.round(3))

    _probe = ["executive", "algorithm", "catalyst", "painting", "revenue", "software", "molecule", "melody",
              "brand", "comet"]
    _nn = {(w, MODEL_LABELS[c]): ", ".join(t for t, _ in e.neighbours(w, 6)) or "—"
           for w in _probe for c, e in INTRINSIC_ENCODERS.items() if c != "minilm_reference"}
    print("Nearest neighbours (self-trained models; MiniLM has no word vocabulary to search):")
    display(pd.Series(_nn).unstack())

    if RUN_TSNE:
        import matplotlib.pyplot as plt
        from sklearn.manifold import TSNE
        _enc = MODEL_REGISTRY[BEST_SELF_TRAINED]
        _known = [i for i, w in enumerate(_words) if _enc.is_known(w)]
        _xy = TSNE(n_components=2, perplexity=min(30, max(5, len(_known) // 6)), metric="cosine", init="pca",
                   random_state=SEED).fit_transform(_enc.encode([_words[i] for i in _known]))
        _colors = {"Business": "#3b82f6", "Technology": "#10b981", "Science": "#a855f7", "Arts": "#f43f5e"}
        fig, ax = plt.subplots(figsize=(9, 7))
        for d in DOMAINS:
            idx = [j for j, i in enumerate(_known) if _labels[i] == d]
            ax.scatter(_xy[idx, 0], _xy[idx, 1], s=18, c=_colors[d], label=d, alpha=0.8)
            for j in idx[::4]:
                ax.annotate(_words[_known[j]], _xy[j], fontsize=7, alpha=0.7)
        ax.set_title(f"t-SNE of domain terms — {MODEL_LABELS[BEST_SELF_TRAINED]} (self-trained)")
        ax.legend()
        ax.set_xticks([])
        ax.set_yticks([])
        plt.show()

## 8. Calibration for the playing model (Phase 6)

Cosine values of word embeddings (especially after SIF) are distributed very differently from MiniLM's, so the old gate and damage thresholds do not work. **Every calibration step of the original Section 5 is re-run** for this model:
- **8.0** Build the engine for the active model with default (pre-calibration) settings.
- **8.1** Gate diagnosis on the old holdout (diagnostic set only).
- **8.2** Gate grid (ANCHOR_TOPK ∈ {1,2,3}, plateau selection, 5-fold grouped CV) → damage grid (per-target loss + pacing, plateau, LOTO-CV). If a chosen value sits on an **expandable grid edge**, the grid is expanded and re-run (up to 6 rounds). Saved per model as `tuned_params_<model>.json`.
- **8.3** Gate review with the tuned thresholds.

**Pass criterion:** pacing met on every stage (expert 3–4 turns, typical 5–8 turns) and out-of-domain leak on the calibration set ≤ 5%.

In [ ]:
# 8.0 Active model -> engine (pre-calibration defaults)
# What this cell does:
#   - Loads the active model and builds the engine with default settings.
#   - Prints domain backgrounds, span floor and stage ceilings.
with section("8.0 Active model"):
    model, ACTIVE_MODEL, MODEL_SOURCE = load_model(MODEL_CHOICE)
    MODEL_IDENTITY = model.identity()
    engine = LexicalCombatEngine(model, ENGINE_DATA, EngineConfig())
    print(f"Active model: {ACTIVE_MODEL} — {MODEL_SOURCE} | available: {available_model_choices()}")
    print("Domain background (mean debias term on everyday nouns): "
          + " | ".join(f"{d}: {v:.3f}" for d, v in engine.background.items()))
    print(f"Span floor ({engine.config.min_span_mode}): {engine.span_floor:.3f} | "
          f"anisotropy (mean cos between everyday nouns): {engine.noise_pair_mean:.3f}")
    print("Terms the model cannot represent (dropped from noise / pools):",
          {k: v for k, v in engine.dropped_terms.items() if v} or "none")
    for stage in GAME_STAGES:
        cal = engine.target_calibration(stage["target"], stage["domain"])
        noise, ceiling = engine.bounds(cal)
        top = sorted(zip(cal["pool_sims"], cal["pool_terms"]), reverse=True)[:engine.config.ceiling_topk]
        print(f"  {stage['target']:<10} noise floor {noise:.3f} | ceiling {ceiling:.3f} | "
              f"ceiling terms: {', '.join(t for _, t in top)}")

In [ ]:
# 8.1 Gate diagnosis on the old holdout with the pre-calibration defaults (diagnostic set, read-only)
# What this cell does:
#   - Diagnoses the gate on the old holdout with the default thresholds (read-only).
#   - Shows the circularity check: phrases written in anchors score higher.
#   - Shows ceiling terms per holdout target (term-pool coverage).
with section("8.1 Gate diagnosis (old holdout)"):
    c = engine.config   # pre-calibration defaults
    diag_params = {"primary": c.gate_primary_min, "secondary": c.gate_secondary_min,
                   "gap": c.gate_secondary_max_gap, "weak_gap": c.gate_weak_max_gap, "k": c.anchor_topk}
    holdout_draft_df = pd.DataFrame(OLD_HOLDOUT_DIAG_CASES, columns=[
        "target", "guess", "required_domain", "human_label", "human_expected_tier"])
    holdout_diag = diagnose_gate_rows(engine, holdout_draft_df, diag_params)
    print(f"Gate used for the diagnosis: {diag_params}")
    display(holdout_diag[["target", "guess", "domain", "human", "gate", "req_k1", "req_k2", "req_k3",
                          "raw", "background", "lead_k2", "top_domain", "nearest_anchor_cos",
                          "nearest_anchor", "rival_anchor", "cause"]].round(3))

    in_rows = holdout_diag[holdout_diag["human"] == "in_domain"]
    print(f"In-domain holdout rows reaching PRIMARY: {(in_rows['gate'] == 'primary').sum()}/{len(in_rows)}")
    print("Cause counts for in-domain rows that missed PRIMARY:")
    display(in_rows.loc[in_rows["gate"] != "primary", "cause"].value_counts().rename("rows").to_frame())
    focus = holdout_diag[holdout_diag["guess"].isin(["harmony", "protein", "cipher"])]
    print("Focus rows (harmony / protein / cipher):")
    display(focus[["guess", "domain", "gate", "req_k1", "req_k2", "req_k3", "lead_k2",
                   "nearest_anchor_cos", "cause"]].round(3))

    # Circularity evidence: in-domain phrases WRITTEN in anchors score far higher
    # than fresh in-domain phrases, so thresholds tuned on them are too strict.
    _all_vectors = dict(zip(sorted(set(gate_set_all_df["phrase"])),
                            engine.encode(sorted(set(gate_set_all_df["phrase"])))))
    _circ = gate_set_all_df[gate_set_all_df["label"] == "in_domain"].copy()
    _circ["required"] = [float(dict(zip(DOMAINS, engine.domain_scores(_all_vectors[p], diag_params["k"])[0][0]))[d])
                         for p, d in zip(_circ["phrase"], _circ["domain"])]
    print("Debiased required-domain score of in-domain gate phrases (circularity check):")
    display(_circ.groupby("anchor_echo")["required"].describe()[["count", "mean", "25%", "50%", "75%"]]
            .rename(index={True: "written in anchors", False: "fresh / not in anchors"}).round(3))
    _req_col = "req_k" + str(diag_params["k"])
    print(f"Holdout in-domain median required score: {in_rows[_req_col].median():.3f}")

    # Ceiling coverage of the term pool for every holdout target.
    print("\nCeiling terms per holdout target (pool coverage):")
    _cov = []
    for target, domain in sorted({(t, d) for t, _, d, _, _ in OLD_HOLDOUT_DIAG_CASES}):
        cal = engine.target_calibration(target, domain)
        noise, ceiling = engine.raw_bounds(cal)
        top = sorted(zip(cal["pool_sims"], cal["pool_terms"]), reverse=True)[:5]
        _cov.append({"target": target, "noise": noise, "ceiling (raw)": ceiling, "span": ceiling - noise,
                     "span floor": engine.span_floor, "top-5 pool terms": ", ".join(f"{t} ({s:.2f})" for s, t in top)})
    display(pd.DataFrame(_cov).round(3))

In [ ]:
# 8.2 Run (or load) the gate + damage calibration for the active model and apply it
# What this cell does:
#   - Runs (or loads) gate + damage calibration for the active model and applies it.
#   - Prints the damage curve and the Phase 6 pass / fail result.
with section("8.2 Calibration") as _s:
    GATE_CAL, DAMAGE_CAL, CALIBRATION_SOURCE = calibrate_engine(engine, ACTIVE_MODEL, MODEL_IDENTITY, verbose=True)
    print(f"\nDamage curve after calibration (span floor {engine.span_floor:.3f}):")
    display(damage_curve_table(engine))
    _leak_tuned, _leak_cv = GATE_CAL["tuned"]["out_leak"], GATE_CAL["cv"]["out_leak"]
    PHASE6_PASS = bool(DAMAGE_CAL["all_pacing_met"] and _leak_tuned <= 0.05)
    print(f"\nPhase 6 criterion — pacing met on every stage: {DAMAGE_CAL['all_pacing_met']} "
          f"(LOTO-CV: {DAMAGE_CAL['cv_all_pacing_met']}) | out-of-domain leak on the calibration set: "
          f"{_leak_tuned:.1%} (CV {_leak_cv:.1%}, limit 5%) -> {'PASS' if PHASE6_PASS else 'FAIL'}")
    if not PHASE6_PASS:
        print("  Reported as is (no test data was changed). Section 10 explains likely causes.")

In [ ]:
# 8.3 Gate review with the tuned thresholds (confusion matrix, anchor-echo sanity set, errors)
# What this cell does:
#   - Confusion matrix of the tuned gate on the calibration set.
#   - Anchor-echo sanity set and cross-domain ambiguous phrases.
#   - Lists the worst gate errors (leaks first).
with section("8.3 Gate review"):
    gate_scores = GATE_CAL["scored"]
    print("Confusion matrix (rows = human, columns = predicted):")
    display(pd.crosstab(gate_scores["label"], gate_scores["predicted"]).reindex(
        index=LABELS, columns=LABELS, fill_value=0))

    _echo_inputs = gate_inputs(engine, anchor_echo_df, [engine.config.anchor_topk])[engine.config.anchor_topk]
    _echo_kinds = engine.gate_kinds(_echo_inputs["required"], _echo_inputs["lead"])
    print("Anchor-echo sanity set:", {**pass_block_metrics(anchor_echo_df["label"], _echo_kinds),
                                      "macro_f1": classification_metrics(anchor_echo_df["label"],
                                                                         kinds_to_labels(_echo_kinds))["macro_f1"]})
    print("CROSS_DOMAIN_AMBIGUOUS with the chosen gate:")
    _amb = anchor_echo_df.assign(gate=_echo_kinds)
    display(_amb[_amb["source"] == "CROSS_DOMAIN_AMBIGUOUS"][["phrase", "domain", "label", "gate"]])

    print("Rows to review (leaks first, then wrongly blocked in-domain phrases):")
    errors = gate_scores[(gate_scores["predicted"] != gate_scores["label"])].copy()
    errors["severity"] = np.select(
        [(errors["label"] == "out_of_domain"), (errors["predicted"] == "out_of_domain")],
        ["1 leak", "2 wrongly blocked"], "3 tier mismatch")
    display(errors.sort_values(["severity", "required"], ascending=[True, False])[
        ["severity", "phrase", "domain", "label", "gate", "top_domain", "required", "lead", "source"]
    ].round(3).head(30))

## 9. Report and model comparison (calibration side)

- **9.1** Results of the playing model through the real pipeline (`evaluate_attack`): banded cases, gate leak (passes the gate but 0 damage) vs damaging leak, pacing per stage, tuned vs CV, and the parameters in use.
- **9.2** Every model (PPMI-SVD, Word2Vec, FastText, MiniLM reference) gets **its own calibration**, then they are compared with intrinsic metrics, AUC and game metrics on the calibration set / CV. The holdout is still untouched.

In [ ]:
# 9.1 Report for the active model: banded cases, leak split, pacing, calibration summary, parameters
# What this cell does:
#   - Runs the calibration cases through evaluate_attack() for the active model.
#   - Banded cases, gate leak vs damaging leak, pacing per stage.
#   - Tuned vs CV summary and the parameters in use.
def snapshot_params(engine: LexicalCombatEngine) -> dict:
    """Tuned parameters under their report names."""
    c = engine.config
    return {"GATE_PRIMARY_MIN": c.gate_primary_min, "GATE_SECONDARY_MIN": c.gate_secondary_min,
            "GATE_SECONDARY_MAX_GAP": c.gate_secondary_max_gap, "GATE_WEAK_MAX_GAP": c.gate_weak_max_gap,
            "ANCHOR_TOPK": c.anchor_topk, "NOISE_PERCENTILE": c.noise_percentile,
            "CEILING_TOPK": c.ceiling_topk, "RELEVANCE_MIN": c.relevance_min,
            "DAMAGE_GAMMA": c.damage_gamma, "MIN_SPAN_MODE": c.min_span_mode}

def gate_kinds_for(engine: LexicalCombatEngine, frame: pd.DataFrame) -> np.ndarray:
    """Gate kinds for a (phrase, domain) frame with the engine's thresholds."""
    k = engine.config.anchor_topk
    inputs = gate_inputs(engine, frame, [k])[k]
    return engine.gate_kinds(inputs["required"], inputs["lead"])

def intrinsic_metrics(engine: LexicalCombatEngine) -> dict:
    """Model-level numbers that do not depend on tuned thresholds."""
    pos, neg, blocked, rel_blocked = [], [], [], []
    for row in damage_cases_df.itertuples(index=False):
        cal = engine.target_calibration(row.target, row.required_domain)
        sim = float(engine.encode([clean_text(row.guess)])[0] @ cal["vector"])
        if row.human_expected_tier in ("strong", "medium"):
            pos.append(sim)
        if row.human_expected_tier == "blocked":
            noise, ceiling = engine.bounds(cal)
            blocked.append(sim)
            rel_blocked.append(float(normalized_relevance(sim, noise, ceiling)))
    for target, domain in sorted(set(zip(damage_cases_df["target"], damage_cases_df["required_domain"]))):
        neg.extend(engine.target_calibration(target, domain)["noise_sims"].tolist())
    gate_in = gate_set_df[gate_set_df["label"] != "ambiguous"]
    k = engine.config.anchor_topk
    g = gate_inputs(engine, gate_in, [k])[k]
    return {"AUC related vs noise": roc_auc_score(np.r_[np.ones(len(pos)), np.zeros(len(neg))], np.r_[pos, neg]),
            "Anisotropy (noise-pair mean cos)": engine.noise_pair_mean,
            "Mean cos strong/medium": float(np.mean(pos)),
            "Out-of-domain mean cos (blocked pairs)": float(np.mean(blocked)),
            "Mean r blocked": float(np.mean(rel_blocked)),
            "Gate AUC in vs out (debiased score)": roc_auc_score((gate_in["label"] == "in_domain").astype(int),
                                                                 g["required"])}

with section("9.1 Report (active model)"):
    # 9.1a Banded cases through the real pipeline
    v2_eval = evaluate_test_set(engine, damage_cases_df)
    v2_eval["Expected tier"] = damage_cases_df["human_expected_tier"].values
    v2_eval["Target band"] = v2_eval["Expected tier"].map(lambda t: f"{TIER_BANDS_V2[t][0]}–{TIER_BANDS_V2[t][1]}%")
    v2_eval["In band"] = [band_distance(p, TIER_BANDS_V2[t]) == 0 if p == p else False
                          for p, t in zip(v2_eval.get("Hit %", pd.Series(np.nan, index=v2_eval.index)),
                                          v2_eval["Expected tier"])]
    print("Banded cases through evaluate_attack():")
    display(v2_eval[["Target", "Guess", "Human Label", "Domain Match", "Label Aligned", "Cosine Similarity",
                     "Relevance r", "Hit %", "Predicted Tier", "Target band", "In band"]])
    print("In-band rate per target:")
    display(v2_eval.groupby("Target")["In band"].agg(["mean", "size"]).round(3))
    print(f"Gate label alignment: {v2_eval['Label Aligned'].mean():.1%} | "
          f"In damage band: {v2_eval['In band'].mean():.1%}")

    # 9.1b Leak split: gate leak (passes, 0 damage) vs damaging leak (passes and damages)
    leak_rows = []
    for name, frame in [("gate tuning set", gate_set_df), ("anchor-echo set", anchor_echo_df)]:
        kinds = gate_kinds_for(engine, frame)
        hits = np.zeros(len(frame))
        need = ((frame["label"] == "out_of_domain") & (kinds != "out_of_domain")).to_numpy()
        if need.any():
            hits[need] = damage_against_reference(engine, frame.loc[need, "phrase"], frame.loc[need, "domain"])
        leak_rows.append({"set": name, **leak_breakdown(frame["label"], kinds, hits)})
    leak_rows.append({"set": "banded cases", **leak_breakdown(v2_eval["Human Label"],
                                                              v2_eval["Domain Match"].fillna("out_of_domain"),
                                                              v2_eval["Hit %"].fillna(0))})
    print("\nLeak split:")
    display(pd.DataFrame(leak_rows).round(3))

    # 9.1c Whole battles: pacing per stage
    pacing_rows = []
    for stage in GAME_STAGES:
        t = stage["target"]
        plays = {style: guesses for style, (guesses, _) in REFERENCE_PLAYS[t].items()}
        plays["playlog"] = PLAYER_LOG_GUESSES_V2[t]
        for style, guesses in plays.items():
            battle = engine.simulate_battle(t, stage["domain"], stage["hp"], guesses)
            if style == "typical":
                print(f"\n{stage['enemy']} — typical play:")
                display(battle)
            target_range = REFERENCE_PLAYS[t].get(style, (None, None))[1]
            summary = engine.summarize_battle(battle, stage["hp"])
            met = (target_range is not None and summary["Turns to defeat"] is not None
                   and target_range[0] <= summary["Turns to defeat"] <= target_range[1])
            pacing_rows.append({"Stage": stage["stage"], "Enemy": stage["enemy"], "Play": style,
                                "Goal turns": f"{target_range[0]}–{target_range[1]}" if target_range else "—",
                                "Goal met": met if target_range else "—", **summary})
    pacing_df = pd.DataFrame(pacing_rows)
    print("\nPacing summary (expert 3–4, typical 5–8):")
    display(pacing_df)

    # 9.1d Calibration summary (tuned vs CV, edge checks)
    calibration_summary = pd.DataFrame([
        {"part": "gate", "metric": "macro F1", "tuned": GATE_CAL["tuned"]["macro_f1"], "cv": GATE_CAL["cv"]["macro_f1"]},
        {"part": "gate", "metric": "in-domain primary", "tuned": GATE_CAL["tuned"]["in_domain_primary"],
         "cv": GATE_CAL["cv"]["in_domain_primary"]},
        {"part": "gate", "metric": "ambiguous pass", "tuned": GATE_CAL["tuned"]["ambiguous_pass"],
         "cv": GATE_CAL["cv"]["ambiguous_pass"]},
        {"part": "gate", "metric": "out-of-domain leak", "tuned": GATE_CAL["tuned"]["out_leak"],
         "cv": GATE_CAL["cv"]["out_leak"]},
        {"part": "damage", "metric": "in-band", "tuned": DAMAGE_CAL["tuned_in_band"], "cv": DAMAGE_CAL["cv_in_band"]},
        {"part": "damage", "metric": "all pacing goals met", "tuned": DAMAGE_CAL["all_pacing_met"],
         "cv": DAMAGE_CAL["cv_all_pacing_met"]},
    ])
    print(f"\nCalibration ({CALIBRATION_SOURCE}): tuned set vs cross-validation")
    display(calibration_summary)
    PARAMS_ON_EDGE = {**{f"gate.{k}": v for k, v in GATE_CAL.get("soft_edges", GATE_CAL["edges"]).items()},
                      **{f"damage.{k}": v for k, v in DAMAGE_CAL.get("soft_edges", DAMAGE_CAL["edges"]).items()}}
    HARD_EDGES = [f"gate.{k}" for k, v in GATE_CAL.get("hard_edges", {}).items() if v] + \
                 [f"damage.{k}" for k, v in DAMAGE_CAL.get("hard_edges", {}).items() if v]
    print("Parameters on an expandable grid edge (after expansion):", [k for k, v in PARAMS_ON_EDGE.items() if v] or "none",
          "| on a natural limit:", HARD_EDGES or "none")
    print("\nActive parameters:")
    c = engine.config
    display(pd.Series({**snapshot_params(engine), "SPAN_FLOOR": round(engine.span_floor, 3),
                       "MAX_HIT_PERCENT": c.max_hit_percent, "GENERIC_PENALTY": c.generic_penalty,
                       "PRIMARY_GRAZE_FLOOR": c.primary_graze_floor, "MODEL_CHOICE": ACTIVE_MODEL,
                       "Echo threshold (raw cos) / penalty": f"{engine.echo_threshold():.2f} / {c.echo_penalty}"})
            .to_frame("value"))

In [ ]:
# 9.2 Model comparison on the calibration side: every model gets its OWN calibration (gate + damage grids)
# What this cell does:
#   - Calibrates every model with its own gate + damage grids.
#   - Compares models on intrinsic metrics, AUC and game metrics (calibration side only).
ENGINES = {ACTIVE_MODEL: engine}
ENGINE_CALS = {ACTIVE_MODEL: (GATE_CAL, DAMAGE_CAL, CALIBRATION_SOURCE)}

def engine_for(choice: str) -> LexicalCombatEngine:
    """Calibrated engine per model (calibration loaded from / saved to tuned_params_<model>.json)."""
    if choice not in ENGINES:
        encoder, _, _ = load_model(choice)
        eng = LexicalCombatEngine(encoder, ENGINE_DATA, EngineConfig())
        ENGINE_CALS[choice] = calibrate_engine(eng, choice, encoder.identity(), verbose=False)
        ENGINES[choice] = eng
    return ENGINES[choice]

def pacing_for(eng: LexicalCombatEngine) -> pd.DataFrame:
    """Expert / typical / play-log battles per stage (same as 9.1c, without printing)."""
    rows = []
    for stage in GAME_STAGES:
        t = stage["target"]
        plays = {style: guesses for style, (guesses, _) in REFERENCE_PLAYS[t].items()}
        plays["playlog"] = PLAYER_LOG_GUESSES_V2[t]
        for style, guesses in plays.items():
            summary = eng.summarize_battle(eng.simulate_battle(t, stage["domain"], stage["hp"], guesses), stage["hp"])
            goal = REFERENCE_PLAYS[t].get(style, (None, None))[1]
            met = (goal is not None and summary["Turns to defeat"] is not None
                   and goal[0] <= summary["Turns to defeat"] <= goal[1])
            rows.append({"Stage": stage["stage"], "Play": style, "Goal": f"{goal[0]}–{goal[1]}" if goal else "—",
                         "Turns to defeat": summary["Turns to defeat"], "Goal met": met if goal else "—"})
    return pd.DataFrame(rows)

def calibration_side_row(choice: str) -> dict:
    eng = engine_for(choice)
    gate, damage, source = ENGINE_CALS[choice]
    kinds = gate_kinds_for(eng, gate_set_df)
    hits = np.zeros(len(gate_set_df))
    need = ((gate_set_df["label"] == "out_of_domain") & (kinds != "out_of_domain")).to_numpy()
    if need.any():
        hits[need] = damage_against_reference(eng, gate_set_df.loc[need, "phrase"], gate_set_df.loc[need, "domain"])
    leaks = leak_breakdown(gate_set_df["label"], kinds, hits)
    pace = pacing_for(eng)
    goals = pace[pace["Goal met"] != "—"]
    return {"Model": MODEL_LABELS[choice], "role": "reference (pretrained)" if choice == "minilm_reference" else "self-trained",
            **{k: v for k, v in INTRINSIC_TABLE.loc[MODEL_LABELS[choice]].items()
               if k in ("WordSim-353 Spearman", "SimLex-999 Spearman", "WordSim-353 OOV pairs", "SimLex-999 OOV pairs",
                        "domain silhouette (cosine)")},
            **intrinsic_metrics(eng),
            "gate macro F1 (tuned)": gate["tuned"]["macro_f1"], "gate macro F1 (CV)": gate["cv"]["macro_f1"],
            "in-domain -> primary (CV)": gate["cv"]["in_domain_primary"], "ambiguous pass (CV)": gate["cv"]["ambiguous_pass"],
            "out-of-domain leak (CV)": gate["cv"]["out_leak"], "gate leak % (tuning set)": leaks["gate leak %"],
            "damaging leak % (tuning set)": leaks["damaging leak %"],
            "damage in-band (tuned)": damage["tuned_in_band"], "damage in-band (LOTO-CV)": damage["cv_in_band"],
            "pacing met": f"{int((goals['Goal met'] == True).sum())}/{len(goals)}",
            "soft grid edges": ", ".join([f"gate.{k}" for k, v in gate.get("soft_edges", {}).items() if v]
                                         + [f"damage.{k}" for k, v in damage.get("soft_edges", {}).items() if v]) or "none",
            "calibration": source}

with section("9.2 Model comparison (calibration side)"):
    COMPARISON_CAL = pd.DataFrame([calibration_side_row(c) for c in available_model_choices()]).set_index("Model")
    print("All models, each with its own calibration (calibration set, CV and intrinsic — no holdout):")
    display(COMPARISON_CAL.T)

## 10. Holdout and comparison (Phase 7)

Uses the same 28-case holdout (all-new targets: brand / email / comet / cartoon) with labels from `HOLDOUT_ANNOTATIONS` (+ `annotator_*.xlsx/.csv` files if present). It is **evaluated once**, after the model, composition and every parameter were chosen on the calibration set (no step goes back and tunes on holdout results). With ≥ 2 annotators and Drive mounted, a run log with a fingerprint is saved; a re-run with different settings prints a warning that it is no longer a clean final result.

**Counting rule:** a guess the model does not know (OOV) counts as a **gate error** (strict, because the player does not get the expected result), but its damage is 0, so it can still be in band when the label is blocked. The number of OOV words is reported separately.

- **10.1** Results of the playing model + the original success criteria.
- **10.2** Comparison table PPMI-SVD vs Word2Vec vs FastText vs MiniLM (reference) on the holdout (each model with its own calibration from Section 9) + **error analysis**: cases our model gets wrong but MiniLM gets right, and the reverse, with likely causes computed from real data (OOV, corpus frequency, phrases averaged from words, direction of the error).

In [ ]:
# 10.1 Holdout: labels (inline + annotator files), majority vote / kappa, final evaluation (from the original 7)
# What this cell does:
#   - Collects holdout labels (inline + optional annotator files), majority vote, kappa.
#   - Evaluates the active model on the holdout once.
#   - Checks the original success criteria and logs a run fingerprint.
HOLDOUT_ITEMS = [(t, g, d) for t, g, d, _, _ in HOLDOUT_CASES_DRAFT]   # labels hidden from annotators
def holdout_run_log(choice: str) -> Path:
    return PATHS.artifacts / f"holdout_final_run_{choice}.json"

def export_label_sheets(names=("annotator_1", "annotator_2", "annotator_3")) -> None:
    """Write blank CSV sheets (no draft labels) to PATHS.labels for independent labelling."""
    sheet = pd.DataFrame(HOLDOUT_ITEMS, columns=["target", "guess", "domain"])
    sheet.insert(0, "item", range(len(sheet)))
    sheet["gate_label"], sheet["tier"] = "", ""
    for name in names:
        sheet.to_csv(PATHS.labels / f"{name}.csv", index=False)
    print(f"Wrote {len(names)} sheets to {PATHS.labels}. Gate labels {GATE_CHOICES}, tiers {TIER_CHOICES}.")

def read_label_sheet(path: Path) -> pd.DataFrame:
    """One annotator file (.xlsx sheet 'Labels' or .csv)."""
    return pd.read_excel(path, sheet_name="Labels") if path.suffix == ".xlsx" else pd.read_csv(path)

def collect_annotations() -> dict:
    """Inline HOLDOUT_ANNOTATIONS plus annotator_* files (files override the same name). Returns a new dict."""
    annotations = {name: list(values) for name, values in HOLDOUT_ANNOTATIONS.items()}
    paths = sorted(p for folder in LABEL_SEARCH_DIRS for ext in ("xlsx", "csv")
                   for p in folder.glob(f"annotator_*.{ext}"))
    for path in paths:
        sheet = read_label_sheet(path)
        sheet = sheet[pd.to_numeric(sheet["item"], errors="coerce").notna()].sort_values("item")
        if len(sheet) != len(HOLDOUT_ITEMS):
            print(f"WARNING: {path.name} has {len(sheet)} rows, expected {len(HOLDOUT_ITEMS)} — file ignored.")
            continue
        values = [(str(g).strip(), str(t).strip())
                  if isinstance(g, str) and g.strip() and isinstance(t, str) and t.strip() else None
                  for g, t in zip(sheet["gate_label"], sheet["tier"])]
        if any(v is not None for v in values):
            annotations[path.stem] = values
            print(f"Loaded {path.stem} from {path}: {sum(v is not None for v in values)}/{len(values)} filled")
    return annotations

def fleiss_kappa(labels_by_rater, categories):
    """Fleiss' kappa for items labelled by the same number of raters."""
    counts = np.array([[sum(1 for r in item if r == c) for c in categories] for item in labels_by_rater])
    n = counts.sum(axis=1)[0]
    if n < 2:
        return np.nan
    p_j = counts.sum(axis=0) / counts.sum()
    p_i = ((counts ** 2).sum(axis=1) - n) / (n * (n - 1))
    p_bar, p_e = p_i.mean(), (p_j ** 2).sum()
    return float((p_bar - p_e) / (1 - p_e)) if p_e < 1 else 1.0

def majority(values):
    """Value with >= 2 votes, else None (needs discussion)."""
    counts = pd.Series(values).value_counts()
    return counts.index[0] if counts.iloc[0] >= 2 else None

def consolidate_labels(annotations: dict) -> tuple:
    """(cases, status, agreement): majority vote, single annotator, or the draft."""
    raters = {}
    for n, v in annotations.items():
        if not all(x is not None for x in v):
            continue
        bad = [(g, t) for g, t in v if g not in GATE_CHOICES or t not in TIER_CHOICES]
        if bad:
            print(f"WARNING: {n} has invalid labels (e.g. {bad[0]}) — annotator ignored.")
            continue
        raters[n] = v
    if not raters:
        print("WARNING: no complete annotation set — using the DRAFT labels (status 'draft').")
        return list(HOLDOUT_CASES_DRAFT), "draft", None
    if len(raters) == 1:
        name, labels = next(iter(raters.items()))
        draft = [(l, t) for _, _, _, l, t in HOLDOUT_CASES_DRAFT]
        tier_rank = {t: i for i, t in enumerate(TIER_CHOICES)}
        agreement = {"annotators": 1,
                     "note": "single human annotator; kappa below is human vs AI draft, not inter-rater",
                     "Cohen kappa gate (human vs AI draft)": cohen_kappa_score([x[0] for x in labels], [x[0] for x in draft]),
                     "Weighted kappa tier (human vs AI draft)": cohen_kappa_score(
                         [tier_rank[x[1]] for x in labels], [tier_rank[x[1]] for x in draft], weights="quadratic")}
        print(f"Single annotator ({name}): these labels are used as final.")
        cases = [(t, g, d, gate, tier) for (t, g, d), (gate, tier) in zip(HOLDOUT_ITEMS, labels)]
        return cases, "single_annotator", agreement
    names = list(raters)
    gates = [[raters[n][i][0] for n in names] for i in range(len(HOLDOUT_ITEMS))]
    tiers = [[raters[n][i][1] for n in names] for i in range(len(HOLDOUT_ITEMS))]
    agreement = {"annotators": len(names),
                 "Fleiss kappa (gate)": fleiss_kappa(gates, GATE_CHOICES),
                 "Fleiss kappa (tier)": fleiss_kappa(tiers, TIER_CHOICES)}
    tier_rank = {t: i for i, t in enumerate(TIER_CHOICES)}
    for a, b in itertools.combinations(names, 2):
        agreement[f"Cohen kappa gate {a}–{b}"] = cohen_kappa_score([x[0] for x in raters[a]], [x[0] for x in raters[b]])
        agreement[f"Weighted kappa tier {a}–{b}"] = cohen_kappa_score(
            [tier_rank[x[1]] for x in raters[a]], [tier_rank[x[1]] for x in raters[b]], weights="quadratic")
    cases, disputed = [], []
    for (t, g, d), gate_votes, tier_votes in zip(HOLDOUT_ITEMS, gates, tiers):
        gate, tier = majority(gate_votes), majority(tier_votes)
        if gate is None or tier is None:
            disputed.append({"target": t, "guess": g, "gate votes": gate_votes, "tier votes": tier_votes})
            continue
        cases.append((t, g, d, gate, tier))
    if disputed:
        print("No majority (discuss, then add to HOLDOUT_ANNOTATIONS):")
        display(pd.DataFrame(disputed))
    return cases, "majority_vote", agreement

def evaluate_holdout(engine: LexicalCombatEngine, cases: list, status: str, verbose: bool = True) -> tuple:
    """Run holdout cases through the engine; returns (result table, summary dict)."""
    holdout_df = pd.DataFrame(cases, columns=[
        "target", "guess", "required_domain", "human_label", "human_expected_tier"])
    seen = set(zip(damage_cases_df["target"].map(clean_text), damage_cases_df["guess"].map(clean_text)))
    overlap = [p for p in zip(holdout_df["target"].map(clean_text), holdout_df["guess"].map(clean_text)) if p in seen]
    if overlap and verbose:
        print("WARNING: holdout pairs also used for calibration:", overlap)

    result = evaluate_test_set(engine, holdout_df)
    result["Expected tier"] = holdout_df["human_expected_tier"].values
    hits = result["Hit %"].fillna(0) if "Hit %" in result else pd.Series(0.0, index=result.index)
    result["In band"] = [band_distance(p, TIER_BANDS_V2[t]) == 0 for p, t in zip(hits, result["Expected tier"])]
    kinds = result["Domain Match"].fillna("out_of_domain").to_numpy()
    metrics = classification_metrics(result["Human Label"], result["Predicted Label"])
    gate_feel = pass_block_metrics(result["Human Label"].to_numpy(), kinds)
    leaks = leak_breakdown(result["Human Label"], kinds, hits)
    summary = {"cases": len(result), "label status": status,
               "gate accuracy": metrics["accuracy"], "gate macro F1": metrics["macro_f1"],
               "in-domain -> primary": gate_feel["in_domain_primary"],
               "ambiguous pass": gate_feel["ambiguous_pass"],
               "gate leak (0 dmg)": leaks["gate leak (0 dmg)"], "damaging leak": leaks["damaging leak"],
               "damage in band": float(result["In band"].mean()),
               "unknown-word guesses": int(result["Predicted Label"].eq("invalid").sum())}
    if not verbose:
        return result, summary
    print("Holdout summary:")
    display(pd.Series(summary).to_frame("value"))
    display(pd.crosstab(result["Human Label"], result["Predicted Label"]))
    columns = ["Target", "Guess", "Human Label", "Domain Match", "Predicted Label", "Cosine Similarity",
               "Relevance r", "Hit %", "Predicted Tier", "Expected tier", "In band"]
    display(result[[c for c in columns if c in result]])
    print("Ceiling terms per holdout target:")
    for target, domain in sorted(set(zip(holdout_df["target"], holdout_df["required_domain"]))):
        cal = engine.target_calibration(target, domain)
        noise, ceiling = engine.bounds(cal)
        top = sorted(zip(cal["pool_sims"], cal["pool_terms"]), reverse=True)[:engine.config.ceiling_topk]
        print(f"  {target:<11} noise {noise:.3f} | ceiling {ceiling:.3f} | span {ceiling - noise:.3f} | "
              f"{', '.join(t for _, t in top)}")
    return result, summary

def success_criteria(summary: dict, pacing_df: pd.DataFrame, damage_cal: dict, params_on_edge: dict) -> pd.DataFrame:
    """Project success criteria (holdout + calibration/CV)."""
    pacing_ok = bool((pacing_df.loc[pacing_df["Goal met"] != "—", "Goal met"] == True).all())
    rows = [
        ("Holdout gate macro F1 >= 0.75", summary["gate macro F1"], summary["gate macro F1"] >= 0.75),
        ("Holdout damaging leak = 0", summary["damaging leak"], summary["damaging leak"] == 0),
        ("Holdout in-domain -> primary >= 80%", summary["in-domain -> primary"], summary["in-domain -> primary"] >= 0.80),
        ("Holdout damage in-band >= 70%", summary["damage in band"], summary["damage in band"] >= 0.70),
        ("Pacing goals met (Section 6 battles)", pacing_ok, pacing_ok),
        ("Pacing goals met (LOTO-CV)", damage_cal["cv_all_pacing_met"], damage_cal["cv_all_pacing_met"]),
        ("No tuned parameter on a grid edge", [k for k, v in params_on_edge.items() if v] or "none",
         not any(params_on_edge.values())),
    ]
    return pd.DataFrame(rows, columns=["criterion", "value", "pass"])

def run_fingerprint(engine: LexicalCombatEngine, cases: list) -> str:
    """Identifies one (model, parameters, labels) holdout run."""
    payload = json.dumps({"model": engine.model.identity(), "params": {k: str(v) for k, v in snapshot_params(engine).items()},
                          "cases": cases}, sort_keys=True)
    return hashlib.sha1(payload.encode()).hexdigest()[:12]

with section("10.1 Holdout (active model)") as _s:
    if not RUN_HOLDOUT:
        _s.skip("RUN_HOLDOUT = False")
        HOLDOUT_LABEL_STATUS, holdout_result, holdout_summary = "not run", None, None
    else:
        HOLDOUT_CASES, HOLDOUT_LABEL_STATUS, HOLDOUT_AGREEMENT = consolidate_labels(collect_annotations())
        print(f"Holdout labels: {HOLDOUT_LABEL_STATUS} | {len(HOLDOUT_CASES)} cases")
        if HOLDOUT_AGREEMENT:
            display(pd.Series(HOLDOUT_AGREEMENT).to_frame("value").round(3))
            _draft = {(t, g): (l, tr) for t, g, _, l, tr in HOLDOUT_CASES_DRAFT}
            _changed = [(t, g, _draft[(t, g)], (l, tr)) for t, g, _, l, tr in HOLDOUT_CASES if _draft[(t, g)] != (l, tr)]
            print("Draft -> final label changes:", _changed or "none")
        IS_FINAL = PERSISTENT and HOLDOUT_LABEL_STATUS == "majority_vote"
        if not IS_FINAL:
            why = [] if PERSISTENT else ["PERSISTENT = False"]
            if HOLDOUT_LABEL_STATUS != "majority_vote":
                why.append(f"label status '{HOLDOUT_LABEL_STATUS}' (needs >= 2 independent annotators)")
            print("=" * 72 + f"\nNOT FINAL — {'; '.join(why)}. No run log is written.\n" + "=" * 72)
        print(f"Model: {ACTIVE_MODEL} ({MODEL_SOURCE}) — every choice (model, composition, parameters) is fixed; "
              "the holdout was not used for any of them.")
        holdout_result, holdout_summary = evaluate_holdout(engine, HOLDOUT_CASES, HOLDOUT_LABEL_STATUS)
        criteria = success_criteria(holdout_summary, pacing_df, DAMAGE_CAL, PARAMS_ON_EDGE)
        print("Success criteria:")
        display(criteria)
        if IS_FINAL:
            fingerprint = run_fingerprint(engine, HOLDOUT_CASES)
            HOLDOUT_RUN_LOG = holdout_run_log(ACTIVE_MODEL)
            previous = json.loads(HOLDOUT_RUN_LOG.read_text()) if HOLDOUT_RUN_LOG.is_file() else None
            if previous is None:
                HOLDOUT_RUN_LOG.write_text(json.dumps({"fingerprint": fingerprint, "summary":
                                                       {k: str(v) for k, v in holdout_summary.items()}}, indent=2))
                print(f"Final run recorded: {fingerprint}")
            elif previous.get("fingerprint") != fingerprint:
                print(f"WARNING: the holdout was already evaluated with other settings ({previous.get('fingerprint')}); "
                      "these numbers are NOT a clean final result.")
            else:
                print(f"Same settings as the recorded final run ({fingerprint}).")

In [ ]:
# 10.2 Holdout comparison (each model with its own calibration) + error analysis vs MiniLM
# What this cell does:
#   - Evaluates every model on the holdout, each with its own calibration.
#   - Error analysis: best self-trained model vs MiniLM, with likely causes.
HOLDOUT_COMPARISON_LOG = PATHS.artifacts / "holdout_comparison_run.json"

def likely_causes(row, encoder) -> str:
    """Text-analytics reasons for one wrong case, computed from the model and the corpus."""
    tags = []
    for role, text in (("guess", row["Guess"]), ("target", row["Target"])):
        kind = encoder.coverage_kind(text) if hasattr(encoder, "coverage_kind") else "subword"
        if kind == "unknown":
            tags.append(f"{role} OOV (no vector)")
        elif kind == "partial":
            tags.append(f"{role} partly OOV")
        elif kind == "subword (FastText)":
            tags.append(f"{role} built from character n-grams only")
    words = clean_text(row["Guess"]).split()
    freq = min(COUNTS.get(w, 0) for w in words)
    if 0 < freq < 50:
        tags.append(f"rare in corpus (min word freq {freq}) -> data sparsity")
    if len(words) > 1 and hasattr(encoder, "tokens_for") and len(encoder.tokens_for(" ".join(words))) == len(words):
        tags.append("phrase = average of word vectors (no phrase vector, short context)")
    predicted, human = row["Predicted Label"], row["Human Label"]
    if human == "out_of_domain" and predicted not in ("out_of_domain", "invalid"):
        tags.append("passes the gate: shares contexts with the domain (relatedness / polysemy)")
    if human == "in_domain" and predicted == "out_of_domain":
        tags.append("blocked: a rival domain's anchors are closer (corpus topic bias / sense mixing)")
    if human in ("in_domain", "ambiguous") and predicted == "in_domain" and not row["In band"]:
        tags.append("gate right, damage band wrong (relevance scale / ceiling of this target)")
    return "; ".join(tags) or "—"

with section("10.2 Holdout comparison + error analysis") as _s:
    if not RUN_HOLDOUT:
        _s.skip("RUN_HOLDOUT = False")
        HOLDOUT_COMPARISON, HOLDOUT_RESULTS, ERROR_TABLES = None, {}, {}
    else:
        HOLDOUT_RESULTS, _rows, _prints = {}, [], {}
        for _choice in available_model_choices():
            _eng = engine_for(_choice)
            _res, _sum = evaluate_holdout(_eng, HOLDOUT_CASES, HOLDOUT_LABEL_STATUS, verbose=False)
            HOLDOUT_RESULTS[_choice] = _res
            _prints[_choice] = run_fingerprint(_eng, HOLDOUT_CASES)
            _cal = COMPARISON_CAL.loc[MODEL_LABELS[_choice]]
            _rows.append({"Model": MODEL_LABELS[_choice], "role": _cal["role"],
                          "WordSim-353 Spearman": _cal.get("WordSim-353 Spearman"),
                          "SimLex-999 Spearman": _cal.get("SimLex-999 Spearman"),
                          "holdout gate macro F1": _sum["gate macro F1"],
                          "holdout in-domain -> primary": _sum["in-domain -> primary"],
                          "holdout ambiguous pass": _sum["ambiguous pass"],
                          "holdout gate leak (0 dmg)": _sum["gate leak (0 dmg)"],
                          "holdout damaging leak": _sum["damaging leak"],
                          "holdout damage in-band": _sum["damage in band"],
                          "holdout unknown-word guesses": _sum["unknown-word guesses"],
                          "calibration gate macro F1 (CV)": _cal["gate macro F1 (CV)"],
                          "calibration damage in-band (CV)": _cal["damage in-band (LOTO-CV)"],
                          "pacing met (calibration battles)": _cal["pacing met"]})
        HOLDOUT_COMPARISON = pd.DataFrame(_rows).set_index("Model")
        print(f"Holdout ({len(HOLDOUT_CASES)} cases, labels: {HOLDOUT_LABEL_STATUS}) — every model with its own calibration:")
        display(HOLDOUT_COMPARISON.T.round(3))
        if PERSISTENT and HOLDOUT_LABEL_STATUS == "majority_vote":
            _previous = json.loads(HOLDOUT_COMPARISON_LOG.read_text()) if HOLDOUT_COMPARISON_LOG.is_file() else None
            if _previous is None:
                HOLDOUT_COMPARISON_LOG.write_text(json.dumps({"fingerprints": _prints,
                                                              "table": to_jsonable(HOLDOUT_COMPARISON)}, indent=1))
                print("Comparison run recorded.")
            elif _previous.get("fingerprints") != _prints:
                print("WARNING: the holdout comparison was already run with other settings; not a clean final result.")
        else:
            print("Comparison run log not written (needs Drive and >= 2 independent annotators), as in Section 10.1.")

        # Error analysis: best self-trained model vs MiniLM (correct = gate label aligned AND damage in band).
        ERROR_TABLES = {}
        _ours_key, _ref_key = BEST_SELF_TRAINED, "minilm_reference"
        _ours, _enc = HOLDOUT_RESULTS[_ours_key], MODEL_REGISTRY[_ours_key]
        _ours = _ours.assign(correct=_ours["Label Aligned"] & _ours["In band"])
        _ours["likely causes"] = [likely_causes(r, _enc) for _, r in _ours.iterrows()]
        print("\nCorrect holdout cases (gate label aligned AND damage in band) per model:",
              {MODEL_LABELS[c]: int((r["Label Aligned"] & r["In band"]).sum()) for c, r in HOLDOUT_RESULTS.items()})
        _cols = ["Target", "Guess", "Human Label", "Expected tier"]
        if _ref_key in HOLDOUT_RESULTS:
            _ref = HOLDOUT_RESULTS[_ref_key]
            _ref = _ref.assign(correct=_ref["Label Aligned"] & _ref["In band"])
            _both = _ours[_cols + ["Predicted Label", "Hit %", "correct", "likely causes"]].rename(
                columns={"Predicted Label": "ours: gate", "Hit %": "ours: hit %", "correct": "ours ok"})
            _both["MiniLM: gate"], _both["MiniLM: hit %"], _both["MiniLM ok"] = (
                _ref["Predicted Label"].values, _ref["Hit %"].values, _ref["correct"].values)
            ERROR_TABLES["ours wrong, MiniLM right"] = _both[~_both["ours ok"] & _both["MiniLM ok"]]
            ERROR_TABLES["MiniLM wrong, ours right"] = _both[_both["ours ok"] & ~_both["MiniLM ok"]]
            ERROR_TABLES["both wrong"] = _both[~_both["ours ok"] & ~_both["MiniLM ok"]]
            for _name, _table in ERROR_TABLES.items():
                print(f"\n{_name} ({MODEL_LABELS[_ours_key]} vs MiniLM): {len(_table)} cases")
                if len(_table):
                    display(_table.drop(columns=["ours ok", "MiniLM ok"]).round(2))
        else:
            ERROR_TABLES["ours wrong"] = _ours[~_ours["correct"]][_cols + ["Predicted Label", "Hit %", "likely causes"]]
            print(f"\nMiniLM is not available — wrong cases of {MODEL_LABELS[_ours_key]} only:")
            display(ERROR_TABLES["ours wrong"].round(2))

### 10.3 Reading the error analysis with text-analytics concepts

Table 10.2 lists causes computed from each case's real data. Use the concepts below to explain them in the report (take examples from the table of the actual run, not from this text).

- **Data sparsity:** word embeddings learn meaning from the contexts a word appears in. Rare words (e.g. < 50 occurrences) get noisy vectors, while MiniLM was pretrained on billions of words and fine-tuned on over a billion sentence pairs, so it has a big advantage on rare words.
- **OOV:** Word2Vec / PPMI cannot represent words outside the vocab at all (the game shows "Unknown word"). FastText can build a vector from character n-grams, but for a truly unseen word the vector only reflects word form (e.g. the `-tion` suffix), not meaning.
- **Polysemy:** a static embedding has **one vector per word**, so all senses are averaged together. For example *brand* (company brand / cattle brand) can make *cowboy* close to *brand*. MiniLM is a contextual encoder, but with 1–3 word phrases its context is also very short.
- **Relatedness ≠ similarity:** the distributional hypothesis captures "words that appear together" well (*comet* – *horoscope* share astronomy/astrology articles), so a guess may pass the gate even though humans judge it out of domain. This shows up as a higher Spearman on WordSim-353 (relatedness) than on SimLex-999 (similarity).
- **Short context / phrase composition:** phrases that `Phrases` did not merge into one token are averaged from their words, which loses word order and compositional meaning (*spam filter* ≈ *spam* + *filter*).
- **Corpus bias:** Wikipedia has many science/history articles but less everyday business/IT language (*inbox*, *chat app*). Domains the corpus covers less tend to have more errors (see the topic estimate in 4.3).

## 11. Play: the game with our own model (Phase 8)

By default the game uses the best self-trained model (`MODEL_CHOICE = "auto"`). **The target bank is re-screened with this model:** targets are dropped if the model does not know them, they are too long/broad, the target's own gate is not primary, the span is too narrow, or fewer than 4 good answers exist (r ≥ 0.6 and passing the gate). Simulated battles run the 3 classic stages (with *typical* guesses) and 1 random stage, showing each step: **gate → relevance → damage**, as in the original notebook. To play yourself, use the widget in the last cell.

In [ ]:
# 11. Play: target bank screened with the active model, simulated battles (3 classic + 1 random) or interactive game
# What this cell does:
#   - Screens the target bank with the active model (drops unknown / weak targets).
#   - Simulated battles: 3 classic stages + 1 random stage, step by step.
#   - Interactive input() game when PLAY_INTERACTIVE = True.
RANDOM_ROUNDS = 4
RANDOM_BOSS_HP = 200
MIN_SPAN_RATIO = 0.6      # raw span must be >= this x span floor
GOOD_R = 0.6
MIN_GOOD_ANSWERS = 4
FALLBACK_TARGETS = 5      # used only if a topic has no eligible target

def extra_targets() -> dict:
    """Calibration and holdout targets per topic (playable after Section 7)."""
    out = {d: [] for d in DOMAINS}
    for t, d in CALIBRATION_TARGETS.items():
        out[d].append(t)
    for t, _, d, _, _ in HOLDOUT_CASES_DRAFT:
        if t not in out[d]:
            out[d].append(t)
    return out

def render_turn_result(engine: LexicalCombatEngine, result: dict, remaining_hp: int, required_domain: str) -> None:
    """Step-by-step feedback: gate, relevance, damage."""
    if not result["valid"]:
        print(f"  [!] {result['reason']}\n")
        return
    d = required_domain
    print(f"  Guess: '{result['word']}'")
    print(f"  Step 1 · Domain gate ({d}): raw {result['domain_raw_scores'][d]:.3f} − background "
          f"{result['domain_background'][d]:.3f} = {result['domain_scores'][d]:+.3f} | "
          f"lead vs best other {result['domain_lead']:+.3f} → {result['domain_match_type'].upper()}"
          + (f" ×{result['multiplier']:.2f}" if result["domain_allowed"] else ""))
    print("          Domain share (display): " + " | ".join(
        f"{x}: {result['domain_probs'][x]:.0%}" for x in engine.data.domains))
    print(f"  Step 2 · Target relevance: cos {result['similarity']:.3f} | noise floor "
          f"{result['noise_floor']:.3f} | ceiling {result['ceiling']:.3f} → r = {result['relevance']:.2f}")
    if result["domain_allowed"]:
        parts = [f"base {result['base_percent']:.1f}%", f"domain ×{result['multiplier']:.2f}"]
        if result["is_generic"]:
            parts.append(f"generic ×{result['generic_multiplier']:.2f}")
        if result.get("graze_floor") and result["base_percent"] <= 0:
            parts.append(f"in-domain graze floor ({engine.config.chip_percent}%)")
        if result["is_echo"]:
            parts.append(f"echo ×{result['echo_multiplier']:.2f} (cos {result['echo_similarity']:.2f} >= "
                         f"{engine.echo_threshold():.2f} to an earlier guess)")
        actual = result.get("actual_damage", result["damage"])
        overkill = f" (overkill {result['overkill']})" if result.get("overkill") else ""
        print(f"  Step 3 · Damage: {' · '.join(parts)} → {result['hit_percent']:.1f}% "
              f"= -{actual} HP{overkill} | {result['tier']}")
    else:
        print(f"  Step 3 · Damage: blocked by the domain gate → 0 HP | {result['tier']}")
    print(f"  Latency: {result['latency_ms']:.1f} ms | Enemy HP left: {max(remaining_hp, 0)}\n")

def play_stage(engine: LexicalCombatEngine, stage: dict):
    """One interactive stage (uses input()). Returns a record, or None on 'quit'."""
    hp, used_guesses, turns = stage["hp"], set(), 0
    print("=" * 72)
    print(f"STAGE {stage['stage']}: {stage['enemy']} | HP: {hp}")
    print(f"Target: '{stage['target'].upper()}' | Domain: {stage['domain'].upper()}")
    print("Enter a word/phrase (1–3 words), 'hint', or 'quit'.")
    print("=" * 72)
    while hp > 0:
        guess = input(f"[Stage {stage['stage']} | HP {hp}] Your guess: ").strip()
        if clean_text(guess) == "quit":
            return None
        if clean_text(guess) == "hint":
            print(f"  [Hint] Try something like: {engine.hint(stage['target'], stage['domain'], used_guesses)}\n")
            continue
        result = engine.evaluate_attack(guess, stage["target"], stage["domain"], used_guesses, stage["hp"])
        if result["valid"]:
            result = engine.resolve_damage(result, stage["hp"], hp)
            turns += 1
            used_guesses.add(result["word"])
            hp -= result["actual_damage"]
        render_turn_result(engine, result, hp, stage["domain"])
    print(f">>> {stage['enemy']} defeated in {turns} valid attack turns!\n")
    return {"Stage": stage["stage"], "Enemy": stage["enemy"], "Domain": stage["domain"],
            "Turns": turns, "Remaining HP": max(hp, 0)}

def pool_gate_kinds(engine: LexicalCombatEngine, domain: str) -> dict:
    """Gate kind of every pool term when played in its own domain."""
    pool, vectors = engine.pools[domain]
    adjusted, _ = engine.domain_scores(vectors, engine.config.anchor_topk)
    col = DOMAINS.index(domain)
    required = adjusted[:, col]
    lead = required - np.delete(adjusted, col, axis=1).max(axis=1)
    return dict(zip(pool, engine.gate_kinds(required, lead)))

def build_target_bank(engine: LexicalCombatEngine) -> pd.DataFrame:
    """Screen candidate targets per topic, with the reason for each decision."""
    rows, extras = [], extra_targets()
    for domain in DOMAINS:
        kinds = pool_gate_kinds(engine, domain)
        candidates = list(dict.fromkeys(extras[domain] + engine.pools[domain][0]))
        vectors = engine.encode(candidates)
        for target, vector in zip(candidates, vectors):
            adjusted, _, _, _ = engine.domain_scores_from_vector(vector)
            _, _, self_kind = engine.gate(adjusted, domain)
            cal = engine.target_calibration(target, domain)
            noise, ceiling_raw = engine.raw_bounds(cal)
            noise, ceiling = engine.bounds(cal)
            r = normalized_relevance(cal["pool_sims"], noise, ceiling)
            good = sum(1 for term, rel in zip(cal["pool_terms"], r)
                       if rel >= GOOD_R and kinds.get(term, "out_of_domain") != "out_of_domain")
            span = ceiling_raw - noise
            known = getattr(engine.model, "is_known", None)
            if known is not None and not known(target):
                reason = "target unknown to the model"
            elif len(target.split()) > 2 or engine.is_generic(target):
                reason = "generic / too long"
            elif self_kind != "primary":
                reason = f"target itself is {self_kind}"
            elif span < MIN_SPAN_RATIO * engine.span_floor:
                reason = "span too narrow"
            elif good < MIN_GOOD_ANSWERS:
                reason = f"only {good} good answers"
            else:
                reason = "ok"
            rows.append({"domain": domain, "target": target, "self gate": self_kind,
                         "span": round(span, 3), "good answers": good, "eligible": reason == "ok",
                         "reason": reason})
    return pd.DataFrame(rows)

def random_stages(bank: pd.DataFrame, rounds: int, seed: int) -> list:
    """Random topic order (all 4 before repeating), random eligible target per topic."""
    rng = random.Random(seed)
    eligible = {}
    for d in DOMAINS:
        rows = bank[bank["domain"] == d]
        ok = rows.loc[rows["eligible"], "target"].tolist()
        if not ok:
            ranked = rows.assign(primary=rows["self gate"] == "primary").sort_values(
                ["primary", "good answers", "span"], ascending=False)
            ok = ranked["target"].head(FALLBACK_TARGETS).tolist()
            print(f"WARNING: no {d} target passed every check; using the top {len(ok)} ranked candidates.")
        eligible[d] = ok
    topics = [d for d in DOMAINS if eligible[d]]
    stages, order, used = [], [], set()
    for number in range(1, rounds + 1):
        if not order:
            order = rng.sample(topics, len(topics))
        domain = order.pop()
        choices = [t for t in eligible[domain] if t not in used] or eligible[domain]
        target = rng.choice(choices)
        used.add(target)
        stages.append({"stage": number, "enemy": rng.choice(BOSS_NAMES[domain]), "target": target,
                       "domain": domain, "hp": RANDOM_BOSS_HP})
    return stages

def run_game(engine: LexicalCombatEngine, stages: list) -> list:
    """Interactive loop over stages; 'quit' ends the game."""
    records = []
    for stage in stages:
        print(f"\nTopic: {stage['domain'].upper()} | Boss: {stage['enemy']}")
        record = play_stage(engine, stage)
        if record is None:
            print("Game ended.")
            break
        records.append(record)
    if records:
        print("Summary:")
        display(pd.DataFrame(records))
    return records

def simulated_battle(engine: LexicalCombatEngine, bank: pd.DataFrame, seed: int) -> pd.DataFrame:
    """One random stage played by an auto-player (no input())."""
    stage = random_stages(bank, 1, seed)[0]
    cal = engine.target_calibration(stage["target"], stage["domain"])
    noise, ceiling = engine.bounds(cal)
    r = normalized_relevance(cal["pool_sims"], noise, ceiling)
    mid = [t for _, t in sorted(zip(np.abs(r - 0.6), cal["pool_terms"]))[:6]]
    guesses = ["banana", "spreadsheet"] + mid
    print(f"Simulated: {stage['enemy']} | target '{stage['target']}' ({stage['domain']}) | guesses {guesses}")
    battle = engine.simulate_battle(stage["target"], stage["domain"], stage["hp"], guesses,
                                    on_turn=lambda res, hp, dom: render_turn_result(engine, res, hp, dom))
    display(battle)
    return battle

def classic_simulation(engine: LexicalCombatEngine) -> pd.DataFrame:
    """The 3 classic stages played with the 'typical' reference guesses, step by step."""
    rows = []
    for stage in GAME_STAGES:
        guesses = REFERENCE_PLAYS[stage["target"]]["typical"][0]
        print("=" * 72 + f"\nClassic stage {stage['stage']}: {stage['enemy']} | target '{stage['target']}' "
              f"({stage['domain']}) | HP {stage['hp']}\n" + "=" * 72)
        battle = engine.simulate_battle(stage["target"], stage["domain"], stage["hp"], guesses,
                                        on_turn=lambda res, hp, dom: render_turn_result(engine, res, hp, dom))
        display(battle)
        rows.append({"Stage": stage["stage"], "Enemy": stage["enemy"], "Target": stage["target"],
                     **engine.summarize_battle(battle, stage["hp"])})
    return pd.DataFrame(rows)

with section("11. Play"):
    print(f"Model: {ACTIVE_MODEL} ({MODEL_SOURCE})")
    TARGET_BANK = build_target_bank(engine)
    print("Target bank (eligible / screened) per topic:")
    display(TARGET_BANK.groupby("domain")["eligible"].agg(["sum", "size"])
            .rename(columns={"sum": "eligible", "size": "screened"}))
    print("Rejected examples:")
    display(TARGET_BANK[~TARGET_BANK["eligible"]].groupby("reason").head(2)[["domain", "target", "reason"]])
    if not PLAY_INTERACTIVE:
        print("PLAY_INTERACTIVE = False — simulated battles: 3 classic stages + 1 random stage "
              "(set it to True to play with input(), or use the Play Game widget in the last cell).")
        CLASSIC_SIM = classic_simulation(engine)
        print("Classic stages summary:")
        display(CLASSIC_SIM)
        print("\nRandom stage (topic and target drawn from the screened bank):")
        RANDOM_SIM = simulated_battle(engine, TARGET_BANK, SEED)
    elif PLAY_MODE == "classic":
        run_game(engine, GAME_STAGES)
    else:
        print("\nRandom-topic game: 1–3 English words per guess | 'hint' | 'quit'")
        run_game(engine, random_stages(TARGET_BANK, RANDOM_ROUNDS, SEED))

## 12. Summary for the report (Phase 9)

> This part describes the **method** (it does not depend on the run). All real numbers are in the next cell, which builds Markdown from this run's results (ready to paste into Notion, and saved as `report_selftrained.md` in the artifacts folder).

### Data sources and licenses
- **Simple English Wikipedia** (`wikimedia/wikipedia`, `20231101.simple`) via Hugging Face `datasets` (~242k articles) — CC BY-SA 3.0 / GFDL (per the dataset card)
- **English Wikipedia** (`20231101.en`, streaming), only articles containing game words (holdout excluded) that still occur fewer than 20 times — CC BY-SA 3.0 / GFDL
- **WordSim-353** and **SimLex-999** evaluation sets (files shipped with gensim), used for evaluation only
- Game data (anchors, term pools, calibration, holdout) is **never put into the corpus**; it is used only to check coverage

### Preprocessing
Sentence split → the game's `clean_text()` (lowercase, non-letters → spaces; prevents train/serve skew) → drop sentences < 3 tokens → `Phrases` bigram + trigram (`min_count=5`, `threshold=10`, connector words) → one sentence per line · vocabulary = tokens seen ≥ 5 times

### Architecture and hyper-parameters
| Model | Details |
|---|---|
| PPMI + SVD | vocab = 50k most frequent words (+ non-holdout game words), window ∈ {2, 5} with weight `(W−d+1)/W`, PPMI + context smoothing α=0.75, randomized SVD → `U·√Σ`, dim ∈ {100, 300} |
| Word2Vec | negative sampling (`negative=10`), `min_count=5`, `sample=1e-3`; search `vector_size` {100, 300} × `window` {5, 10} × `sg` {CBOW, skip-gram}; final `epochs=5` (fewer when over the time budget) |
| FastText | skip-gram + character n-grams 3–6, `bucket=500k`; search `vector_size` {100, 300} at the best Word2Vec window |
| Phrase / sentence composition | `mean` or `sif` (SIF weighting `a/(a+p(w))`, a=10⁻³, stopwords removed, first principal component from 20k corpus sentences removed) — chosen per model on the calibration set |
| Game | original engine, gate (top-k anchor cosine − background), damage curve and tiers; gate + damage re-calibrated per model (grid expands automatically at its edges) |

### Evaluation design (preventing data leakage)
- Hyper-parameters / composition / model are chosen with the **selection score** from the calibration set (gate CV, damage LOTO-CV, pacing) + intrinsic metrics only
- The **holdout** (28 cases, all-new targets) is used once at the end and never used to select data, models or thresholds
- MiniLM is a **reference baseline**, calibrated with the same procedure for a fair comparison

### Limitations
- Small corpus (tens of millions of tokens) compared with MiniLM's training data (billions of sentences) → rare words get poor vectors, and our models are expected to lose to MiniLM, which is acceptable
- Static word embeddings: one vector per word (polysemy), phrases are averaged from words (word order is lost), they capture relatedness more than similarity
- Multi-threaded gensim is not 100% deterministic · if training exceeds the time budget, epochs are reduced / part of the corpus is used (recorded in `config.json`)
- Small calibration set and holdout (28 holdout cases, 1 annotator) → wide confidence intervals

### Future work
- Larger corpus (full English Wikipedia, business news, academic abstracts such as arXiv) and longer training on a GPU / many-core machine
- **A Transformer sentence encoder from scratch** (e.g. a small BERT + contrastive learning on Wikipedia sentence pairs) — needs many times more data and GPU, so it is not part of the main version
- Sense embeddings / contextual vectors for polysemy, retrofitting with WordNet, ≥ 2 holdout annotators to measure inter-rater agreement

In [ ]:
# 12. Notion-ready report built from THIS run's numbers (+ saved as report_selftrained.md) and run summary
# What this cell does:
#   - Builds a Notion-ready Markdown report from this run's numbers.
#   - Saves it as report_selftrained.md and prints the run summary table.
def md_table(frame: pd.DataFrame, index: bool = True, digits: int = 3) -> str:
    """Markdown table without extra packages."""
    frame = frame.reset_index() if index else frame
    def fmt(v):
        if isinstance(v, (float, np.floating)):
            return "—" if v != v else f"{v:.{digits}f}"
        return str(v).replace("|", "/").replace("\n", " ")
    head = "| " + " | ".join(map(str, frame.columns)) + " |"
    sep = "|" + "---|" * len(frame.columns)
    rows = ["| " + " | ".join(fmt(v) for v in row) + " |" for row in frame.itertuples(index=False)]
    return "\n".join([head, sep] + rows)

def build_report() -> str:
    """Markdown report: corpus, training, intrinsic, calibration, holdout, pass criteria."""
    cs = CORPUS_STATS
    parts = [f"# Lexical Combat Game — Self-Trained Models: Results ({time.strftime('%Y-%m-%d')})", "",
             f"**Playing model:** {MODEL_LABELS[ACTIVE_MODEL]} — {MODEL_SOURCE} · "
             f"**Best self-trained model (calibration set):** {MODEL_LABELS[BEST_SELF_TRAINED]} · "
             f"QUICK_RUN = {QUICK_RUN}", "",
             "## 1. Corpus", "",
             md_table(pd.DataFrame(cs["sources"]).T.rename_axis("source")), "",
             f"- Articles {cs['articles']:,} · sentences {cs['sentences']:,} · tokens {cs['tokens']:,} · "
             f"vocab (≥{CORPUS_CONFIG['min_count']}) {cs['vocab_min_count']:,} · phrase tokens {cs['phrase_tokens_min_count']:,}",
             "- License: " + "; ".join(cs["licenses"].values()),
             f"- Coverage of game words: {CORPUS_COVERAGE_TOTALS['all game words']:.1%} (non-holdout "
             f"{CORPUS_COVERAGE_TOTALS['non-holdout game words']:.1%}) → {'PASS' if COVERAGE_PASS else 'FAIL'} (target 95%)",
             f"- Targeted supplement: scanned {cs.get('supplement', {}).get('scanned', 0):,} articles, kept "
             f"{cs.get('supplement', {}).get('kept', 0):,} articles", "",
             "## 2. Training", ""]
    train_rows = [{"model": MODEL_LABELS[f], "params": json.dumps(MODEL_INFO[f].get("params", {}), ensure_ascii=False),
                   "composition": MODEL_INFO[f]["composition"], "train s": MODEL_INFO[f].get("train_seconds"),
                   "corpus share": MODEL_INFO[f].get("train_fraction", 1.0), "size MB": MODEL_INFO[f].get("model_mb"),
                   "selection score": MODEL_INFO[f]["selection"]["score"]} for f in MODEL_REGISTRY]
    parts += [md_table(pd.DataFrame(train_rows), index=False), "",
              "## 3. Intrinsic evaluation", "", md_table(INTRINSIC_TABLE), "",
              "## 4. Calibration set (each model uses its own calibration)", "",
              md_table(COMPARISON_CAL.drop(columns=["calibration"]).T.rename_axis("metric")), ""]
    if HOLDOUT_COMPARISON is not None:
        parts += [f"## 5. Holdout ({len(HOLDOUT_CASES)} cases, labels: {HOLDOUT_LABEL_STATUS}, evaluated once)", "",
                  md_table(HOLDOUT_COMPARISON.T.rename_axis("metric")), ""]
        for name, table in ERROR_TABLES.items():
            if len(table):
                cols = [c for c in ["Target", "Guess", "Human Label", "ours: gate", "MiniLM: gate", "Predicted Label",
                                    "likely causes"] if c in table.columns]
                parts += [f"**{name}** ({len(table)})", "", md_table(table[cols], index=False), ""]
    checks = [("Phase 2: coverage ≥ 95%", COVERAGE_PASS),
              ("Phase 5: adapter unit tests", bool(UNIT_TESTS["pass"].all())),
              ("Phase 6: pacing on every stage + leak ≤ 5% (active model)", PHASE6_PASS)]
    if RUN_HOLDOUT and holdout_summary is not None:
        checks += [(f"Holdout: {r[0]}", bool(r[2])) for r in criteria.itertuples(index=False)]
    parts += ["## 6. Pass criteria", "", md_table(pd.DataFrame(checks, columns=["criterion", "pass"]), index=False), "",
              "> All numbers are reported as measured in this run; no test data was changed · MiniLM is a reference baseline only"]
    return "\n".join(parts)

with section("12. Report"):
    REPORT_MD = build_report()
    (PATHS.artifacts / "report_selftrained.md").write_text(REPORT_MD, encoding="utf-8")
    display(Markdown(REPORT_MD))
    print(f"\nSaved: {PATHS.artifacts / 'report_selftrained.md'}")

print("\nRun summary")
display(pd.DataFrame(RUN_LOG)[["section", "status", "reason", "seconds"]])
print(f"Total section time: {sum(e['seconds'] for e in RUN_LOG):.1f} s | model: {ACTIVE_MODEL} ({MODEL_SOURCE}) | "
      f"PERSISTENT: {PERSISTENT} | calibration: {CALIBRATION_SOURCE} | holdout labels: {HOLDOUT_LABEL_STATUS}")

## Play Game
Run this cell (or Run all) and play in the box that appears: each game draws random topics and a random target per topic from the screened target bank. Type 1–3 English words and press **Enter** or **Attack**; **Hint** suggests a word, **Skip boss** moves on, **Quit** ends the game, **New game** draws new stages. Each turn is shown as a card with one detail per line; when a stage ends you get its summary table (every guess, damage, boss HP left), and when the game ends a game summary. Run all does not wait here, because the game is a widget, not `input()`.

The game uses the self-trained model (default `MODEL_CHOICE = "auto"`). If you type a word the model does not know, you will see *Unknown word …* and the turn is not counted.

In [ ]:
# Play Game — random topic + random target; type your own guesses in the box below.
# What this cell does:
#   - GameSession: game state (stages, HP, guesses, hints, skips).
#   - HTML cards for each turn, stage summaries and the game summary.
#   - ipywidgets UI (Attack / Hint / Skip / Quit / New game) with a text-mode fallback.
# Uses widgets, so Run all never waits for input. Without ipywidgets: call play_in_text_mode(GAME).
import warnings

TOPIC_COLORS = {"Business": "#3b82f6", "Technology": "#10b981", "Science": "#a855f7", "Arts": "#f43f5e"}
TIER_COLORS = {"CRITICAL STRIKE": "#ef4444", "STRONG HIT": "#f97316", "SOLID HIT": "#eab308",
               "WEAK GRAZE": "#22c55e", "NO EFFECT": "#71717a", "BLOCKED — OUT OF DOMAIN": "#52525b"}
STATUS_COLORS = {"Defeated": "#22c55e", "Skipped": "#eab308", "Quit": "#f43f5e", "In progress": "#3b82f6"}
GATE_SHORT = {"primary": "primary", "secondary": "secondary", "weak_secondary": "weak secondary",
              "out_of_domain": "blocked"}
GATE_TEXT = {"primary": "PRIMARY — in the topic", "secondary": "SECONDARY — related topic",
             "weak_secondary": "WEAK SECONDARY — loosely related", "out_of_domain": "BLOCKED — wrong topic"}
CARD = "background:#1e1e24;border:1px solid #3f3f46;border-radius:10px;color:#f4f4f5;font-family:sans-serif;"

def _badge(text: str, color: str) -> str:
    """Small coloured label."""
    return (f"<span style='background:{color};color:#fff;padding:2px 8px;border-radius:999px;"
            f"font-size:11px;font-weight:bold;white-space:nowrap'>{_html.escape(text)}</span>")

def _line(label: str, value: str, note: str = "") -> str:
    """One labelled detail line of a turn card."""
    note_html = f"<span style='color:#a1a1aa;font-size:12px'> {note}</span>" if note else ""
    return (f"<div style='display:flex;gap:10px;padding:3px 0;font-size:13px'>"
            f"<div style='width:150px;color:#a1a1aa;flex-shrink:0'>{label}</div>"
            f"<div style='color:#f4f4f5'>{value}{note_html}</div></div>")

def turn_card_html(engine: LexicalCombatEngine, result: dict, hp_before: int, hp_after: int,
                   domain: str, turn: int) -> str:
    """One guess, one detail per line."""
    if not result["valid"]:
        return (f"<div style='{CARD}padding:8px 14px;margin:6px 0;border-color:#a16207'>"
                f"⚠️ <b>Not counted:</b> {_html.escape(result['reason'])}</div>")
    tier = result["tier"]
    color = TIER_COLORS.get(tier, "#71717a")
    damage = result.get("actual_damage", result["damage"])
    share = " · ".join(f"{d} {result['domain_probs'][d]:.0%}" for d in engine.data.domains)
    rival = max((d for d in engine.data.domains if d != domain), key=lambda d: result["domain_scores"][d])
    multipliers = [f"base {result['base_percent']:.1f}%", f"topic ×{result['multiplier']:.2f}"]
    if result["is_generic"]:
        multipliers.append(f"too generic ×{result['generic_multiplier']:.2f}")
    if result["is_echo"]:
        multipliers.append(f"repeat idea ×{result['echo_multiplier']:.2f}")
    if result.get("graze_floor") and result["base_percent"] <= 0:
        multipliers.append(f"in-topic graze floor {engine.config.chip_percent}%")
    damage_line = (" · ".join(multipliers) + f" → <b>{result['hit_percent']:.1f}%</b> of boss HP"
                   if result["domain_allowed"] else "blocked by the topic gate")
    lines = [
        _line("1 · Topic gate", f"<b>{GATE_TEXT[result['domain_match_type']]}</b>",
              f"({domain} score {result['domain_scores'][domain]:+.3f}, lead {result['domain_lead']:+.3f} vs {rival})"),
        _line("&nbsp;&nbsp;&nbsp;Topic share", share),
        _line("2 · Relevance", f"<b>r = {result['relevance']:.2f}</b>",
              f"(cos {result['similarity']:.3f} · noise floor {result['noise_floor']:.3f} · ceiling {result['ceiling']:.3f})"),
        _line("3 · Damage", damage_line),
        _line("&nbsp;&nbsp;&nbsp;Boss HP", f"{hp_before} → <b>{max(hp_after, 0)}</b>",
              f"({result['latency_ms']:.0f} ms)"),
    ]
    head = (f"<div style='display:flex;justify-content:space-between;align-items:center;margin-bottom:6px'>"
            f"<div style='font-size:15px'><span style='color:#a1a1aa'>Turn {turn}</span> · "
            f"<b>'{_html.escape(result['word'])}'</b></div>"
            f"<div>{_badge(tier, color)} <b style='color:{color};font-size:16px;margin-left:6px'>"
            f"{'−' + str(damage) if damage else '0'} HP</b></div></div>")
    return f"<div style='{CARD}padding:10px 14px;margin:8px 0;border-left:5px solid {color}'>{head}{''.join(lines)}</div>"

def _table_html(frame: pd.DataFrame, colors: dict = None) -> str:
    """Dark HTML table; colors maps column -> {value: colour} for badge cells."""
    colors = colors or {}
    head = "".join(f"<th style='text-align:left;padding:6px 10px;color:#a1a1aa;font-size:12px;"
                   f"border-bottom:1px solid #3f3f46'>{_html.escape(str(c))}</th>" for c in frame.columns)
    rows = []
    for _, row in frame.iterrows():
        cells = []
        for c in frame.columns:
            value = row[c]
            text = _badge(str(value), colors[c].get(value, "#52525b")) if c in colors else _html.escape(str(value))
            cells.append(f"<td style='padding:5px 10px;font-size:13px;color:#e4e4e7;border-bottom:1px solid #27272a'>{text}</td>")
        rows.append("<tr>" + "".join(cells) + "</tr>")
    return f"<table style='border-collapse:collapse;width:100%'><tr>{head}</tr>{''.join(rows)}</table>"

def stage_summary_html(stage_log: dict) -> str:
    """Per-stage result: outcome, HP left and every counted guess."""
    color = STATUS_COLORS[stage_log["status"]]
    guesses = pd.DataFrame(stage_log["guesses"]) if stage_log["guesses"] else pd.DataFrame(
        columns=["Turn", "Guess", "Topic gate", "Relevance r", "Damage %", "Damage HP", "Tier", "Boss HP after"])
    total = int(guesses["Damage HP"].sum()) if len(guesses) else 0
    best = guesses.loc[guesses["Damage HP"].idxmax(), "Guess"] if len(guesses) and total > 0 else "—"
    hp_pct = stage_log["hp_left"] / stage_log["hp_max"]
    facts = (f"<div style='display:flex;gap:18px;flex-wrap:wrap;font-size:13px;color:#d4d4d8;margin:8px 0'>"
             f"<div>Outcome {_badge(stage_log['status'], color)}</div>"
             f"<div>Boss HP left <b>{stage_log['hp_left']} / {stage_log['hp_max']}</b> ({hp_pct:.0%})</div>"
             f"<div>Turns <b>{stage_log['turns']}</b></div><div>Total damage <b>{total} HP</b></div>"
             f"<div>Best guess <b>{_html.escape(str(best))}</b></div></div>")
    title = (f"<div style='font-size:16px;font-weight:bold'>📋 Stage {stage_log['stage']} summary — "
             f"{_html.escape(stage_log['enemy'])} · {stage_log['domain']} · target "
             f"<span style='color:#fde047'>'{_html.escape(stage_log['target'])}'</span></div>")
    body = _table_html(guesses, {"Tier": TIER_COLORS}) if len(guesses) else \
        "<div style='color:#a1a1aa;font-size:13px'>No counted guesses.</div>"
    return f"<div style='{CARD}padding:12px 16px;margin:10px 0;border:2px solid {color}'>{title}{facts}{body}</div>"

def game_summary_html(stage_logs: list) -> str:
    """Whole-game table: one row per stage played."""
    rows = []
    for s in stage_logs:
        damage = sum(g["Damage HP"] for g in s["guesses"])
        rows.append({"Stage": s["stage"], "Topic": s["domain"], "Target": s["target"], "Boss": s["enemy"],
                     "Outcome": s["status"], "Boss HP left": f"{s['hp_left']} / {s['hp_max']}",
                     "Turns": s["turns"], "Damage dealt": damage})
    frame = pd.DataFrame(rows)
    defeated = sum(s["status"] == "Defeated" for s in stage_logs)
    turns = sum(s["turns"] for s in stage_logs)
    title = (f"<div style='font-size:18px;font-weight:bold;margin-bottom:8px'>🏁 Game summary — "
             f"{defeated}/{len(stage_logs)} bosses defeated · {turns} turns</div>")
    return f"<div style='{CARD}padding:14px 16px;margin:10px 0;border:2px solid #a855f7'>{title}" \
           f"{_table_html(frame, {'Outcome': STATUS_COLORS})}</div>"

class GameSession:
    """One game: random stages played one guess at a time, with a per-stage log."""

    def __init__(self, engine: LexicalCombatEngine, bank: pd.DataFrame, rounds: int, seed: int):
        self.engine, self.bank, self.rounds = engine, bank, rounds
        self.seed = seed
        self.new_game(seed)

    def new_game(self, seed: int = None) -> None:
        """Draw new stages (topic order, target per topic) and reset."""
        self.seed = self.seed + 1 if seed is None else seed
        self.stages = random_stages(self.bank, self.rounds, self.seed)
        self.index, self.stage_logs, self.quit = 0, [], False
        self._start_stage()

    def _start_stage(self) -> None:
        s = self.stage
        self.hp, self.used, self.turns = s["hp"], set(), 0
        self.stage_logs.append({"stage": s["stage"], "enemy": s["enemy"], "domain": s["domain"],
                                "target": s["target"], "hp_max": s["hp"], "hp_left": s["hp"],
                                "turns": 0, "status": "In progress", "guesses": []})

    @property
    def stage(self) -> dict:
        return self.stages[min(self.index, len(self.stages) - 1)]

    @property
    def current_log(self) -> dict:
        return self.stage_logs[-1]

    @property
    def finished(self) -> bool:
        return self.quit or self.index >= len(self.stages)

    @property
    def records(self) -> list:
        """Defeated stages (kept for compatibility)."""
        return [s for s in self.stage_logs if s["status"] == "Defeated"]

    def status(self) -> str:
        if self.finished:
            return f"Game over — {len(self.records)}/{len(self.stage_logs)} bosses defeated."
        s = self.stage
        return (f"Stage {s['stage']}/{len(self.stages)} · Topic {s['domain'].upper()} · Boss {s['enemy']} · "
                f"Target '{s['target'].upper()}' · HP {self.hp}/{s['hp']} · turns {self.turns}")

    def _end_stage(self, status: str) -> str:
        """Close the current stage; returns its summary HTML."""
        log = self.current_log
        log.update(status=status, hp_left=max(self.hp, 0), turns=self.turns)
        self.index += 1
        summary = stage_summary_html(log)
        if not self.finished:
            self._start_stage()
        return summary

    def attack(self, guess: str) -> list:
        """Score one guess. Returns HTML blocks to show (turn card, then summaries if the stage ended)."""
        if self.finished:
            return [f"<div style='{CARD}padding:8px 14px'>Game over. Press <b>New game</b> to play again.</div>"]
        s = self.stage
        hp_before = self.hp
        result = self.engine.evaluate_attack(guess, s["target"], s["domain"], self.used, s["hp"])
        if result["valid"]:
            result = self.engine.resolve_damage(result, s["hp"], self.hp)
            self.turns += 1
            self.used.add(result["word"])
            self.hp -= result["actual_damage"]
            self.current_log["guesses"].append({
                "Turn": self.turns, "Guess": result["word"], "Topic gate": GATE_SHORT[result["domain_match_type"]],
                "Relevance r": round(result["relevance"], 2), "Damage %": round(result["hit_percent"], 1),
                "Damage HP": result["actual_damage"], "Tier": result["tier"], "Boss HP after": max(self.hp, 0)})
            self.current_log.update(hp_left=max(self.hp, 0), turns=self.turns)
        blocks = [turn_card_html(self.engine, result, hp_before, self.hp, s["domain"], self.turns)]
        if self.hp <= 0:
            blocks.append(self._end_stage("Defeated"))
            if self.finished:
                blocks.append(game_summary_html(self.stage_logs))
        return blocks

    def hint(self) -> list:
        if self.finished:
            return []
        word = self.engine.hint(self.stage["target"], self.stage["domain"], self.used)
        return [f"<div style='{CARD}padding:8px 14px;margin:6px 0;border-color:#06b6d4'>💡 <b>Hint:</b> "
                f"try something like <b style='color:#67e8f9'>{_html.escape(word)}</b></div>"]

    def skip(self) -> list:
        """Give up the current boss and move on."""
        if self.finished:
            return []
        blocks = [self._end_stage("Skipped")]
        if self.finished:
            blocks.append(game_summary_html(self.stage_logs))
        return blocks

    def end_game(self) -> list:
        """Quit now: summary of the current stage (with the boss's HP left) and of the whole game."""
        if self.finished:
            return [game_summary_html(self.stage_logs)] if self.stage_logs else []
        log = self.current_log
        log.update(status="Quit", hp_left=max(self.hp, 0), turns=self.turns)
        self.quit = True
        return [stage_summary_html(log), game_summary_html(self.stage_logs)]

def play_in_text_mode(session: GameSession) -> None:
    """Fallback without widgets: input() loop ('hint', 'skip', 'quit'); prints the summary tables."""
    while not session.finished:
        guess = input(f"[{session.status()}] Your guess: ").strip()
        command = clean_text(guess)
        if command == "quit":
            session.end_game()
            break
        if command == "hint":
            print(f"  Hint: {session.engine.hint(session.stage['target'], session.stage['domain'], session.used)}")
        elif command == "skip":
            session.skip()
        else:
            stage_log = session.current_log
            count = len(stage_log["guesses"])
            session.attack(guess)
            if len(stage_log["guesses"]) > count:
                g = stage_log["guesses"][-1]
                print(f"  {g['Guess']}: {g['Tier']} · r {g['Relevance r']} · -{g['Damage HP']} HP · boss HP {g['Boss HP after']}")
            else:
                print("  Not counted (invalid guess).")
    for s in session.stage_logs:
        print(f"\nStage {s['stage']} {s['enemy']} ({s['domain']}, target '{s['target']}'): {s['status']}, "
              f"boss HP left {s['hp_left']}/{s['hp_max']}, {s['turns']} turns")
        if s["guesses"]:
            display(pd.DataFrame(s["guesses"]))

def build_game_widget(session: GameSession):
    """Topic / target cards, HP bar, input row with Attack / Hint / Skip / Quit / New game, styled log."""
    import ipywidgets as widgets
    from IPython.display import HTML

    header_card = widgets.HTML()
    hp_bar = widgets.IntProgress(min=0, layout=widgets.Layout(width="100%", height="16px", margin="6px 0 12px 0"))
    box = widgets.Text(placeholder="Type 1–3 English words, then Enter or Attack…",
                       layout=widgets.Layout(width="330px", height="36px"))
    btn = widgets.Layout(height="36px", width="auto", margin="0 0 0 6px")
    buttons = {
        "Attack": widgets.Button(description="⚔️ Attack", button_style="danger", layout=btn),
        "Hint": widgets.Button(description="💡 Hint", button_style="info", layout=btn),
        "Skip": widgets.Button(description="⏭️ Skip boss", layout=btn),
        "Quit": widgets.Button(description="🏳️ Quit", button_style="warning", layout=btn),
        "New": widgets.Button(description="🔄 New game", button_style="success", layout=btn),
    }
    log = widgets.Output(layout=widgets.Layout(border="1px solid #3f3f46", padding="6px 10px", margin="12px 0 0 0",
                                               max_height="460px", overflow="auto"))

    def show(blocks: list) -> None:
        with log:
            for block in blocks:
                display(HTML(block))

    def refresh() -> None:
        for name in ("Attack", "Hint", "Skip", "Quit"):
            buttons[name].disabled = session.finished
        box.disabled = session.finished
        if session.finished:
            defeated = len(session.records)
            played = len(session.stage_logs)
            title = "🏆 VICTORY — every boss defeated!" if defeated == len(session.stages) else "🏁 GAME OVER"
            color = "#22c55e" if defeated == len(session.stages) else "#a855f7"
            header_card.value = (f"<div style='{CARD}padding:16px;border:2px solid {color}'>"
                                 f"<div style='font-size:20px;font-weight:bold;color:{color}'>{title}</div>"
                                 f"<div style='margin-top:6px;font-size:14px;color:#d4d4d8'>Defeated <b>{defeated}/{played}</b> "
                                 f"bosses played. The summary tables are below. Press <b>New game</b> to play again.</div></div>")
            hp_bar.max, hp_bar.value = 100, 0
            return
        s = session.stage
        color = TOPIC_COLORS.get(s["domain"], "#eab308")
        hp_left, hp_max = max(session.hp, 0), s["hp"]
        hp_pct = hp_left / hp_max if hp_max else 0
        hp_bar.max, hp_bar.value = hp_max, hp_left
        hp_bar.bar_style = "success" if hp_pct > 0.5 else "warning" if hp_pct > 0.25 else "danger"
        header_card.value = f"""
        <div style="{CARD}padding:14px 16px">
          <div style="display:flex;justify-content:space-between;align-items:center;border-bottom:1px solid #27272a;padding-bottom:8px;margin-bottom:12px">
            <div><span style="background:#27272a;color:#a1a1aa;padding:3px 8px;border-radius:4px;font-size:12px;font-weight:bold">
              STAGE {s['stage']} / {len(session.stages)}</span>
              <span style="margin-left:10px;font-size:16px;font-weight:bold;color:#e4e4e7">👾 Boss: {_html.escape(s['enemy'])}</span></div>
            <div style="font-size:13px;color:#a1a1aa">Turns: <b style="color:#fff;font-size:15px">{session.turns}</b></div>
          </div>
          <div style="display:flex;gap:14px;flex-wrap:wrap;margin-bottom:12px">
            <div style="flex:1;min-width:180px;background:#27272a;border:2px solid {color};border-radius:8px;padding:10px 14px">
              <div style="font-size:11px;letter-spacing:1px;color:{color};font-weight:bold">📌 REQUIRED TOPIC (DOMAIN)</div>
              <div style="font-size:22px;font-weight:900;color:#fff;margin-top:2px">{s['domain'].upper()}</div></div>
            <div style="flex:1;min-width:180px;background:#27272a;border:2px solid #f59e0b;border-radius:8px;padding:10px 14px">
              <div style="font-size:11px;letter-spacing:1px;color:#fbbf24;font-weight:bold">🎯 TARGET WORD</div>
              <div style="font-size:22px;font-weight:900;color:#fde047;margin-top:2px">'{_html.escape(s['target'].upper())}'</div></div>
          </div>
          <div style="display:flex;justify-content:space-between;font-size:13px;font-weight:bold;color:#d4d4d8">
            <span>❤️ BOSS HP</span><span>{hp_left} / {hp_max} HP ({hp_pct:.0%})</span></div>
        </div>"""

    def on_attack(_=None) -> None:
        guess = box.value.strip()
        if not guess or session.finished:
            return
        box.value = ""
        show(session.attack(guess))
        refresh()

    def on_hint(_=None) -> None:
        show(session.hint())

    def on_skip(_=None) -> None:
        show(session.skip())
        refresh()

    def on_quit(_=None) -> None:
        show(session.end_game())
        refresh()

    def on_new(_=None) -> None:
        log.clear_output()
        session.new_game()
        s = session.stage
        show([f"<div style='{CARD}padding:8px 14px'>🎮 New game · {len(session.stages)} stages · first topic "
              f"<b>{s['domain'].upper()}</b>, target <b style='color:#fde047'>'{_html.escape(s['target'].upper())}'</b></div>"])
        refresh()

    buttons["Attack"].on_click(on_attack)
    buttons["Hint"].on_click(on_hint)
    buttons["Skip"].on_click(on_skip)
    buttons["Quit"].on_click(on_quit)
    buttons["New"].on_click(on_new)
    with warnings.catch_warnings():            # Text.on_submit is deprecated but still the simplest Enter hook
        warnings.simplefilter("ignore")
        if hasattr(box, "on_submit"):
            box.on_submit(on_attack)
    refresh()
    controls = widgets.HBox([box, *buttons.values()], layout=widgets.Layout(align_items="center", flex_flow="row wrap"))
    return widgets.VBox([header_card, hp_bar, controls, log], layout=widgets.Layout(max_width="820px", padding="4px"))

with section("Play Game"):
    GAME = GameSession(engine, TARGET_BANK, RANDOM_ROUNDS, SEED)
    print(f"Scoring model: {MODEL_LABELS[ACTIVE_MODEL]} — {MODEL_SOURCE}")
    print("Type a guess and press Enter or Attack. Hint suggests a word, Skip boss moves on, "
          "Quit ends the game and shows the summary tables, New game draws new topics and targets.")
    try:
        display(build_game_widget(GAME))
    except ImportError:
        print("WARNING: ipywidgets is not available — run play_in_text_mode(GAME) in a new cell to play.")